# ConvNeXt on CIFAR-10 — 可在 Google Colab 直接运行的完整实现

**课程**：CISC3024 Pattern Recognition ｜ **作业**：AIassign2（AI Assignment #1）

**算法**：ConvNeXt —— *A ConvNet for the 2020s*, **CVPR 2022**（arXiv:2201.03545）
｜ **类别**：Deep CNN ｜ **应用**：计算机视觉 / 模式识别（图像分类）


## 本 Notebook 如何使用

1. 建议先在 Colab 中开启 GPU：**代码执行程序 → 更改运行时类型 → T4 GPU**；
2. 从上到下依次执行单元格：第 1 步做环境自检，第 2–10 步会**自动生成完整工程源码**，
   第 11 步（可选）跑快速冒烟测试，第 12 步开始正式训练，之后自动展示曲线与指标；
3. 所有源码与工程结构完全一致，产物写入 `outputs/`（模型、日志、图表、指标）。

## 流水线总览

```
CIFAR-10 下载/划分/增强  ->  ConvNeXt 构建  ->  显存自适应 batch size 探测
        ->  训练(warmup+cosine, AdamW, AMP, Mixup/CutMix, 梯度累积)
        ->  每轮验证并保存最优检查点  ->  测试集评估(Top-1/Top-5, 逐类准确率, 混淆矩阵)
        ->  绘制损失曲线 / 准确率曲线 / 混淆矩阵 / 逐类准确率
```


## 1. 环境自检

确认 Colab 的 Python / PyTorch 版本与 GPU 是否可用。


In [ ]:
#@title 1. 环境自检（版本 / GPU / 显存）
import os, sys, platform
import torch
import torchvision

print("=" * 80)
print("Python       :", sys.version.split()[0], "|", platform.platform())
print("PyTorch      :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("NumPy        :", __import__("numpy").__version__)
print("-" * 80)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("CUDA         : 可用")
    print("GPU          :", torch.cuda.get_device_name(0))
    print(f"显存总量      : {props.total_memory / 1024 ** 3:.1f} GB | 计算能力 sm_{props.major}{props.minor}")
else:
    print("CUDA         : 不可用 —— 将在 CPU 上训练（速度较慢）")
    print("提示：代码执行程序 -> 更改运行时类型 -> 选择 T4 GPU")
print("=" * 80)


## 2. 创建工作目录

在 Colab 中于 `/content/convnext_cifar10` 下创建完整的工程目录结构。


In [ ]:
#@title 2. 创建工作目录与子目录
import os

PROJECT_DIR = "/content/convnext_cifar10"
SUBDIRS = [
    "src/data", "src/models", "src/engine", "src/utils", "notebooks", "scripts",
    "outputs/figures", "outputs/logs", "outputs/models", "outputs/metrics",
]

os.makedirs(PROJECT_DIR, exist_ok=True)
for sub in SUBDIRS:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

# 切换到工程根目录（后续 !python train.py 都在此目录执行）
os.chdir(PROJECT_DIR)
print("工作目录 :", os.getcwd())
print("已创建子目录 :", SUBDIRS)


## 3. 生成工程源码

以下每个单元格都会把一个源码文件写入磁盘（`%%writefile`），内容与 `03_code/` 工程完全一致。依次执行即可。


### 2. 全局配置


`configs.py`（358 行）


In [ ]:
%%writefile configs.py
# -*- coding: utf-8 -*-
"""
全局配置模块（ConvNeXt + CIFAR-10）。

本文件集中管理**数据 / 模型 / 训练**三组超参数，避免超参数散落在代码各处。
所有字段都可以通过命令行参数（见 ``train.py`` 与 ``evaluate.py``）覆盖。

设计说明
--------
* 使用 ``dataclass`` 组织配置，便于 `asdict()` 序列化为 JSON 存档（可复现性证据）。
* 训练相关的默认值参考 ConvNeXt 原论文的训练配方：
  AdamW + 余弦退火 + 热身（warmup）+ 标签平滑（label smoothing）
  + Mixup / CutMix + 权重衰减 0.05（不对权重为 1 维的参数施加）。
"""

from __future__ import annotations

import argparse
import json
import os
from dataclasses import asdict, dataclass, field
from typing import List, Optional, Tuple

# 模型预设名称列表（真实定义见 src/models/convnext.py）
CONVNEXT_PRESET_NAMES: Tuple[str, ...] = ("atto", "femto", "pico", "nano", "tiny")


# --------------------------------------------------------------------------- #
# 1. 数据配置
# --------------------------------------------------------------------------- #
@dataclass
class DataConfig:
    """CIFAR-10 数据加载与预处理配置。"""

    # 数据集类型：
    #   "CIFAR10"   —— 真实数据集（首次运行需下载约 170 MB）
    #   "synthetic" —— 本地生成的合成数据，**零下载**，用于网络受限时的流程冒烟测试
    dataset: str = "CIFAR10"
    data_root: str = "./data"          # 数据集下载 / 缓存目录
    num_classes: int = 10              # CIFAR-10 有 10 个类别
    val_ratio: float = 0.1             # 从 50000 张训练集中划出 10% 作为验证集
    # 合成数据规模（仅 dataset="synthetic" 时生效）
    synthetic_train_size: int = 4096
    synthetic_test_size: int = 2048
    # ---- 快速验证子集（CPU 上做 MVP/冒烟实验用；None 表示使用全量数据）---- #
    # 二者均按固定随机种子抽样，且会写入结果文件，保证可追溯、可复现。
    train_subset_size: Optional[int] = None   # 从 50000 张训练池中抽取的样本数
    test_subset_size: Optional[int] = None    # 从 10000 张测试集中抽取的样本数
    num_workers: int = 2               # Colab 免费版一般是 2 核 CPU
    pin_memory: bool = True            # GPU 训练时开启可加速 Host->Device 拷贝
    random_erase: float = 0.25         # RandomErasing 的应用概率；<=0 表示关闭
    rand_augment: bool = False         # 是否叠加 RandAugment（默认关闭，避免过强增强）
    # CIFAR-10 训练集统计得到的通道均值与标准差（在 ToTensor() 缩放到 [0,1] 之后）
    mean: Tuple[float, float, float] = (0.4914, 0.4822, 0.4465)
    std: Tuple[float, float, float] = (0.2470, 0.2435, 0.2616)


# --------------------------------------------------------------------------- #
# 2. 模型配置
# --------------------------------------------------------------------------- #
@dataclass
class ModelConfig:
    """ConvNeXt 结构配置。"""

    name: str = "tiny"                 # atto / femto / pico / nano / tiny
    # 若为 None，则使用 name 对应的预设；也可显式指定以做消融实验
    depths: Optional[List[int]] = None            # 每个阶段的 Block 数量
    dims: Optional[List[int]] = None              # 每个阶段的通道数
    drop_path_rate: float = 0.1                   # Stochastic Depth 的最大丢弃率
    layer_scale_init_value: float = 1e-6          # Layer Scale 初始值
    stem_mode: str = "conv"                       # "conv"(3x3, s1，适配 32x32) | "patchify"(4x4, s4)
    head_init_scale: float = 1.0                  # 分类头初始权重的缩放系数


# --------------------------------------------------------------------------- #
# 3. 训练配置
# --------------------------------------------------------------------------- #
@dataclass
class TrainConfig:
    """训练与评估超参数。"""

    epochs: int = 60                   # 训练总轮数
    batch_size: int = 128              # **期望**的 batch size（显存不足时自动下调）
    auto_batch_size: bool = True       # 是否启用"显存自适应 batch size"
    max_batch_size: int = 512          # 自动探测时允许的最大 batch size
    min_batch_size: int = 8            # 自动探测时允许的最小 batch size
    lr: float = 1e-3                   # 基础学习率（AdamW）
    min_lr: float = 1e-5               # 余弦退火的下界学习率
    weight_decay: float = 0.05         # AdamW 权重衰减
    warmup_epochs: int = 5             # 线性热身轮数
    label_smoothing: float = 0.1       # 标签平滑系数
    grad_clip: float = 1.0             # 梯度裁剪阈值；<=0 表示关闭
    use_mixup: bool = True             # 是否启用 Mixup / CutMix
    mixup_prob: float = 0.5            # 每个 batch 应用 Mixup/CutMix 的概率
    mixup_alpha: float = 0.8           # Mixup 的 Beta 分布参数
    cutmix_alpha: float = 1.0          # CutMix 的 Beta 分布参数
    amp: bool = True                   # 是否在 GPU 上启用混合精度训练
    # AMP 安全网：fp16 的 GradScaler 动态 scale 一旦崩塌（典型值 <1），
    # 说明梯度已在下溢、部分参数停止更新；开启本项后会自动关闭 AMP 并以 fp32 续训。
    # 详见 src/utils/amp.py 与 src/engine/trainer.py::_track_amp_scale
    amp_fallback: bool = True
    tta: bool = False                  # 最终测试评估是否启用测试时增强（原图 + 水平翻转）
    tta_hflip: bool = True             # TTA 是否包含水平翻转视角（tta=True 时生效）
    seed: int = 42                     # 全局随机种子
    deterministic: bool = False        # 是否强制 cuDNN 确定性（会略微降低速度）
    plot_every: int = 10               # 每多少轮刷新一次曲线图；<=0 表示只在结束时绘制
    # 每个 epoch 最多训练多少个 step；None 表示不限制。
    # 用途：让"快速冒烟测试"的耗时**只取决于 step 数**，与数据集规模、硬件性能解耦 ——
    # 在只有 2 vCPU 的 Colab CPU 运行时上，即使很小的数据集，完整跑完一个 epoch 也可能超出预期。
    max_steps_per_epoch: Optional[int] = None
    early_stop_patience: int = 0       # 早停耐心值（以轮为单位）；0 表示不启用
    resume: Optional[str] = None       # 断点续训的 checkpoint 路径


# --------------------------------------------------------------------------- #
# 4. 顶层配置
# --------------------------------------------------------------------------- #
@dataclass
class Config:
    """完整配置对象，聚合数据、模型与训练配置，并派生输出目录。"""

    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    output_dir: str = "./outputs"      # 所有产物（模型/日志/图表）的根目录
    device: str = "auto"               # auto / cuda / mps / cpu
    exp_name: str = "convnext_cifar10" # 实验名称，写入日志与指标文件
    # 实验标签：设置后，全部产物写入 ``<output_dir>/<run_tag>/``，
    # 从而让"全量训练 / 对比实验 / 消融实验 / 多种子重复"各自独立成目录、
    # 互不覆盖、互不交叉引用。为 None 时保持旧行为（直接写入 output_dir）。
    run_tag: Optional[str] = None

    # ---------------- 派生路径 ---------------- #
    # 输出目录按"结果三分类"组织，与作业 05_results 的结构一一对应：
    #   figures(图表) / logs(日志) / models(模型) / metrics(指标)
    @property
    def run_root(self) -> str:
        """本次实验的产物根目录：``<output_dir>/<run_tag>``（未设置 run_tag 时即 output_dir）。"""
        if self.run_tag:
            return os.path.join(self.output_dir, str(self.run_tag))
        return self.output_dir

    @property
    def fig_dir(self) -> str:
        """图表目录：训练曲线、混淆矩阵、逐类准确率、数据样例。"""
        return os.path.join(self.run_root, "figures")

    @property
    def log_dir(self) -> str:
        """日志目录：训练/评估日志与逐轮训练历史。"""
        return os.path.join(self.run_root, "logs")

    @property
    def ckpt_dir(self) -> str:
        """模型目录：最优与最后一轮的检查点文件。"""
        return os.path.join(self.run_root, "models")

    @property
    def result_dir(self) -> str:
        """指标目录：超参数、测试指标、逐类指标、指标汇总表等结构化结果。"""
        return os.path.join(self.run_root, "metrics")

    def make_dirs(self) -> None:
        """创建所有输出目录（幂等）。"""
        for d in (self.output_dir, self.run_root, self.ckpt_dir, self.log_dir,
                  self.fig_dir, self.result_dir):
            os.makedirs(d, exist_ok=True)

    # ---------------- 序列化 ---------------- #
    def to_dict(self) -> dict:
        return asdict(self)

    def save(self, path: str) -> None:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=2, ensure_ascii=False)

    @classmethod
    def from_dict(cls, data: dict) -> "Config":
        """由字典重建 :class:`Config`（用于从检查点中恢复训练时的完整配置）。

        只保留与 dataclass 字段同名的键，多余键会被忽略，
        因此可以安全地读取旧版本保存的检查点。
        """

        def _filtered(dc_type, payload: dict) -> dict:
            fields = set(dc_type.__dataclass_fields__.keys())
            return {k: v for k, v in dict(payload or {}).items() if k in fields}

        data = data or {}
        return cls(
            data=DataConfig(**_filtered(DataConfig, data.get("data", {}))),
            model=ModelConfig(**_filtered(ModelConfig, data.get("model", {}))),
            train=TrainConfig(**_filtered(TrainConfig, data.get("train", {}))),
            output_dir=data.get("output_dir", cls.output_dir),
            device=data.get("device", cls.device),
            exp_name=data.get("exp_name", cls.exp_name),
            run_tag=data.get("run_tag", None),
        )

    def copy(self) -> "Config":
        """返回配置的深拷贝（避免在覆盖参数时污染原始对象）。"""
        return Config.from_dict(self.to_dict())


# --------------------------------------------------------------------------- #
# 5. 命令行参数解析
# --------------------------------------------------------------------------- #
def parse_args(argv: Optional[List[str]] = None, base: Optional[Config] = None) -> Config:
    """解析命令行参数并返回覆盖后的 :class:`Config`。

    参数
    ----
    argv : list[str] | None
        参数列表；为 None 时读取 ``sys.argv[1:]``。
    base : Config | None
        基准配置。**只覆盖用户显式传入的参数**（所有选项的默认值均为 None），
        因此可以先用检查点中保存的配置作为基准，再由命令行做局部覆盖，
        从而保证评估时模型结构与训练时完全一致。

    返回
    ----
    Config
        合并命令行覆盖项后的配置对象。
    """
    parser = argparse.ArgumentParser(
        description="ConvNeXt on CIFAR-10 (PyTorch, Colab-friendly)",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    # 数据
    parser.add_argument("--data-root", type=str, default=None, help="数据集缓存目录")
    parser.add_argument("--dataset", type=str, default=None, choices=["CIFAR10", "synthetic"],
                        help="数据集类型：CIFAR10（真实，需下载）或 synthetic（本地合成，零下载）")
    parser.add_argument("--synthetic-train-size", type=int, default=None,
                        help="合成数据训练集规模（dataset=synthetic 时生效）")
    parser.add_argument("--synthetic-test-size", type=int, default=None,
                        help="合成数据测试集规模（dataset=synthetic 时生效）")
    parser.add_argument("--num-workers", type=int, default=None, help="DataLoader 工作进程数")
    parser.add_argument("--val-ratio", type=float, default=None, help="验证集比例")
    parser.add_argument("--train-subset-size", type=int, default=None,
                        help="快速验证子集：从训练池中抽取的样本数（默认使用全量 50000）")
    parser.add_argument("--test-subset-size", type=int, default=None,
                        help="快速验证子集：从测试集中抽取的样本数（默认使用全量 10000）")
    # 模型
    parser.add_argument("--model", type=str, default=None, choices=list(CONVNEXT_PRESET_NAMES),
                        help="ConvNeXt 预设规模")
    parser.add_argument("--stem-mode", type=str, default=None, choices=["conv", "patchify"],
                        help="Stem 类型：conv=3x3 s1（适配 32x32），patchify=论文 4x4 s4")
    parser.add_argument("--drop-path", type=float, default=None, help="Stochastic Depth 丢弃率")
    # 训练
    parser.add_argument("--epochs", type=int, default=None, help="训练轮数")
    parser.add_argument("--batch-size", type=int, default=None, help="期望 batch size")
    parser.add_argument("--lr", type=float, default=None, help="基础学习率")
    parser.add_argument("--weight-decay", type=float, default=None, help="权重衰减")
    parser.add_argument("--warmup-epochs", type=int, default=None, help="线性热身轮数")
    parser.add_argument("--label-smoothing", type=float, default=None, help="标签平滑系数")
    parser.add_argument("--plot-every", type=int, default=None, help="每多少轮刷新一次曲线图（0=仅结束时绘制）")
    parser.add_argument("--max-steps-per-epoch", type=int, default=None,
                        help="每个 epoch 最多训练多少个 step（用于限时冒烟测试；默认不限制）")
    parser.add_argument("--early-stop-patience", type=int, default=None, help="早停耐心值（0=不启用）")
    parser.add_argument("--seed", type=int, default=None, help="全局随机种子")
    parser.add_argument("--resume", type=str, default=None,
                        help="断点续训的 checkpoint 路径；填 'auto' 则自动从本次实验的 last.pt 续训")
    parser.add_argument("--output-dir", type=str, default=None, help="输出根目录")
    parser.add_argument("--exp-name", type=str, default=None, help="实验名称")
    parser.add_argument("--run-tag", type=str, default=None,
                        help="实验标签：产物写入 <output-dir>/<run-tag>/，实现实验间目录隔离")
    parser.add_argument("--device", type=str, default=None, help="auto / cuda / mps / cpu")
    # 开关型参数
    parser.add_argument("--no-amp", action="store_true", help="关闭混合精度训练（fp16 下溢时的兜底做法）")
    parser.add_argument("--no-amp-fallback", action="store_true",
                        help="禁用 AMP 安全网：GradScaler scale 崩塌时不再自动降级为 fp32")
    parser.add_argument("--tta", action="store_true",
                        help="最终测试评估启用 TTA（原图 + 水平翻转，平均 softmax 概率）")
    parser.add_argument("--no-tta-hflip", action="store_true",
                        help="TTA 只保留原图视角（等价于不启用翻转分支）")
    parser.add_argument("--no-mixup", action="store_true", help="关闭 Mixup / CutMix")
    parser.add_argument("--no-auto-batch-size", action="store_true", help="关闭显存自适应 batch size")
    parser.add_argument("--deterministic", action="store_true", help="强制 cuDNN 确定性算法")

    args = parser.parse_args(argv)
    cfg = base.copy() if base is not None else Config()

    # ---- 数据 ----
    if args.data_root is not None:
        cfg.data.data_root = args.data_root
    if args.dataset is not None:
        cfg.data.dataset = args.dataset
    if args.synthetic_train_size is not None:
        cfg.data.synthetic_train_size = args.synthetic_train_size
    if args.synthetic_test_size is not None:
        cfg.data.synthetic_test_size = args.synthetic_test_size
    if args.num_workers is not None:
        cfg.data.num_workers = args.num_workers
    if args.val_ratio is not None:
        cfg.data.val_ratio = args.val_ratio
    if args.train_subset_size is not None:
        cfg.data.train_subset_size = args.train_subset_size
    if args.test_subset_size is not None:
        cfg.data.test_subset_size = args.test_subset_size

    # ---- 模型 ----
    if args.model is not None:
        cfg.model.name = args.model
    if args.stem_mode is not None:
        cfg.model.stem_mode = args.stem_mode
    if args.drop_path is not None:
        cfg.model.drop_path_rate = args.drop_path

    # ---- 训练 ----
    if args.epochs is not None:
        cfg.train.epochs = args.epochs
    if args.batch_size is not None:
        cfg.train.batch_size = args.batch_size
    if args.lr is not None:
        cfg.train.lr = args.lr
    if args.weight_decay is not None:
        cfg.train.weight_decay = args.weight_decay
    if args.warmup_epochs is not None:
        cfg.train.warmup_epochs = args.warmup_epochs
    if args.label_smoothing is not None:
        cfg.train.label_smoothing = args.label_smoothing
    if args.plot_every is not None:
        cfg.train.plot_every = args.plot_every
    if args.max_steps_per_epoch is not None:
        cfg.train.max_steps_per_epoch = args.max_steps_per_epoch
    if args.early_stop_patience is not None:
        cfg.train.early_stop_patience = args.early_stop_patience
    if args.seed is not None:
        cfg.train.seed = args.seed
    if args.resume is not None:
        cfg.train.resume = args.resume
    if args.no_amp:
        cfg.train.amp = False
    if args.no_amp_fallback:
        cfg.train.amp_fallback = False
    if args.tta:
        cfg.train.tta = True
    if args.no_tta_hflip:
        cfg.train.tta_hflip = False
    if args.no_mixup:
        cfg.train.use_mixup = False
    if args.no_auto_batch_size:
        cfg.train.auto_batch_size = False
    if args.deterministic:
        cfg.train.deterministic = True

    # ---- 其他 ----
    if args.output_dir is not None:
        cfg.output_dir = args.output_dir
    if args.exp_name is not None:
        cfg.exp_name = args.exp_name
    if args.run_tag is not None:
        cfg.run_tag = args.run_tag
    if args.device is not None:
        cfg.device = args.device

    return cfg


`src/__init__.py`（4 行）


In [ ]:
%%writefile src/__init__.py
# -*- coding: utf-8 -*-
"""ConvNeXt + CIFAR-10 工程源码包。"""

__version__ = "1.0.0"


`src/utils/__init__.py`（2 行）


In [ ]:
%%writefile src/utils/__init__.py
# -*- coding: utf-8 -*-
"""通用工具：随机种子、设备与显存、日志、AMP 兼容层、Mixup、TTA、CSV 导出、可视化、指标。"""


### 3. 工具模块 —— 日志


`src/utils/logger.py`（51 行）


In [ ]:
%%writefile src/utils/logger.py
# -*- coding: utf-8 -*-
"""轻量日志工具：同时输出到控制台与文件，便于在 ``05_results/`` 归档训练日志。"""

from __future__ import annotations

import logging
import os
import sys
from typing import Optional


def get_logger(name: str = "convnext", log_file: Optional[str] = None,
               level: int = logging.INFO) -> logging.Logger:
    """获取（或创建）一个带控制台输出的 logger。

    参数
    ----
    name : str
        logger 名称，同名 logger 复用已有 handler，避免重复打印。
    log_file : str | None
        日志文件路径；为 None 时只打印到控制台。
    level : int
        日志级别。

    返回
    ----
    logging.Logger
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = False  # 避免被 root logger 重复输出

    if logger.handlers:      # 已经初始化过，直接复用
        return logger

    fmt = logging.Formatter(
        fmt="[%(asctime)s] %(levelname)-5s | %(message)s",
        datefmt="%H:%M:%S",
    )

    stream_handler = logging.StreamHandler(stream=sys.stdout)
    stream_handler.setFormatter(fmt)
    logger.addHandler(stream_handler)

    if log_file:
        os.makedirs(os.path.dirname(os.path.abspath(log_file)), exist_ok=True)
        file_handler = logging.FileHandler(log_file, mode="a", encoding="utf-8")
        file_handler.setFormatter(fmt)
        logger.addHandler(file_handler)

    return logger


### 3. 工具模块 —— 随机种子


`src/utils/seed.py`（63 行）


In [ ]:
%%writefile src/utils/seed.py
# -*- coding: utf-8 -*-
"""
随机种子控制（可复现性保证）。

为保证多次运行结果一致，需要同时固定：
1. Python 内置 ``random`` 模块的种子；
2. NumPy 的随机种子（数据增强、Mixup 的 Beta 采样依赖它）；
3. PyTorch CPU / GPU 的随机种子；
4. DataLoader 多进程 worker 的随机种子（见 :func:`seed_worker`）。
"""

from __future__ import annotations

import os
import random

import numpy as np
import torch


def set_seed(seed: int = 42, deterministic: bool = False) -> None:
    """固定所有随机源。

    参数
    ----
    seed : int
        全局随机种子。
    deterministic : bool
        为 True 时强制 cuDNN 使用确定性算法（速度略降，但结果完全可复现）；
        为 False 时允许 cuDNN 自动搜索最优卷积算法（速度更快）。
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # 多卡场景
    os.environ["PYTHONHASHSEED"] = str(seed)  # 影响 str/bytes 的 hash，保证 dict 顺序稳定

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # PyTorch >= 1.8 提供该开关，可进一步降低非确定性算子的影响
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:  # pragma: no cover - 老版本兜底
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def seed_worker(worker_id: int) -> None:  # noqa: ARG001 (worker_id 由 PyTorch 传入)
    """DataLoader 多进程 worker 的初始化函数：让每个 worker 拥有独立的可复现种子。"""
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed: int) -> torch.Generator:
    """创建固定种子的 ``torch.Generator``，用于 DataLoader 的 shuffle 复现。"""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


### 3. 工具模块 —— 指标累加与 JSON


`src/utils/misc.py`（110 行）


In [ ]:
%%writefile src/utils/misc.py
# -*- coding: utf-8 -*-
"""杂项工具：指标累加器、JSON 读写、参数量统计、时间格式化。

说明
----
CSV 类导出（``metrics.csv`` / ``history.csv``）已统一收敛到
:mod:`src.utils.csv_logger`，本模块只保留与格式无关的通用小工具。
"""

from __future__ import annotations

import json
import os
from typing import Any, Dict, Optional, Tuple


class AverageMeter:
    """滑动平均指标累加器（用于按样本数加权的 epoch 平均损失/准确率）。"""

    def __init__(self, name: str = "") -> None:
        self.name = name
        self.reset()

    def reset(self) -> None:
        self.val = 0.0     # 当前累计值（总和）
        self.avg = 0.0     # 平均值
        self.sum = 0.0     # 总和
        self.count = 0     # 样本计数

    def update(self, val: float, n: int = 1) -> None:
        """累加一次观测值。

        参数
        ----
        val : float
            本次观测的指标值（如某个 batch 的损失或准确率）。
        n : int
            本次观测对应的样本数量，用于加权平均。
        """
        self.val = float(val)
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(1, self.count)

    def __str__(self) -> str:  # 便于格式化输出
        return f"{self.name}={self.avg:.4f}" if self.name else f"{self.avg:.4f}"


def save_json(obj: Any, path: str, indent: int = 2) -> None:
    """把任意可序列化对象以 UTF-8 JSON 写入磁盘（自动创建父目录）。"""
    directory = os.path.dirname(os.path.abspath(path))
    os.makedirs(directory, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent, ensure_ascii=False)


def load_json(path: str) -> Any:
    """读取 JSON 文件。"""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def count_parameters(model) -> Tuple[int, int]:
    """统计模型参数量。

    返回
    ----
    (total, trainable) : tuple[int, int]
        总参数量与可训练参数量。
    """
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def format_params(num_params: int) -> str:
    """把参数量格式化为易读字符串（如 "27.83 M"）。"""
    if num_params >= 1_000_000:
        return f"{num_params / 1_000_000:.2f} M"
    if num_params >= 1_000:
        return f"{num_params / 1_000:.2f} K"
    return str(num_params)


def human_time(seconds: float) -> str:
    """把秒数格式化为 "1h 02m 03s" 形式的字符串。"""
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    return f"{m}m {s:02d}s"


def as_plain_dict(cfg_dict: Dict[str, Any]) -> Dict[str, Any]:
    """把配置字典中不可序列化的类型（如 tuple）转换为 list。"""
    out: Dict[str, Any] = {}
    for k, v in cfg_dict.items():
        if isinstance(v, tuple):
            out[k] = list(v)
        elif isinstance(v, dict):
            out[k] = as_plain_dict(v)
        else:
            out[k] = v
    return out


def optional(value: Optional[Any], default: Any) -> Any:
    """若 value 为 None 则返回 default，否则返回 value。"""
    return default if value is None else value


### 3. 工具模块 —— CSV 导出（history.csv / metrics.csv）


`src/utils/csv_logger.py`（204 行）


In [ ]:
%%writefile src/utils/csv_logger.py
# -*- coding: utf-8 -*-
"""
CSV 导出工具：把训练过程的逐轮指标与最终结果汇总写成 CSV。

为什么需要 CSV
--------------
报告中的表格（逐轮训练指标表、多实验对比表）若从 JSON 手工誊抄，既费时又易出错。
本模块把结构化结果**自动**导出为**列顺序稳定**的 CSV，从而：

* ``history.csv`` —— 一行一轮的训练历史，可直接作为报告"逐轮指标表"；
* ``metrics.csv`` —— 一行一次实验的结果汇总（含逐类准确率），
  把多个实验的该文件**纵向拼接**即可得到对比 / 消融表格。

由于列顺序由下面的常量显式固定（而非按字典自动展开），
不同实验、不同代码版本产出的 CSV 可以直接堆叠，不存在列错位风险。
"""

from __future__ import annotations

import csv
import os
from typing import Any, Dict, List, Sequence, Tuple


def _ensure_parent(path: str) -> None:
    """确保 CSV 的父目录存在。"""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)


# --------------------------------------------------------------------------- #
# 1. 逐轮训练历史 -> history.csv
# --------------------------------------------------------------------------- #
#: ``history.csv`` 的列定义：(表头, history 字典中的键)。
_HISTORY_COLUMNS: List[Tuple[str, str]] = [
    ("epoch",         "epoch"),
    ("train_loss",    "train_loss"),
    ("train_acc",     "train_acc"),
    ("val_loss",      "val_loss"),
    ("val_acc",       "val_acc"),
    ("lr",            "lr"),
    ("epoch_time_sec", "epoch_time"),
    # AMP 稳定性证据：fp16 的 GradScaler 动态 scale。正常应稳定在 65536 附近；
    # 若持续低于 1，说明梯度已大面积下溢、部分参数停止更新（见 src/utils/amp.py）。
    # 未启用 AMP、或已自动降级为 fp32 的轮次记为 1.0（意为"未做放大"）；
    # 崩塌发生的那一轮会如实记下极小值，并另有 amp.fell_back_to_fp32 标记。
    ("amp_scale",     "amp_scale"),
]


def _format_history_value(key: str, value: Any) -> str:
    """按列类型格式化单个取值。

    * 准确率统一输出为**百分数**（保留 4 位小数），便于直接进报告表格；
      因此既接受 0~1 的小数，也接受已经是百分数的输入。
    * 学习率用科学计数法，便于观察 warmup + cosine 的衰减轨迹。
    """
    if value is None or value == "":
        return ""
    number = float(value)
    if key in ("train_acc", "val_acc"):
        return f"{number * 100.0:.4f}" if number <= 1.0 else f"{number:.4f}"
    if key in ("lr", "amp_scale"):
        return f"{number:.3e}"
    if key == "epoch_time":
        return f"{number:.2f}"
    if key == "epoch":
        return str(int(number))
    return f"{number:.6f}"


def write_history_csv(history: Dict[str, Sequence[float]], path: str) -> str:
    """把逐轮训练历史写成 CSV（一行一轮）。

    参数
    ----
    history : dict
        训练历史字典，通常包含
        ``epoch, train_loss, train_acc, val_loss, val_acc, lr, epoch_time, amp_scale``。
        缺列或各列长度不一致时会自动补空值，不会抛异常。
    path : str
        输出 CSV 路径（自动创建父目录）。

    返回
    ----
    str
        实际写入的路径。
    """
    series: Dict[str, List[Any]] = {
        key: list(history.get(key) or []) for _, key in _HISTORY_COLUMNS
    }
    num_rows = max((len(values) for values in series.values()), default=0)
    for values in series.values():
        values.extend([""] * (num_rows - len(values)))

    _ensure_parent(path)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([header for header, _ in _HISTORY_COLUMNS])
        for row_index in range(num_rows):
            writer.writerow([
                _format_history_value(key, series[key][row_index])
                for _, key in _HISTORY_COLUMNS
            ])
    return path


# --------------------------------------------------------------------------- #
# 2. 结果汇总 -> metrics.csv
# --------------------------------------------------------------------------- #
#: ``metrics.csv`` 的列定义：(列名, 在结果字典中的取值路径)。
_METRIC_COLUMNS: List[Tuple[str, Tuple[str, ...]]] = [
    ("experiment",            ("experiment",)),
    ("algorithm",             ("algorithm",)),
    ("model_preset",          ("architecture", "preset")),
    ("stem_mode",             ("architecture", "stem_mode")),
    ("drop_path_rate",        ("architecture", "drop_path_rate")),
    ("num_params",            ("architecture", "num_params")),
    ("dataset",               ("dataset", "name")),
    ("split",                 ("dataset", "split")),
    ("is_full_scale",         ("dataset", "is_full_scale")),
    ("train_subset_size",     ("dataset", "train_subset_size")),
    ("test_subset_size",      ("dataset", "test_subset_size")),
    ("val_ratio",             ("dataset", "val_ratio")),
    ("data_root",             ("dataset", "data_root")),
    ("device",                ("device",)),
    ("gpu_total_memory_gb",   ("gpu_total_memory_gb",)),
    ("batch_size_requested",  ("batch_size_requested",)),
    ("batch_size_actual",     ("batch_size_actual",)),
    ("grad_accum_steps",      ("grad_accum_steps",)),
    ("effective_batch_size",  ("effective_batch_size",)),
    ("epochs",                ("epochs",)),
    ("epochs_completed",      ("epochs_completed",)),
    ("best_val_acc",          ("best_val_acc",)),
    ("best_val_epoch",        ("best_val_epoch",)),
    ("test_loss",             ("test_metrics", "loss")),
    ("test_top1_acc",         ("test_metrics", "top1_acc")),
    ("test_top5_acc",         ("test_metrics", "top5_acc")),
    ("test_num_samples",      ("test_metrics", "num_samples")),
    ("macro_accuracy",        ("macro_accuracy",)),
    ("total_training_time",   ("total_training_time",)),
    ("seed",                  ("seed",)),
    # 混合精度可追溯性（fp16 下溢事故的判定依据，详见 src/utils/amp.py）：
    #   amp_dtype     —— autocast 实际使用的低精度类型（bfloat16 / float16 / 空）
    #   amp_min_scale —— GradScaler 动态 scale 的最小值；正常应接近 65536，
    #                    若为 <1 的小值则说明该次运行的骨干网络可能已停止更新
    ("amp_dtype",             ("amp", "dtype")),
    ("amp_min_scale",         ("amp", "min_scale")),
]


def _dig(data: Any, keys: Tuple[str, ...], default: Any = "") -> Any:
    """按 ``keys`` 路径逐层读取嵌套字典；任一层缺失或为 None 时返回 ``default``。"""
    current = data
    for key in keys:
        if not isinstance(current, dict) or key not in current or current[key] is None:
            return default
        current = current[key]
    return current


def write_metrics_csv(summary: Dict[str, Any], path: str) -> str:
    """把一次实验的结果汇总写成单行 ``metrics.csv``（含逐类准确率列）。

    参数
    ----
    summary : dict
        与 ``metrics/test_metrics.json`` 结构一致的结果汇总字典。
        本函数对缺失字段容错（取空字符串），因此也兼容 ``evaluate.py`` 产出的
        ``eval_metrics.json``。
    path : str
        输出 CSV 路径（自动创建父目录）。

    返回
    ----
    str
        实际写入的路径。

    说明
    ----
    采用"一行一次实验"的宽表格式：前 30 列是实验设置与总体指标，
    其后追加 ``acc_<类别名>`` 逐类准确率列。因此把多个实验的 ``metrics.csv``
    纵向拼接即可直接得到报告中的对比/消融表格。
    """
    columns = [name for name, _ in _METRIC_COLUMNS]
    row = [_dig(summary, keys) for _, keys in _METRIC_COLUMNS]

    per_class = summary.get("per_class_accuracy") if isinstance(summary, dict) else None
    if isinstance(per_class, dict):
        for class_name, accuracy in per_class.items():
            columns.append(f"acc_{class_name}")
            row.append(accuracy)

    _ensure_parent(path)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerow(row)
    return path


def read_metrics_csv(path: str) -> List[Dict[str, str]]:
    """读取 ``metrics.csv`` 为字典列表（供后续做实验对比表使用）。"""
    with open(path, "r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


### 3. 工具模块 —— 混合精度 AMP


`src/utils/amp.py`（125 行）


In [ ]:
%%writefile src/utils/amp.py
# -*- coding: utf-8 -*-
"""
混合精度（AMP）兼容层。

PyTorch 2.4 起推荐使用 ``torch.amp.autocast(device_type=...)`` 与
``torch.amp.GradScaler(device=...)``；旧版本则使用 ``torch.cuda.amp``。
本模块屏蔽这一差异，使工程在 Colab 各版本 PyTorch 上都能直接运行。

精度选择（本项目踩过的坑）
--------------------------
低精度默认**优先 bfloat16**（Ampere 及以后原生支持），T4 等旧卡回退 fp16。
原因是 fp16 的动态范围过窄：本模型 LayerScale 初值为 ``1e-6``，18 个残差块叠加后
浅层 Block 的真实梯度只有 ``1e-6`` 甚至更低；一旦 GradScaler 的动态 scale 因
fp16 溢出被压到极小值（实测降到 ``0.0156``），梯度乘上 scale 后仍小于 fp16 的
最小次正规数（``6e-8``），反向传播会把它**四舍五入成精确的 0** ——
这些参数从此再也收不到梯度（检查点里 ``exp_avg_sq`` 恒为 0），
表现为"loss 不下降、精度停在 25%"的假收敛。
bfloat16 的阶码与 fp32 相同（8 位），不存在这一问题。
"""

from __future__ import annotations

import contextlib
from typing import Optional

import torch

# 是否可用新版 AMP API
try:
    from torch.amp import GradScaler as _NewGradScaler  # type: ignore
    from torch.amp import autocast as _new_autocast    # type: ignore

    _HAS_NEW_AMP = True
except Exception:  # pragma: no cover - 旧版本 PyTorch 兜底
    _NewGradScaler = None  # type: ignore
    _HAS_NEW_AMP = False


def device_type_of(device) -> str:
    """返回设备的类型字符串（如 "cuda" / "cpu" / "mps"）。"""
    if isinstance(device, torch.device):
        return device.type
    return str(device).split(":")[0]


def bf16_natively_supported(device) -> bool:
    """判断设备是否**原生**支持 bfloat16（计算能力 >= 8.0，即 Ampere 及以后）。

    为什么要看计算能力而不是 ``torch.cuda.is_bf16_supported()``：
    新版 PyTorch 的后者在部分设备上会因为"可以用 fp32 模拟"而返回 True，
    而模拟 bf16 在 T4（7.5）上会显著拖慢训练。这里只认原生支持。
    """
    if device_type_of(device) != "cuda":
        return False
    try:
        index = device.index if isinstance(device, torch.device) and device.index is not None \
            else torch.cuda.current_device()
        return int(torch.cuda.get_device_capability(index)[0]) >= 8
    except Exception:  # pragma: no cover - 查询失败时保守回退 fp16
        return False


def resolve_amp_dtype(device, prefer: str = "auto") -> torch.dtype:
    """选择 autocast 使用的低精度类型。

    为什么默认优先 bfloat16
    ----------------------
    fp16 的最小次正规数约为 ``6e-8``：当真实梯度小于该量级（本工程在
    LayerScale 初值 ``1e-6`` + 18 个残差块的组合下，浅层 Block 的梯度量级仅 ``1e-6``
    甚至更低）时，**即使乘上 GradScaler 的动态 scale 也会下溢成精确的 0**，
    对应参数从此再也收不到梯度（检查点中表现为 ``exp_avg_sq`` 恒为 0，
    训练日志表现为 loss 不下降、精度停在随机水平）。

    bfloat16 的指数范围与 fp32 完全相同（阶码 8 位），因此不存在这一下溢问题。
    Ampere 及以后的原生支持 bf16，故默认优先 bf16；T4 等旧卡回退 fp16，
    并由 :func:`describe_amp` / 训练日志中的 ``amp_scale`` 做显式监控。
    """
    key = str(prefer).lower()
    if key in ("bf16", "bfloat16"):
        return torch.bfloat16
    if key in ("fp16", "float16", "half"):
        return torch.float16
    return torch.bfloat16 if bf16_natively_supported(device) else torch.float16


def autocast_context(device, enabled: bool = True, dtype: Optional[torch.dtype] = None):
    """返回 autocast 上下文管理器。

    仅在 CUDA 且 ``enabled=True`` 时真正启用低精度混合精度；
    其他情况（CPU / MPS / 显式关闭）返回空上下文，逻辑上完全等价于直接前向。

    参数
    ----
    dtype : torch.dtype | None
        autocast 使用的低精度类型；None 表示按 :func:`resolve_amp_dtype` 自动选择
        （默认优先 bf16，回退 fp16）。显式传入时按其执行，便于做精度对照实验。
    """
    dtype_str = device_type_of(device)
    if not enabled or dtype_str != "cuda":
        return contextlib.nullcontext()
    use_dtype = dtype if dtype is not None else resolve_amp_dtype(device)
    if _HAS_NEW_AMP:
        return _new_autocast(device_type=dtype_str, enabled=True, dtype=use_dtype)
    return torch.cuda.amp.autocast(enabled=True, dtype=use_dtype)  # type: ignore[attr-defined]


def describe_amp(device, enabled: bool, dtype: Optional[torch.dtype] = None) -> str:
    """生成人类可读的混合精度说明（用于日志与结果文件）。"""
    if not enabled or device_type_of(device) != "cuda":
        return "False（fp32 全精度）"
    use_dtype = dtype if dtype is not None else resolve_amp_dtype(device)
    name = "bfloat16" if use_dtype == torch.bfloat16 else "float16"
    return f"True（autocast {name} + GradScaler）"


def make_grad_scaler(device, enabled: bool = True) -> Optional[object]:
    """创建 ``GradScaler``；仅在 CUDA + 启用 AMP 时返回实例，否则返回 None。"""
    if not enabled or device_type_of(device) != "cuda":
        return None
    if _HAS_NEW_AMP:
        try:
            return _NewGradScaler(device="cuda", enabled=True)  # PyTorch >= 2.4
        except TypeError:  # pragma: no cover - 参数名差异兜底
            return _NewGradScaler(enabled=True)
    return torch.cuda.amp.GradScaler(enabled=True)  # type: ignore[attr-defined]


### 3. 工具模块 —— 设备与显存自适应 batch size


`src/utils/device.py`（271 行）


In [ ]:
%%writefile src/utils/device.py
# -*- coding: utf-8 -*-
"""
设备管理与**显存自适应 batch size**。

本模块解决 Colab 上最常见的一个问题：不同 GPU（T4 15GB / L4 24GB / A100 40GB，
或免费版随时可能被降级）能承受的 batch size 不同，用户写死的 batch size 很容易 OOM。

做法：在正式训练前做一次**真实的前向 + 反向 + 优化器更新**探针（dry run），
从期望的 batch size 开始，若显存不足则逐次减半，直到能够跑通；
在 ``allow_growth=True`` 时还会向上倍增，尽量吃满显存以提高吞吐。
"""

from __future__ import annotations

import gc
from typing import Callable, Optional, Tuple

import torch
import torch.nn as nn

from .amp import autocast_context, device_type_of, make_grad_scaler
from .logger import get_logger

#: 探针允许占用的“可用显存”比例上限：只有 `探针峰值 <= 可用显存 × 该比例` 才允许继续放大 batch size。
#:
#: 为什么需要留余量
#: --------------
#: Windows 的 WDDM 允许显存**过度提交**：CUDA 可以分配超过物理显存的显存（超出部分落到共享内存），
#: 于是探针可能在 8 GB 卡上“成功”跑出 10 GB 的峰值，而真实训练随即 OOM 或抛出
#: ``CUDA driver error: device not ready / Failed to create GPU mapping``。
#: 留出 30% 余量（并同时考虑桌面/其它进程已占用的显存）可以把这类误判挡在训练之前。
_PROBE_FREE_RATIO: float = 0.7


# --------------------------------------------------------------------------- #
# 设备选择
# --------------------------------------------------------------------------- #
def get_device(prefer: str = "auto") -> torch.device:
    """自动选择可用设备：CUDA > Apple MPS > CPU。

    参数
    ----
    prefer : str
        "auto" 自动选择；也可显式指定 "cuda" / "mps" / "cpu"。
    """
    if prefer and prefer != "auto":
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def describe_device(device: torch.device) -> str:
    """生成人类可读的设备描述，用于日志与报告。"""
    dtype = device_type_of(device)
    if dtype == "cuda":
        idx = device.index if device.index is not None else torch.cuda.current_device()
        name = torch.cuda.get_device_name(idx)
        total = torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)
        return f"CUDA:{idx} | {name} | 显存 {total:.1f} GB"
    if dtype == "mps":
        return "Apple MPS (Metal Performance Shaders)"
    return "CPU"


def empty_cache(device: torch.device) -> None:
    """释放 Python 侧未引用对象并清空 CUDA 显存缓存。"""
    gc.collect()
    if device_type_of(device) == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def gpu_total_memory_gb(device: torch.device) -> float:
    """返回 GPU 总显存（GB）；非 CUDA 设备返回 0。"""
    if device_type_of(device) != "cuda":
        return 0.0
    idx = device.index if device.index is not None else torch.cuda.current_device()
    return torch.cuda.get_device_properties(idx).total_memory / (1024 ** 3)


# --------------------------------------------------------------------------- #
# 显存自适应 batch size
# --------------------------------------------------------------------------- #
def _is_oom_error(err: BaseException) -> bool:
    """判断异常是否为“显存不足/显存不可用”。

    除了标准的 ``torch.cuda.OutOfMemoryError``，这里还覆盖 Windows(WDDM) 上
    显存过度提交失败时的驱动报错（``CUDA driver error: device not ready``、
    ``Failed to create GPU mapping``）。二者在语义上都是“当前 batch size 放不下”，
    因此都应该触发**下调 batch size** 而不是让训练直接崩掉。
    """
    oom_cls = getattr(torch.cuda, "OutOfMemoryError", None) if hasattr(torch, "cuda") else None
    if oom_cls is not None and isinstance(err, oom_cls):
        return True
    message = str(err).lower()
    return any(key in message for key in (
        "out of memory",
        "cuda driver error",
        "device not ready",
        "failed to create gpu mapping",
    ))


def _free_memory_mb(device) -> float:
    """返回当前可用显存（MB）；查询失败时返回 -1。"""
    try:
        free, _total = torch.cuda.mem_get_info(device)
        return float(free) / 1024 ** 2
    except Exception:  # pragma: no cover - 极少数驱动不支持
        return -1.0


def _dry_run(
    build_model_fn: Callable[[], nn.Module],
    batch_size: int,
    img_size: int,
    num_classes: int,
    device: torch.device,
    amp: bool,
) -> None:
    """用真实数据形状执行一次 前向+反向+优化器更新，以探测显存是否足够。

    探针使用与正式训练一致的模型结构、输入尺寸与精度（AMP），
    因此其显存占用可以很好地代表真实训练时的峰值占用。

    若显存不足，抛出的 OOM 异常会向上传播，由调用方捕获并下调 batch size。
    """
    model = build_model_fn().to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = make_grad_scaler(device, enabled=amp)

    images = torch.randn(batch_size, 3, img_size, img_size, device=device)
    targets = torch.randint(0, num_classes, (batch_size,), device=device)

    with autocast_context(device, enabled=amp):
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, targets)

    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    # 主动释放探针占用的资源，避免其影响后续真实训练的显存统计
    del model, optimizer, scaler, images, targets, logits, loss
    empty_cache(device)


def resolve_batch_size(
    build_model_fn: Callable[[], nn.Module],
    img_size: int,
    num_classes: int,
    device: torch.device,
    requested_batch_size: int = 128,
    min_batch_size: int = 8,
    max_batch_size: int = 512,
    amp: bool = True,
    auto: bool = True,
    logger: Optional[object] = None,
) -> int:
    """探测并返回当前设备实际可用的 batch size。

    策略
    ----
    1. 非 CUDA 设备（CPU / MPS）不存在显存不足问题，直接返回期望值；
    2. CUDA 设备上先做一次 dry run；若显存不足（含 Windows/WDDM 的
       "CUDA driver error: device not ready" 等显存不可用错误），
       则将 batch size 减半后重试，直到成功或降到 ``min_batch_size`` 为止；
    3. 若 ``auto=True``，成功后再尝试逐级放大（最细粒度到 ``max_batch_size``），
       但要求 ``探针峰值 <= 当前可用显存 × :data:`_PROBE_FREE_RATIO```，
       即**必须为真实训练留出余量** —— 否则 WDDM 的显存过度提交会让探针
       "通过"一个物理上放不下的 batch size，真实训练随即失败。

    返回
    ----
    int
        最终可用的 batch size（保证不小于 ``min_batch_size``，除非连它都跑不通，
        此时抛出 ``RuntimeError`` 提示用户减小模型或输入尺寸）。
    """
    log = logger if logger is not None else get_logger()
    log_fn = getattr(log, "info", print)

    if device_type_of(device) != "cuda":
        log_fn(f"[batch-size] 设备为 {device_type_of(device)}，无显存限制，使用 batch size = {requested_batch_size}")
        return max(1, int(requested_batch_size))

    def can_run(bs: int) -> bool:
        """尝试以 bs 跑一次探针；成功返回 True，显存放不下返回 False。"""
        try:
            _dry_run(build_model_fn, bs, img_size, num_classes, device, amp)
            return True
        except BaseException as err:  # noqa: BLE001 - 需要区分显存问题与其他异常
            if _is_oom_error(err):
                empty_cache(device)
                return False
            raise

    def probe(bs: int) -> Tuple[bool, float, float]:
        """跑一次探针，返回 ``(是否跑通, 峰值显存 MB, 探针结束后可用显存 MB)``。

        每次都先 ``empty_cache`` 并重置峰值统计，避免上一次探针的缓存/峰值污染判定
        （否则会出现"探针峰值 10856 MB > 物理显存 8 GB"这种误导性日志）。
        """
        empty_cache(device)
        torch.cuda.reset_peak_memory_stats(device)
        ok = can_run(bs)
        peak_mb = torch.cuda.max_memory_allocated(device) / 1024 ** 2
        empty_cache(device)
        return ok, peak_mb, _free_memory_mb(device)

    # ---- 1) 向下探测：找到能跑通的上界 ---- #
    bs = max(int(min_batch_size), min(int(requested_batch_size), int(max_batch_size)))
    log_fn(f"[batch-size] 开始显存探测：期望 {requested_batch_size}，候选 {bs}，"
           f"范围 [{min_batch_size}, {max_batch_size}]，AMP={amp}")
    found = False
    peak_mb = free_mb = -1.0
    while bs >= int(min_batch_size):
        ok, peak_mb, free_mb = probe(bs)
        if ok:
            found = True
            break
        next_bs = bs // 2
        log_fn(f"[batch-size] batch size = {bs} 显存不足或显存不可用，自动下调至 {next_bs}")
        bs = next_bs

    if not found:
        raise RuntimeError(
            f"即使 batch size = {min_batch_size} 仍显存不足，请换用更小的模型预设"
            f"（--model atto/femto/pico）或关闭 AMP 之外的其他开销。"
        )

    # ---- 2) 向上探测：在“留足余量”的前提下尽量放大 ---- #
    if auto:
        while bs * 2 <= int(max_batch_size):
            ok, next_peak_mb, next_free_mb = probe(bs * 2)
            if not ok:
                break
            if next_free_mb > 0 and next_peak_mb > next_free_mb * _PROBE_FREE_RATIO:
                log_fn(
                    f"[batch-size] batch size = {bs * 2} 探针峰值 {next_peak_mb:.0f} MB 超过可用显存 "
                    f"{next_free_mb:.0f} MB 的 {_PROBE_FREE_RATIO:.0%}（WDDM 允许过度提交，"
                    f"需为训练留余量），停止放大，最终采用 {bs}"
                )
                break
            bs *= 2
            peak_mb, free_mb = next_peak_mb, next_free_mb
            log_fn(f"[batch-size] 显存仍有富余，自动上调至 {bs}")

    log_fn(f"[batch-size] 最终确定 batch size = {bs}（探针峰值显存 {peak_mb:.0f} MB，"
           f"当前可用显存 {free_mb:.0f} MB，GPU 总显存 {gpu_total_memory_gb(device):.1f} GB）")
    return bs


def compute_accum_steps(target_batch_size: int, actual_batch_size: int) -> int:
    """计算梯度累积步数，使"等效 batch size"尽量接近用户期望值。

    例如期望 128、实际只能用 32，则累积 4 步，优化动态上仍接近 batch=128。
    """
    if actual_batch_size <= 0:
        return 1
    steps = int(round(target_batch_size / float(actual_batch_size)))
    return max(1, steps)


### 3. 工具模块 —— Mixup / CutMix


`src/utils/mixup.py`（137 行）


In [ ]:
%%writefile src/utils/mixup.py
# -*- coding: utf-8 -*-
"""
Mixup 与 CutMix 数据增强。

ConvNeXt 原论文的训练配方中使用了 Mixup / CutMix 来提升精度与鲁棒性：

* **Mixup**（Zhang et al., 2018）：两张图像按 Beta 分布采样得到的比例做线性插值，
  标签同样按该比例混合 —— ``x = λ·x_i + (1-λ)·x_j``。
* **CutMix**（Yun et al., 2019）：把一张图像的矩形区域直接裁剪并粘贴到另一张图像上，
  λ 由矩形面积占比决定 —— 相比 Mixup，它保留了局部真实的像素统计。

两者的共同点是：一个 batch 内构造出"软标签"，
因此损失函数需要用 :func:`mixup_criterion` 做加权求和。
"""

from __future__ import annotations

from typing import Tuple

import numpy as np
import torch
import torch.nn as nn


def rand_bbox(size: Tuple[int, int, int, int], lam: float) -> Tuple[int, int, int, int]:
    """按 λ 计算 CutMix 的矩形框坐标 (x1, y1, x2, y2)。

    参数
    ----
    size : tuple
        输入张量形状 ``(N, C, H, W)``。
    lam : float
        从 Beta 分布采样的混合系数，矩形面积占比约为 ``1 - lam``。
    """
    _, _, height, width = size
    ratio = float(np.sqrt(1.0 - lam))          # 裁剪边长比例
    cut_w = int(width * ratio)
    cut_h = int(height * ratio)

    # 矩形中心在图像内随机取点
    cx = int(np.random.randint(width))
    cy = int(np.random.randint(height))

    x1 = int(np.clip(cx - cut_w // 2, 0, width))
    y1 = int(np.clip(cy - cut_h // 2, 0, height))
    x2 = int(np.clip(cx + cut_w // 2, 0, width))
    y2 = int(np.clip(cy + cut_h // 2, 0, height))
    return x1, y1, x2, y2


def mixup(
    images: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.8,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """对 batch 内样本做 Mixup。

    返回
    ----
    mixed_images : Tensor
        混合后的图像。
    targets_a, targets_b : Tensor
        混合前的两组标签。
    lam : float
        targets_a 对应的权重（targets_b 的权重为 ``1 - lam``）。
    """
    lam = float(np.random.beta(alpha, alpha))
    # 生成 batch 内的随机置换索引
    index = torch.randperm(images.size(0), device=images.device)
    mixed = images * lam + images[index] * (1.0 - lam)
    return mixed, targets, targets[index], lam


def cutmix(
    images: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """对 batch 内样本做 CutMix。

    注意：λ 需要用"实际粘贴面积占比"重新计算，而不是直接用 Beta 采样值。
    """
    lam = float(np.random.beta(alpha, alpha))
    index = torch.randperm(images.size(0), device=images.device)

    # 先把另一张图备份出来，再做原地粘贴，避免"读到自己刚写入的内容"
    shuffled = images[index].clone()
    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images[:, :, y1:y2, x1:x2] = shuffled[:, :, y1:y2, x1:x2]

    # 根据真实粘贴面积重算 λ
    _, _, height, width = images.size()
    area = max(0, (x2 - x1)) * max(0, (y2 - y1))
    lam = 1.0 - area / float(height * width)
    return images, targets, targets[index], lam


def mixup_criterion(
    criterion: nn.Module,
    predictions: torch.Tensor,
    targets_a: torch.Tensor,
    targets_b: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    """Mixup / CutMix 的加权交叉熵损失。

    ``criterion`` 一般带标签平滑（label smoothing），两个分支都会享受到平滑，
    这也是 ConvNeXt 官方训练脚本的做法。
    """
    return lam * criterion(predictions, targets_a) + (1.0 - lam) * criterion(predictions, targets_b)


def apply_mixup_cutmix(
    images: torch.Tensor,
    targets: torch.Tensor,
    use_mixup: bool,
    prob: float,
    mixup_alpha: float = 0.8,
    cutmix_alpha: float = 1.0,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, float, bool]:
    """统一入口：以 ``prob`` 概率随机选择 Mixup 或 CutMix，否则保持原样。

    返回
    ----
    images, targets_a, targets_b, lam, applied
        ``applied=False`` 时 ``targets_a == targets_b == targets`` 且 ``lam=1.0``，
        可直接套用 :func:`mixup_criterion` 而无需分支判断。
    """
    if (not use_mixup) or (prob <= 0.0) or (np.random.rand() >= prob):
        return images, targets, targets, 1.0, False

    # 50% 概率二选一
    if np.random.rand() < 0.5:
        images, targets_a, targets_b, lam = mixup(images, targets, alpha=mixup_alpha)
    else:
        images, targets_a, targets_b, lam = cutmix(images, targets, alpha=cutmix_alpha)
    return images, targets_a, targets_b, lam, True


### 3. 工具模块 —— 测试时增强（TTA）


`src/utils/tta.py`（97 行）


In [ ]:
%%writefile src/utils/tta.py
# -*- coding: utf-8 -*-
"""
测试时增强（Test-Time Augmentation, TTA）。

原理
----
对同一批图像构造多组**确定性**的视角，分别前向得到类别概率，再取算术平均，
用平均后的概率做预测。图像分类中性价比最高的是「原图 + 水平翻转」两路 TTA：
代价约等于一次额外前向（推理时间 ×2），通常可带来 **+0.2 ~ +0.5 个百分点**的 Top-1。

与训练时增强的区别
------------------
训练时的 ``RandomHorizontalFlip`` 是**随机采样**，用于扩充数据分布；
TTA 的翻转是**成对、确定性**的：只是把"翻转前 / 翻转后"两个视角的预测平均，
不引入任何随机性，因此结果**完全可复现**。

实现细节
--------
* 概率使用 ``softmax`` 后求平均（先归一化再平均，避免不同视角 logits 尺度差异的影响）；
* 若调用方已经算过原图 logits，可通过 ``base_logits`` 传入以**避免重复前向**；
* 水平翻转对应张量第 3 维（``N, C, H, W`` 中的 ``W``）。
"""

from __future__ import annotations

from typing import Optional, Sequence

import torch
import torch.nn as nn

#: 支持的 TTA 视角（当前实现：原图 + 水平翻转）
SUPPORTED_TTA_VIEWS = ("identity", "hflip")


def hflip(x: torch.Tensor) -> torch.Tensor:
    """按宽度维做水平翻转（``N, C, H, W`` 中翻转 ``W``）。"""
    return torch.flip(x, dims=[3])


@torch.no_grad()
def tta_average_probs(
    model: nn.Module,
    images: torch.Tensor,
    base_logits: Optional[torch.Tensor] = None,
    use_hflip: bool = True,
) -> torch.Tensor:
    """计算多视角 TTA 的平均类别概率。

    参数
    ----
    model : nn.Module
        模型（调用方需保证已处于 ``eval()`` 模式）。
    images : Tensor
        输入图像，形状 ``(N, C, H, W)``。
    base_logits : Tensor | None
        原图视角的 logits；为 None 时函数内部会自行前向一次。
    use_hflip : bool
        是否加入水平翻转视角。为 False 时退化为普通的单视角预测
        （此时返回 softmax 概率，与 ``tta`` 语义一致但无增益）。

    返回
    ----
    Tensor
        形状 ``(N, num_classes)`` 的平均概率，``.sum(dim=1) ≈ 1``。

    备注
    ----
    函数内部用 ``float32`` 做 softmax 与平均，避免半精度下的数值误差累积；
    调用方可放心在 ``autocast`` 上下文中使用。
    """
    if base_logits is None:
        base_logits = model(images)
    probs = torch.softmax(base_logits.float(), dim=1)

    if use_hflip:
        flip_logits = model(hflip(images))
        probs = probs + torch.softmax(flip_logits.float(), dim=1)
        probs = probs / float(len(SUPPORTED_TTA_VIEWS))

    return probs


@torch.no_grad()
def tta_predict(
    model: nn.Module,
    images: torch.Tensor,
    use_hflip: bool = True,
) -> torch.Tensor:
    """TTA 推理并返回预测类别索引，形状 ``(N,)``。"""
    probs = tta_average_probs(model, images, base_logits=None, use_hflip=use_hflip)
    return probs.argmax(dim=1)


def describe_tta(use_hflip: bool = True) -> str:
    """返回 TTA 配置的可读描述，用于写入日志与结果文件。"""
    views: Sequence[str] = ("identity", "hflip") if use_hflip else ("identity",)
    return f"TTA views = {list(views)} (平均 softmax 概率)"


### 3. 工具模块 —— 训练曲线与混淆矩阵绘图


`src/utils/visualize.py`（279 行）


In [ ]:
%%writefile src/utils/visualize.py
# -*- coding: utf-8 -*-
"""
可视化工具：训练曲线（损失 / 准确率 / 学习率）、混淆矩阵、数据样本网格。

所有绘图函数都会把图片保存到 ``outputs/figures/``，
供报告（``06_report/``）与结果归档（``05_results/``）直接引用。
"""

from __future__ import annotations

import os
from typing import Dict, List, Optional, Sequence

import matplotlib
import numpy as np
import matplotlib.pyplot as plt

# 说明：此处**不强制**指定 matplotlib 后端，以便两种环境都能正常工作：
#   * Google Colab / Jupyter：自动使用 inline 后端，plt.show() 可内联显示图片；
#   * 无显示环境的服务器 / 后台脚本：自动回退到 Agg 后端，savefig 依然可以正常出图。


# 统一的配色
_COLOR_TRAIN = "#2E86C1"
_COLOR_VAL = "#E74C3C"
_COLOR_LR = "#27AE60"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})


def _ensure_dir(path: str) -> None:
    """确保输出图片的父目录存在。"""
    if path:
        parent = os.path.dirname(os.path.abspath(path))
        os.makedirs(parent, exist_ok=True)


def _finalize(fig, save_path: Optional[str], show: bool):
    """统一收尾：保存图片 ->（可选）显示 -> 释放图形对象，避免内存泄漏。"""
    if save_path:
        _ensure_dir(save_path)
        fig.savefig(save_path)
    if show:
        plt.show()
    else:
        plt.close(fig)


# --------------------------------------------------------------------------- #
# 1. 训练曲线
# --------------------------------------------------------------------------- #
def plot_training_curves(
    history: Dict[str, List[float]],
    save_path: Optional[str] = None,
    title: str = "ConvNeXt on CIFAR-10",
    show: bool = False,
):
    """绘制训练/验证的**损失曲线**与**准确率曲线**（外加学习率曲线）。

    参数
    ----
    history : dict
        至少包含键 ``epoch, train_loss, train_acc, val_loss, val_acc, lr``。
        ``train_acc`` / ``val_acc`` 以 0~1 或 0~100 的百分比形式给出均可。
    save_path : str | None
        保存路径（PNG）。为 None 时不保存。
    title : str
        图标题。
    show : bool
        是否调用 ``plt.show()``（在 Colab 中可显示内联图像）。

    返回
    ----
    matplotlib.figure.Figure
    """
    epochs = list(range(1, len(history.get("epoch", history.get("train_loss", []))) + 1))
    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])
    train_acc = history.get("train_acc", [])
    val_acc = history.get("val_acc", [])
    lr = history.get("lr", [])

    # 若准确率是 0~1 的小数，统一换算成百分比便于阅读
    def _to_percent(seq: Sequence[float]) -> List[float]:
        if not seq:
            return []
        return [v * 100.0 for v in seq] if max(seq) <= 1.0 else list(seq)

    train_acc_pct, val_acc_pct = _to_percent(train_acc), _to_percent(val_acc)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

    # ---- (a) 损失曲线 ---- #
    ax = axes[0]
    ax.plot(epochs, train_loss, color=_COLOR_TRAIN, marker="o", markersize=3, label="Train loss")
    if val_loss:
        ax.plot(epochs, val_loss, color=_COLOR_VAL, marker="s", markersize=3, label="Val loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.set_title("(a) Loss curves")
    ax.legend(frameon=False)

    # ---- (b) 准确率曲线 ---- #
    ax = axes[1]
    if train_acc_pct:
        ax.plot(epochs, train_acc_pct, color=_COLOR_TRAIN, marker="o", markersize=3, label="Train acc")
    if val_acc_pct:
        ax.plot(epochs, val_acc_pct, color=_COLOR_VAL, marker="s", markersize=3, label="Val acc")
        best_epoch = int(np.argmax(val_acc_pct))
        ax.scatter([epochs[best_epoch]], [val_acc_pct[best_epoch]], color="black", zorder=5, s=30)
        ax.annotate(f"best {val_acc_pct[best_epoch]:.2f}%",
                    xy=(epochs[best_epoch], val_acc_pct[best_epoch]),
                    xytext=(5, -14), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("(b) Accuracy curves")
    ax.legend(frameon=False)

    # ---- (c) 学习率曲线 ---- #
    ax = axes[2]
    if lr:
        ax.plot(epochs, lr, color=_COLOR_LR, marker="^", markersize=3)
        ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Learning rate (log scale)")
    ax.set_title("(c) LR schedule (warmup + cosine)")

    fig.suptitle(f"{title} — training curves", fontsize=13, y=1.02)
    fig.tight_layout()

    _finalize(fig, save_path, show)
    return fig


# --------------------------------------------------------------------------- #
# 2. 混淆矩阵
# --------------------------------------------------------------------------- #
def plot_confusion_matrix(
    cm: np.ndarray,
    class_names: Sequence[str],
    save_path: Optional[str] = None,
    normalize: bool = True,
    title: str = "Confusion matrix (test set)",
    show: bool = False,
):
    """绘制混淆矩阵热力图。

    参数
    ----
    cm : np.ndarray, shape (C, C)
        ``cm[i, j]`` 表示真实类别 i 被预测为类别 j 的样本数。
    class_names : Sequence[str]
        类别名称列表（长度必须为 C）。
    save_path : str | None
        保存路径。
    normalize : bool
        True 时按真实类别（行）归一化，便于比较各类召回率。
    """
    cm = np.asarray(cm, dtype=np.float64)
    if normalize:
        denom = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, np.maximum(denom, 1e-12))

    fig, ax = plt.subplots(figsize=(7.2, 6.2))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues", vmin=0.0, vmax=cm.max() if cm.size else 1.0)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label="Proportion" if normalize else "Count")

    n = len(class_names)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)

    # 在每个格子中标注数值
    thresh = cm.max() * 0.6 if cm.size else 0.5
    for i in range(n):
        for j in range(n):
            value = f"{cm[i, j]:.2f}" if normalize else f"{int(cm[i, j])}"
            ax.text(j, i, value, ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > thresh else "black")

    fig.tight_layout()
    _finalize(fig, save_path, show)
    return fig


# --------------------------------------------------------------------------- #
# 3. 数据样本网格
# --------------------------------------------------------------------------- #
def plot_data_samples(
    images,
    labels,
    class_names: Sequence[str],
    save_path: Optional[str] = None,
    mean: Optional[Sequence[float]] = None,
    std: Optional[Sequence[float]] = None,
    max_items: int = 16,
    cols: int = 8,
    show: bool = False,
):
    """把若干张训练图像拼成网格，用于报告中的"数据样例"插图。

    参数
    ----
    images : Tensor | np.ndarray
        形状 ``(N, 3, H, W)`` 的图像张量。
    labels : Tensor | np.ndarray
        对应的整数标签。
    mean, std : Sequence[float] | None
        若提供则执行反归一化，使图像颜色正确。
    """
    images = np.asarray(images)
    labels = np.asarray(labels)
    n = min(int(max_items), images.shape[0])
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(1.5 * cols, 1.6 * rows))
    axes = np.atleast_1d(axes).ravel()

    for idx in range(rows * cols):
        ax = axes[idx]
        ax.axis("off")
        if idx >= n:
            continue
        img = images[idx]
        if img.ndim == 3 and img.shape[0] == 3:      # (C, H, W) -> (H, W, C)
            img = np.transpose(img, (1, 2, 0))
        if mean is not None and std is not None:
            img = img * np.asarray(std).reshape(1, 1, 3) + np.asarray(mean).reshape(1, 1, 3)
        ax.imshow(np.clip(img, 0.0, 1.0))
        ax.set_title(class_names[int(labels[idx])], fontsize=8)

    fig.suptitle("CIFAR-10 sample images", fontsize=12)
    fig.tight_layout()
    _finalize(fig, save_path, show)
    return fig


# --------------------------------------------------------------------------- #
# 4. 逐类准确率条形图
# --------------------------------------------------------------------------- #
def plot_per_class_accuracy(
    per_class: Sequence[float],
    class_names: Sequence[str],
    save_path: Optional[str] = None,
    title: str = "Per-class accuracy (test set)",
    show: bool = False,
):
    """绘制逐类准确率条形图（可快速定位表现最差的类别）。"""
    fig, ax = plt.subplots(figsize=(8.0, 4.2))
    values = np.asarray(per_class, dtype=np.float64) * 100.0
    bars = ax.bar(range(len(values)), values, color=_COLOR_TRAIN, alpha=0.85)

    # 在柱顶标注数值
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2.0, value + 0.5, f"{value:.1f}",
                ha="center", va="bottom", fontsize=8)

    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha="right")
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 105)
    ax.set_title(f"{title} — mean {values.mean():.2f}%")
    fig.tight_layout()
    _finalize(fig, save_path, show)
    return fig


`src/data/__init__.py`（58 行）


In [ ]:
%%writefile src/data/__init__.py
# -*- coding: utf-8 -*-
"""数据模块：数据集构造、划分、预处理与 DataLoader。

结构
----
* :mod:`src.data.cifar10`   —— CIFAR-10 常量、变换与数据底座；
* :mod:`src.data.synthetic` —— 零下载合成数据（冒烟测试用）；
* :mod:`src.data.loader`    —— 划分协议、``TransformSubset`` 与 DataLoader 组装。
"""

from .cifar10 import (
    ARCHIVE_MD5,
    ARCHIVE_NAME,
    ARCHIVE_SIZE,
    BASE_FOLDER,
    CIFAR10_CLASSES,
    CIFAR10_MEAN,
    CIFAR10_STD,
    IMAGE_SIZE,
    build_cifar10_base,
    build_test_transform,
    build_train_transform,
    is_extracted,
)
from .loader import (
    OFFICIAL_TEST_SIZE,
    OFFICIAL_TRAIN_SIZE,
    TransformSubset,
    build_base_datasets,
    build_dataloaders,
    build_datasets,
)
from .synthetic import SyntheticCIFAR, build_synthetic_dataset

__all__ = [
    # 常量
    "ARCHIVE_MD5",
    "ARCHIVE_NAME",
    "ARCHIVE_SIZE",
    "BASE_FOLDER",
    "CIFAR10_CLASSES",
    "CIFAR10_MEAN",
    "CIFAR10_STD",
    "IMAGE_SIZE",
    "OFFICIAL_TEST_SIZE",
    "OFFICIAL_TRAIN_SIZE",
    # 数据集
    "SyntheticCIFAR",
    "TransformSubset",
    "build_base_datasets",
    "build_cifar10_base",
    "build_dataloaders",
    "build_datasets",
    "build_synthetic_dataset",
    "build_test_transform",
    "build_train_transform",
    "is_extracted",
]


### 4. 数据模块 —— CIFAR-10 常量、变换与数据底座


`src/data/cifar10.py`（152 行）


In [ ]:
%%writefile src/data/cifar10.py
# -*- coding: utf-8 -*-
"""
CIFAR-10 常量、预处理变换与"数据底座"构造。

本模块只负责三件事：

1. 定义 CIFAR-10 的类别名与归一化统计量（:data:`CIFAR10_CLASSES` 等）；
2. 定义训练 / 评估两套**确定性可控**的变换流水线；
3. 构造**不带变换**的数据底座（:func:`build_cifar10_base`）。

数据划分、子集抽取与 DataLoader 组装见 :mod:`src.data.loader`。

为什么把"底座"与"变换"解耦
---------------------------
``torchvision.datasets.CIFAR10`` 在 ``__init__`` 时会把 ``transform`` 固定下来，
并且**每次构造**都会执行两类代价不低的操作：

* **完整性校验**：对全部 6 个批次文件（共约 186 MB）逐个做 MD5；
* **pickle 反序列化**：训练集约 146 MB 数组读入内存。

训练集需要"带增强"和"不带增强"两个版本。若各构造一个 ``CIFAR10`` 实例，
上述两项开销都会翻倍（实测：热路径 4.5 s 里几乎全部由重复 MD5 校验贡献）。

因此本工程改为：**每个 split 只构造一个底座**（``transform=None``），
再用 :class:`src.data.loader.TransformSubset` 叠加不同变换，
既保证划分与索引语义**完全不变**，又消除重复校验与重复加载。
"""

from __future__ import annotations

import os
from typing import List, Sequence, Tuple

from torch.utils.data import Dataset
from torchvision import datasets, transforms

# --------------------------------------------------------------------------- #
# 常量
# --------------------------------------------------------------------------- #
CIFAR10_CLASSES: List[str] = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# CIFAR-10 训练集在 [0, 1] 尺度下统计得到的通道均值与标准差
CIFAR10_MEAN: Tuple[float, float, float] = (0.4914, 0.4822, 0.4465)
CIFAR10_STD: Tuple[float, float, float] = (0.2470, 0.2435, 0.2616)

#: CIFAR-10 图像尺寸
IMAGE_SIZE: int = 32

#: torchvision 解压后生成的目录名（用于判断数据是否已就绪）
BASE_FOLDER: str = "cifar-10-batches-py"

#: 官方压缩包文件名与 MD5（用于下载加速脚本与文档引用）
ARCHIVE_NAME: str = "cifar-10-python.tar.gz"
ARCHIVE_MD5: str = "c58f30108f718f92721af3b95e74349a"
ARCHIVE_SIZE: int = 170498071


# --------------------------------------------------------------------------- #
# 变换
# --------------------------------------------------------------------------- #
def build_train_transform(
    mean: Sequence[float] = CIFAR10_MEAN,
    std: Sequence[float] = CIFAR10_STD,
    rand_augment: bool = False,
    random_erase: float = 0.25,
) -> transforms.Compose:
    """构建训练集的预处理与数据增强流水线。

    参数
    ----
    mean, std : Sequence[float]
        归一化使用的通道均值与标准差。
    rand_augment : bool
        是否叠加 RandAugment（更激进的自动增强，小数据集上可能过强，默认关闭）。
    random_erase : float
        RandomErasing 的触发概率；<=0 表示关闭该增强。
    """
    ops: List[object] = [
        # CIFAR 图像的填充方式使用 reflect 效果通常优于默认的常量填充
        transforms.RandomCrop(IMAGE_SIZE, padding=4, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
    ]
    if rand_augment:
        ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
    ops += [
        transforms.ToTensor(),                      # HWC uint8 -> CHW float，并缩放到 [0, 1]
        transforms.Normalize(mean, std),            # 标准化到零均值、单位方差
    ]
    if random_erase is not None and random_erase > 0:
        # RandomErasing 必须在 Normalize 之后（它作用在 Tensor 上，单位已归一化）
        ops.append(transforms.RandomErasing(p=float(random_erase), scale=(0.02, 0.2), value=0.0))
    return transforms.Compose(ops)


def build_test_transform(
    mean: Sequence[float] = CIFAR10_MEAN,
    std: Sequence[float] = CIFAR10_STD,
) -> transforms.Compose:
    """构建验证/测试集的确定性预处理（仅 ToTensor + Normalize，不做增强）。"""
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


# --------------------------------------------------------------------------- #
# 数据底座
# --------------------------------------------------------------------------- #
def is_extracted(root: str) -> bool:
    """判断 CIFAR-10 是否已解压就绪（不触发任何校验或下载）。"""
    return os.path.isfile(os.path.join(root, BASE_FOLDER, "test_batch"))


def build_cifar10_base(
    cfg,
    train: bool,
    download: bool = False,
    logger=None,
) -> Dataset:
    """构造一个**不带变换**的 CIFAR-10 底座数据集。

    参数
    ----
    cfg : Config
        全局配置对象（读取 ``data.data_root``）。
    train : bool
        True 取官方训练集（50000 张），False 取官方测试集（10000 张）。
    download : bool
        数据缺失时是否允许下载。调用方应先用 :func:`is_extracted` 判断缓存情况，
        **只有在确实缺失时才传 True**，以避免每次都触发下载分支中的额外校验。
    logger : Logger | None
        日志器，用于打印数据目录与是否复用缓存。

    返回
    ----
    Dataset
        返回的样本为 ``(PIL.Image, int)``，变换由调用方（``TransformSubset``）叠加。
    """
    root = cfg.data.data_root
    if download:
        os.makedirs(root, exist_ok=True)

    if logger is not None:
        logger.info(
            f"[data] 加载 CIFAR-10 {'训练集' if train else '测试集'}底座"
            f"（transform=None，目录 {os.path.abspath(root)}）"
        )

    return datasets.CIFAR10(root=root, train=train, download=download, transform=None)


`src/models/__init__.py`（22 行）


In [ ]:
%%writefile src/models/__init__.py
# -*- coding: utf-8 -*-
"""模型模块：ConvNeXt（CVPR 2022）的 PyTorch 实现。"""

from .convnext import (
    CONVNEXT_PRESETS,
    ConvNeXt,
    ConvNeXtBlock,
    ConvNeXtDownsample,
    LayerNorm,
    build_convnext,
    convnext_model_summary,
)

__all__ = [
    "CONVNEXT_PRESETS",
    "ConvNeXt",
    "ConvNeXtBlock",
    "ConvNeXtDownsample",
    "LayerNorm",
    "build_convnext",
    "convnext_model_summary",
]


### 5. 模型模块 —— ConvNeXt（CVPR 2022）


`src/models/convnext.py`（389 行）


In [ ]:
%%writefile src/models/convnext.py
# -*- coding: utf-8 -*-
"""
ConvNeXt 模型定义（适配 CIFAR-10 的 32x32 输入）。

参考文献
--------
Zhuang Liu, Hanzi Mao, Chao-Yuan Wu, Christoph Feichtenhofer, Trevor Darrell, Saining Xie.
"A ConvNet for the 2020s". CVPR 2022. arXiv:2201.03545.
官方实现: https://github.com/facebookresearch/ConvNeXt

论文关键设计（本实现全部保留）
------------------------------
1. **宏观结构**：四个阶段，堆叠比例由 ResNet 的 (3,4,6,3) 改为 (3,3,9,3)，通道数逐阶段翻倍。
2. **卷积式 Stem**：用卷积（而非 Transformer 的 patch embedding + attention）完成 patchify。
3. **倒置瓶颈（Inverted Bottleneck）**：``1x1 升维 4 倍 -> 深度卷积 -> 1x1 降维``。
4. **大卷积核深度卷积**：深度卷积核为 **7x7**，以扩大有效感受野。
5. **ResNeXt 化**：深度卷积（groups = channels）替代普通卷积，显著降低计算量。
6. **微观设计**：GELU 替代 ReLU、减少激活层数量（每个 Block 仅 1 个 GELU）、
   LayerNorm 替代 BatchNorm、阶段之间使用**独立的下采样层**（LayerNorm + 2x2 卷积）。
7. **训练稳定性技巧**：Layer Scale（可学习缩放，初值 1e-6）与 Stochastic Depth（DropPath）。

针对 CIFAR-10 的适配
--------------------
ImageNet 的输入为 224x224，论文 Stem 为 4x4、stride=4 的 patchify 卷积，
特征图序列为 56 -> 28 -> 14 -> 7。
CIFAR-10 图像只有 32x32，若沿用 stride=4，特征图会迅速缩到 8 -> 4 -> 2 -> 1，
空间信息损失过大。因此本实现默认使用 **3x3、stride=1 的卷积式 Stem**
（``stem_mode="conv"``），四阶段分辨率变为 32 -> 16 -> 8 -> 4，
在保留论文全部关键模块的前提下适配小分辨率输入。
将 ``stem_mode`` 设为 ``"patchify"`` 即可切换回论文原版 4x4/s4 的 Stem。

速度与精度的取舍：模型容量主要由 ``depths``（每个阶段的 Block 数）与 ``dims``（通道数）
决定，可通过 :data:`CONVNEXT_PRESETS` 中的不同规模进行消融实验。
"""

from __future__ import annotations

from typing import Dict, List, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.init import trunc_normal_


# --------------------------------------------------------------------------- #
# 1. 基础组件
# --------------------------------------------------------------------------- #
class LayerNorm(nn.Module):
    """支持 ``channels_last`` 与 ``channels_first`` 两种数据格式的 LayerNorm。

    PyTorch 原生 ``nn.LayerNorm`` 默认只对最后一维做归一化（即 channels_last）。
    ConvNeXt 模型中张量主体是 ``(N, C, H, W)`` 的 channels_first 格式，
    因此这里提供 channels_first 的等价实现（对通道维做归一化）。
    """

    def __init__(self, normalized_shape: int, eps: float = 1e-6, data_format: str = "channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ("channels_last", "channels_first"):
            raise NotImplementedError(f"不支持的 data_format: {self.data_format}")
        self.normalized_shape = (normalized_shape,)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        # channels_first：手写归一化，避免频繁 permute 带来的额外拷贝
        mean = x.mean(1, keepdim=True)
        var = (x - mean).pow(2).mean(1, keepdim=True)
        x = (x - mean) / torch.sqrt(var + self.eps)
        x = self.weight[:, None, None] * x + self.bias[:, None, None]
        return x


def drop_path(x: torch.Tensor, drop_prob: float = 0.0, training: bool = False) -> torch.Tensor:
    """Stochastic Depth（逐样本随机丢弃整个残差分支）。

    论文写法（亦见 ``timm`` 实现）：训练时以 ``drop_prob`` 的概率把某个样本的
    残差分支整体置零，并把保留下来的样本除以 ``1 - drop_prob`` 以保持期望不变。
    """
    if drop_prob <= 0.0 or not training:
        return x
    keep_prob = 1.0 - drop_prob
    # 每个样本一个独立的伯努利掩码，其余维度广播
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    mask = x.new_empty(shape).bernoulli_(keep_prob)
    return x.div(keep_prob) * mask


class DropPath(nn.Module):
    """``drop_path`` 的 ``nn.Module`` 封装。"""

    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return drop_path(x, self.drop_prob, self.training)

    def extra_repr(self) -> str:
        return f"drop_prob={self.drop_prob:.4f}"


class ConvNeXtBlock(nn.Module):
    """ConvNeXt 的基本模块：深度卷积 + 倒置瓶颈（Inverted Bottleneck）。

    结构（对应论文 Figure 4 的最终设计）::

        x ---------------------------------- (残差)
        |
        DepthwiseConv2d(7x7, groups=C)   <- 大卷积核，扩大感受野
        LayerNorm (channels_last)
        Linear(C -> 4C)                  <- 倒置瓶颈：升维 4 倍
        GELU                             <- 全块仅有 1 个激活函数
        Linear(4C -> C)                  <- 降维回原通道数
        Layer Scale (gamma)
        |
        + ---------------------------------- (相加)

    参数
    ----
    dim : int
        输入/输出通道数。
    drop_path : float
        该 Block 的 Stochastic Depth 概率。
    layer_scale_init_value : float
        Layer Scale 初始值；<=0 表示不使用 Layer Scale。
    kernel_size : int
        深度卷积核大小，论文为 7。
    """

    def __init__(
        self,
        dim: int,
        drop_path: float = 0.0,
        layer_scale_init_value: float = 1e-6,
        kernel_size: int = 7,
    ):
        super().__init__()
        # 7x7 深度卷积：groups=dim，参数量仅 dim*k*k
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=kernel_size,
                                padding=kernel_size // 2, groups=dim)
        self.norm = LayerNorm(dim, eps=1e-6, data_format="channels_last")
        # 倒置瓶颈：升维 4 倍，再降维（用 Linear 等价于 1x1 卷积）
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = (
            nn.Parameter(layer_scale_init_value * torch.ones(dim))
            if layer_scale_init_value > 0 else None
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.dwconv(x)                                  # (N, C, H, W)
        x = x.permute(0, 2, 3, 1)                           # -> (N, H, W, C)，便于用 LayerNorm/Linear
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)                           # -> (N, C, H, W)
        return residual + self.drop_path(x)


class ConvNeXtDownsample(nn.Module):
    """阶段之间的**独立下采样层**：LayerNorm + 2x2、stride=2 卷积。

    论文指出：使用独立的下采样层（而非像 ResNet 那样在残差分支首层做 stride=2）
    可以进一步提升精度。
    """

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.norm = LayerNorm(in_dim, eps=1e-6, data_format="channels_first")
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(self.norm(x))


# --------------------------------------------------------------------------- #
# 2. 模型预设
# --------------------------------------------------------------------------- #
#: 不同规模的 ConvNeXt 预设。
#: ``tiny`` 对应论文 ConvNeXt-T 的原始配置 (depths=(3,3,9,3), dims=(96,192,384,768))；
#: 其余为更轻量的规模（源自 ConvNeXt V2 的设计），便于在 Colab 免费 GPU 上快速验证。
CONVNEXT_PRESETS: Dict[str, Dict[str, Sequence[int]]] = {
    "atto":  {"depths": (2, 2, 6, 2), "dims": (40, 80, 160, 320)},
    "femto": {"depths": (2, 2, 6, 2), "dims": (48, 96, 192, 384)},
    "pico":  {"depths": (2, 2, 6, 2), "dims": (64, 128, 256, 512)},
    "nano":  {"depths": (2, 2, 8, 2), "dims": (80, 160, 320, 640)},
    "tiny":  {"depths": (3, 3, 9, 3), "dims": (96, 192, 384, 768)},
}


# --------------------------------------------------------------------------- #
# 3. 主模型
# --------------------------------------------------------------------------- #
class ConvNeXt(nn.Module):
    """面向 CIFAR-10 的 ConvNeXt 分类网络（纯卷积，不含任何注意力机制）。

    参数
    ----
    in_chans : int
        输入通道数，RGB 图像为 3。
    num_classes : int
        分类类别数，CIFAR-10 为 10。
    depths : Sequence[int]
        每个阶段的 Block 数量，论文为 (3, 3, 9, 3)。
    dims : Sequence[int]
        每个阶段的通道数，论文 Tiny 为 (96, 192, 384, 768)。
    drop_path_rate : float
        Stochastic Depth 的最大丢弃率；各 Block 的概率由 0 线性递增到该值。
    layer_scale_init_value : float
        Layer Scale 的初始值。
    stem_mode : str
        ``"conv"``   : 3x3、stride=1 卷积 Stem（适配 32x32，默认）；
        ``"patchify"``: 4x4、stride=4 卷积 Stem（论文原版，适用于大分辨率输入）。
    head_init_scale : float
        分类头初始权重的缩放系数，用于稳定训练早期的输出尺度。
    """

    def __init__(
        self,
        in_chans: int = 3,
        num_classes: int = 10,
        depths: Sequence[int] = (3, 3, 9, 3),
        dims: Sequence[int] = (96, 192, 384, 768),
        drop_path_rate: float = 0.0,
        layer_scale_init_value: float = 1e-6,
        stem_mode: str = "conv",
        head_init_scale: float = 1.0,
    ):
        super().__init__()
        assert len(depths) == 4 and len(dims) == 4, "ConvNeXt 需要 4 个阶段"
        self.depths = list(depths)
        self.dims = list(dims)
        self.stem_mode = stem_mode

        # ---------------- Stem / 下采样层 ---------------- #
        self.downsample_layers = nn.ModuleList()
        if stem_mode == "patchify":
            # 论文原版：4x4、stride=4 卷积完成 patchify
            stem = nn.Sequential(
                nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4),
                LayerNorm(dims[0], eps=1e-6, data_format="channels_first"),
            )
        elif stem_mode == "conv":
            # 小分辨率适配版：3x3、stride=1 卷积，保持空间分辨率
            stem = nn.Sequential(
                nn.Conv2d(in_chans, dims[0], kernel_size=3, stride=1, padding=1),
                LayerNorm(dims[0], eps=1e-6, data_format="channels_first"),
            )
        else:
            raise ValueError(f"未知的 stem_mode: {stem_mode}（可选 'conv' / 'patchify'）")

        self.downsample_layers.append(stem)
        for stage_idx in range(3):
            self.downsample_layers.append(
                ConvNeXtDownsample(dims[stage_idx], dims[stage_idx + 1])
            )

        # ---------------- 四个阶段的 Block ---------------- #
        # Stochastic Depth：概率在各 Block 之间从 0 线性递增到 drop_path_rate
        total_blocks = int(sum(depths))
        dp_rates = torch.linspace(0.0, float(drop_path_rate), total_blocks).tolist()

        self.stages = nn.ModuleList()
        cursor = 0
        for stage_idx in range(4):
            blocks = [
                ConvNeXtBlock(
                    dim=dims[stage_idx],
                    drop_path=dp_rates[cursor + block_idx],
                    layer_scale_init_value=layer_scale_init_value,
                )
                for block_idx in range(depths[stage_idx])
            ]
            self.stages.append(nn.Sequential(*blocks))
            cursor += depths[stage_idx]

        # ---------------- 分类头 ---------------- #
        # 官方实现用 GroupNorm(1, C) 作为末端归一化，等价于对每个样本做 channels_first LayerNorm
        self.norm = nn.GroupNorm(1, dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)

        # 权重初始化
        self.apply(self._init_weights)
        # 分类头单独缩放（论文中用于稳定输出尺度）
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.constant_(self.head.bias, 0.0)
        self.head.weight.data.mul_(head_init_scale)
        self.head.bias.data.mul_(head_init_scale)

    # ------------------------------------------------------------------ #
    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        """按论文/官方实现初始化权重：截断正态 N(0, 0.02)，偏置置零。"""
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0.0)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """提取特征图（返回下采样后的最后一层特征，形状 ``(N, C, H, W)``）。"""
        for stage_idx in range(4):
            x = self.downsample_layers[stage_idx](x)
            x = self.stages[stage_idx](x)
        return self.norm(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """前向传播：特征提取 -> 全局平均池化 -> 线性分类头。"""
        x = self.forward_features(x)
        x = x.mean(dim=(-2, -1))          # 全局平均池化 (N, C, H, W) -> (N, C)
        return self.head(x)               # (N, num_classes)


# --------------------------------------------------------------------------- #
# 4. 工厂函数
# --------------------------------------------------------------------------- #
def build_convnext(model_cfg, num_classes: int = 10) -> ConvNeXt:
    """根据 :class:`configs.ModelConfig` 构建 ConvNeXt 模型。

    若 ``model_cfg.depths`` / ``model_cfg.dims`` 为 None，则使用 ``model_cfg.name``
    对应的预设；否则以显式指定的结构为准（便于做结构消融实验）。

    参数
    ----
    model_cfg : ModelConfig
        模型配置对象。
    num_classes : int
        分类类别数。

    返回
    ----
    ConvNeXt
    """
    name = str(model_cfg.name).lower()
    if model_cfg.depths is not None and model_cfg.dims is not None:
        depths: Tuple[int, ...] = tuple(model_cfg.depths)
        dims: Tuple[int, ...] = tuple(model_cfg.dims)
    else:
        if name not in CONVNEXT_PRESETS:
            raise KeyError(f"未知的 ConvNeXt 预设 '{name}'，可选：{list(CONVNEXT_PRESETS)}")
        preset = CONVNEXT_PRESETS[name]
        depths = tuple(preset["depths"])
        dims = tuple(preset["dims"])

    return ConvNeXt(
        in_chans=3,
        num_classes=num_classes,
        depths=depths,
        dims=dims,
        drop_path_rate=float(model_cfg.drop_path_rate),
        layer_scale_init_value=float(model_cfg.layer_scale_init_value),
        stem_mode=str(model_cfg.stem_mode),
        head_init_scale=float(model_cfg.head_init_scale),
    )


def convnext_model_summary(model_cfg, num_classes: int = 10, input_size: int = 32) -> str:
    """生成模型结构摘要字符串（含各阶段维度与参数量），用于日志与报告。"""
    name = str(model_cfg.name).lower()
    if model_cfg.depths is not None and model_cfg.dims is not None:
        depths, dims = list(model_cfg.depths), list(model_cfg.dims)
    else:
        preset = CONVNEXT_PRESETS[name]
        depths, dims = list(preset["depths"]), list(preset["dims"])

    lines = [
        f"ConvNeXt-{name} | stem={model_cfg.stem_mode} | drop_path={model_cfg.drop_path_rate}",
        f"  input            : 3 x {input_size} x {input_size}",
    ]
    resolution = input_size if model_cfg.stem_mode == "conv" else input_size // 4
    for stage_idx in range(4):
        lines.append(
            f"  stage {stage_idx + 1}          : {resolution:>2} x {resolution:>2} x {dims[stage_idx]:<4}"
            f" | blocks={depths[stage_idx]}"
        )
        if stage_idx < 3:
            resolution = max(1, resolution // 2)
    lines.append(f"  head             : global average pool -> Linear({dims[-1]} -> {num_classes})")
    return "\n".join(lines)


`src/engine/__init__.py`（20 行）


In [ ]:
%%writefile src/engine/__init__.py
# -*- coding: utf-8 -*-
"""训练与评估引擎。"""

from .evaluator import (
    analyze_predictions,
    collect_predictions,
    evaluate,
    topk_accuracy,
)
from .trainer import Trainer, build_optimizer, build_scheduler

__all__ = [
    "Trainer",
    "build_optimizer",
    "build_scheduler",
    "evaluate",
    "collect_predictions",
    "analyze_predictions",
    "topk_accuracy",
]


### 6. 评估模块 —— Top-k 准确率 / 逐类指标 / 混淆矩阵


`src/engine/evaluator.py`（288 行）


In [ ]:
%%writefile src/engine/evaluator.py
# -*- coding: utf-8 -*-
"""
测试 / 验证评估模块。

包含四类功能：
1. :func:`topk_accuracy` —— Top-k 准确率计算（k=1,5）；
2. :func:`evaluate`      —— 在给定 DataLoader 上计算平均损失与 Top-1/Top-5 准确率；
3. :func:`collect_predictions` —— 收集全部预测结果与标签（用于混淆矩阵、误差分析）；
4. :func:`analyze_predictions` —— 逐类准确率、混淆矩阵、分类报告，并落盘为 JSON。
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from ..utils.amp import autocast_context
from ..utils.misc import AverageMeter, save_json
from ..utils.tta import tta_average_probs


# --------------------------------------------------------------------------- #
# 1. Top-k 准确率
# --------------------------------------------------------------------------- #
@torch.no_grad()
def topk_accuracy(
    outputs: torch.Tensor,
    targets: torch.Tensor,
    topk: Sequence[int] = (1, 5),
) -> List[float]:
    """计算一个 batch 的 Top-k 准确率（百分比，0~100）。

    参数
    ----
    outputs : Tensor, shape (N, C)
        模型输出的 logits（未经过 softmax）。
    targets : Tensor, shape (N,)
        真实标签。
    topk : Sequence[int]
        需要计算的 k 列表。

    返回
    ----
    List[float]
        与 ``topk`` 一一对应的准确率（百分数）。

    实现细节
    --------
    借助 ``topk`` + ``eq`` 的广播比较，一次性算出所有 k，避免多次排序。
    当类别数小于 max(k) 时（如 CIFAR-10 只有 10 类），会自动把 k 收敛到有效范围。
    """
    num_classes = outputs.size(1)
    effective_k = sorted({min(int(k), num_classes) for k in topk})
    max_k = max(effective_k)

    batch_size = targets.size(0)
    _, pred = outputs.topk(max_k, dim=1, largest=True, sorted=True)   # (N, max_k)
    pred = pred.t()                                                   # (max_k, N)
    correct = pred.eq(targets.view(1, -1).expand_as(pred))            # (max_k, N) bool

    results: Dict[int, float] = {}
    for k in effective_k:
        correct_k = correct[:k].reshape(-1).float().sum(dim=0)
        results[k] = float(correct_k.mul_(100.0 / max(1, batch_size)).item())
    return [results[min(int(k), num_classes)] for k in topk]


# --------------------------------------------------------------------------- #
# 2. 通用评估
# --------------------------------------------------------------------------- #
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    amp: bool = True,
    topk: Sequence[int] = (1, 5),
    max_batches: Optional[int] = None,
    tta: bool = False,
    tta_hflip: bool = True,
) -> Dict[str, float]:
    """在给定 DataLoader 上评估模型。

    参数
    ----
    model : nn.Module
        待评估模型（函数内部会切换到 ``eval()`` 模式）。
    loader : DataLoader
        验证集或测试集的数据加载器。
    criterion : nn.Module
        损失函数（一般与训练一致，带标签平滑）。
    device : torch.device
        计算设备。
    amp : bool
        是否启用混合精度推理（仅 CUDA 生效，可显著加速）。
    topk : Sequence[int]
        需要统计的 Top-k 指标。
    max_batches : int | None
        最多评估多少个 batch（用于快速冒烟测试）；None 表示全量评估。
    tta : bool
        是否启用**测试时增强**（原图 + 水平翻转两路，平均 softmax 概率后再取 Top-k）。
        推理耗时约翻倍，通常可带来 +0.2 ~ +0.5 个百分点的 Top-1。
    tta_hflip : bool
        TTA 是否包含水平翻转视角（``tta=True`` 时生效）。

   返回
    ----
    dict
        ``{"loss": float, "top1": float, "top5": float, "num_samples": int}``，
        其中准确率为百分数。

    说明
    ----
    为避免"损失与准确率口径不一致"，``loss`` 始终由**不做 TTA 的原始前向**计算
    （与训练期验证、非 TTA 评估严格可比）；只有 Top-k 准确率使用 TTA 平均概率。
    返回值中另含 ``tta`` 字段用于标记本次评估是否启用了 TTA。
    """
    model.eval()
    loss_meter = AverageMeter("loss")
    topk_meters = {int(k): AverageMeter(f"top{k}") for k in topk}

    for batch_idx, (images, targets) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True).long()

        with autocast_context(device, enabled=amp):
            logits = model(images)
            loss = criterion(logits, targets)
            if tta:
                # 传入已算好的原图 logits，避免重复前向
                scores = tta_average_probs(model, images, base_logits=logits, use_hflip=tta_hflip)
            else:
                scores = logits

        batch_size = targets.size(0)
        loss_meter.update(loss.item(), batch_size)
        accs = topk_accuracy(scores.float(), targets, topk=topk)
        for k, acc in zip(topk, accs):
            topk_meters[min(int(k), scores.size(1))].update(acc, batch_size)

    metrics: Dict[str, float] = {
        "loss": loss_meter.avg,
        "num_samples": loss_meter.count,
        "tta": 1.0 if tta else 0.0,
    }
    for k, meter in topk_meters.items():
        metrics[f"top{k}"] = meter.avg
    return metrics


# --------------------------------------------------------------------------- #
# 3. 收集预测结果
# --------------------------------------------------------------------------- #
@torch.no_grad()
def collect_predictions(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    amp: bool = True,
    max_batches: Optional[int] = None,
    tta: bool = False,
    tta_hflip: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """遍历数据集并收集所有预测标签与真实标签。

    参数
    ----
    tta : bool
        是否启用测试时增强（与原图 + 水平翻转的平均概率保持一致），
        需与 :func:`evaluate` 的 ``tta`` 参数取相同值，否则混淆矩阵与总体准确率口径不一致。
    tta_hflip : bool
        TTA 是否包含水平翻转视角。

    返回
    ----
    (y_true, y_pred) : tuple[np.ndarray, np.ndarray]
        两个形状均为 ``(N,)`` 的整数数组。
    """
    model.eval()
    all_preds: List[np.ndarray] = []
    all_targets: List[np.ndarray] = []

    for batch_idx, (images, targets) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        images = images.to(device, non_blocking=True)
        with autocast_context(device, enabled=amp):
            logits = model(images)
            if tta:
                scores = tta_average_probs(model, images, base_logits=logits, use_hflip=tta_hflip)
            else:
                scores = logits
        preds = scores.float().argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(targets.numpy())

    if not all_preds:
        return np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64)
    return np.concatenate(all_targets), np.concatenate(all_preds)


# --------------------------------------------------------------------------- #
# 4. 预测结果分析
# --------------------------------------------------------------------------- #
def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int) -> np.ndarray:
    """计算混淆矩阵（不依赖 scikit-learn）。

    返回
    ----
    np.ndarray, shape (C, C)
        ``cm[i, j]`` = 真实类别 i 被预测为 j 的样本数。
    """
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true.astype(np.int64), y_pred.astype(np.int64)):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1
    return cm


def per_class_accuracy(cm: np.ndarray) -> np.ndarray:
    """由混淆矩阵计算逐类准确率（即召回率）。"""
    row_sum = cm.sum(axis=1).astype(np.float64)
    correct = np.diag(cm).astype(np.float64)
    return np.divide(correct, np.maximum(row_sum, 1e-12))


def analyze_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: Sequence[str],
    logger=None,
    save_path: Optional[str] = None,
) -> Dict[str, object]:
    """汇总预测结果：总体准确率、逐类准确率、混淆矩阵。

    参数
    ----
    y_true, y_pred : np.ndarray
        :func:`collect_predictions` 的输出。
    class_names : Sequence[str]
        类别名称。
    logger : Logger | None
        日志器，若提供则打印每类准确率。
    save_path : str | None
        结果 JSON 的保存路径。

    返回
    ----
    dict
        包含 ``accuracy / per_class_accuracy / confusion_matrix / macro_accuracy`` 等字段。
    """
    num_classes = len(class_names)
    total = int(len(y_true))
    correct = int((y_true == y_pred).sum())
    overall_acc = 100.0 * correct / max(1, total)

    cm = confusion_matrix(y_true, y_pred, num_classes)
    per_class = per_class_accuracy(cm)

    result: Dict[str, object] = {
        "num_samples": total,
        "num_correct": correct,
        "accuracy": overall_acc,
        "macro_accuracy": float(np.mean(per_class) * 100.0),
        "per_class_accuracy": {name: float(per_class[i]) for i, name in enumerate(class_names)},
        "confusion_matrix": cm.tolist(),
        "class_names": list(class_names),
    }

    if logger is not None:
        logger.info("逐类准确率（测试集）：")
        for i, name in enumerate(class_names):
            logger.info(f"    {name:<11s}: {per_class[i] * 100.0:6.2f}%  "
                        f"(支撑样本 {int(cm[i].sum())} 张)")
        logger.info(f"宏平均（各类准确率均值）：{np.mean(per_class) * 100.0:.2f}%")

    if save_path:
        save_json(result, save_path)

    return result


### 7. 训练模块 —— 训练循环与检查点管理


`src/engine/trainer.py`（553 行）


In [ ]:
%%writefile src/engine/trainer.py
# -*- coding: utf-8 -*-
"""
训练引擎。

本模块实现 ConvNeXt 在 CIFAR-10 上的完整训练循环，包含：

* **优化器**：AdamW，并对权重为 1 维的参数（LayerNorm/BatchNorm 的 weight/bias、
  卷积与线性层的 bias）不施加权重衰减 —— 这是 ConvNeXt 官方训练脚本的做法；
* **学习率调度**：线性热身（warmup）+ 余弦退火（cosine decay）；
* **混合精度训练（AMP）**：CUDA 上使用 fp16 + GradScaler；
* **梯度累积**：显存不足导致 batch size 被自动调小后，
  用累积步数维持与期望 batch size 接近的优化动态；
* **梯度裁剪**：抑制偶发的梯度爆炸；
* **Mixup / CutMix**：论文训练配方中的强增强；
* **检查点管理**：保存最优与最后一次的模型权重，支持断点续训；
* **训练历史记录**：逐轮记录损失 / 准确率 / 学习率，供可视化与报告使用。
"""

from __future__ import annotations

import math
import os
import time
from typing import Dict, List, Optional

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader

from ..utils.amp import autocast_context, describe_amp, make_grad_scaler, resolve_amp_dtype
from ..utils.csv_logger import write_history_csv
from ..utils.misc import AverageMeter, human_time, save_json
from ..utils.mixup import apply_mixup_cutmix, mixup_criterion
from ..utils.visualize import plot_training_curves
from .evaluator import evaluate

#: GradScaler 动态 scale 的安全下限。低于该值说明 fp16 已频繁溢出、
#: scale 被压到极小，梯度继续乘 scale 后将大面积下溢为 0（详见 src/utils/amp.py）。
AMP_SCALE_FLOOR: float = 1.0


# --------------------------------------------------------------------------- #
# 优化器与学习率调度
# --------------------------------------------------------------------------- #
def build_optimizer(model: nn.Module, lr: float, weight_decay: float, betas=(0.9, 0.999)) -> torch.optim.AdamW:
    """构建 AdamW 优化器，并对参数做**权重衰减分组**。

    为 1 维参数（如 LayerNorm 的 weight/bias、所有 bias）单独分组，
    其 ``weight_decay=0``；其余（卷积核、线性层权重）使用设定的权重衰减。
    这一细节对 ConvNeXt 的最终精度有可观影响。
    """
    decay_params: List[nn.Parameter] = []
    no_decay_params: List[nn.Parameter] = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        # 1 维张量必然是 norm 层的缩放/偏移或 bias，不应施加权重衰减
        if param.ndim <= 1 or name.endswith(".bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    param_groups = [
        {"params": decay_params, "weight_decay": float(weight_decay)},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(param_groups, lr=float(lr), betas=betas)


def build_scheduler(
    optimizer: torch.optim.Optimizer,
    epochs: int,
    warmup_epochs: int,
    min_lr: float,
    base_lr: Optional[float] = None,
) -> LambdaLR:
    """构建「线性热身 + 余弦退火」学习率调度器。

    前 ``warmup_epochs`` 轮学习率从接近 0 线性升到基础学习率；
    之后按余弦曲线从基础学习率衰减到 ``min_lr``。

    参数
    ----
    base_lr : float | None
        基础学习率；为 None 时取优化器参数组中的初始 lr。
    """
    if base_lr is None:
        base_lr = optimizer.param_groups[0]["lr"]
    min_lr_ratio = float(min_lr) / float(base_lr) if base_lr > 0 else 0.0
    warmup_epochs = max(0, int(warmup_epochs))
    total_epochs = max(1, int(epochs))

    def lr_lambda(epoch: int) -> float:
        """epoch 从 0 开始计数，返回当前轮的学习率缩放系数。"""
        if warmup_epochs > 0 and epoch < warmup_epochs:
            # 线性热身：+1 保证第 0 轮的学习率不为 0
            return float(epoch + 1) / float(warmup_epochs)
        progress = (epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        progress = min(1.0, max(0.0, progress))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return LambdaLR(optimizer, lr_lambda=lr_lambda)


# --------------------------------------------------------------------------- #
# 训练器
# --------------------------------------------------------------------------- #
class Trainer:
    """封装训练循环、验证、检查点保存与训练历史记录。

    参数
    ----
    cfg : Config
        全局配置对象。
    model : nn.Module
        ConvNeXt 模型。
    train_loader, val_loader : DataLoader
        训练集与验证集加载器。
    device : torch.device
        计算设备。
    logger : Logger
        日志器。
    accum_steps : int
        梯度累积步数（由显存自适应结果推导）。
    batch_size : int
        实际使用的训练 batch size（用于日志）。
    """

    def __init__(
        self,
        cfg,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        logger,
        accum_steps: int = 1,
        batch_size: int = 0,
    ) -> None:
        self.cfg = cfg
        self.logger = logger
        self.device = device
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.accum_steps = max(1, int(accum_steps))
        self.batch_size = int(batch_size)

        # 仅在 CUDA 上启用 AMP
        self.amp = bool(cfg.train.amp) and device.type == "cuda"
        # autocast 精度：Ampere 及以后用 bfloat16（无梯度下溢风险），T4 等回退 fp16
        self.amp_dtype = resolve_amp_dtype(device) if self.amp else None
        # AMP 稳定性监控：GradScaler 动态 scale 的最小值 / 是否已自动回退到 fp32
        self.amp_min_scale: Optional[float] = None
        self.amp_fell_back = False
        # 标签平滑：与 Mixup/CutMix 的软标签叠加使用
        self.criterion = nn.CrossEntropyLoss(label_smoothing=float(cfg.train.label_smoothing))
        self.optimizer = build_optimizer(self.model, lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        self.scheduler = build_scheduler(
            self.optimizer,
            epochs=cfg.train.epochs,
            warmup_epochs=cfg.train.warmup_epochs,
            min_lr=cfg.train.min_lr,
            base_lr=cfg.train.lr,
        )
        self.scaler = make_grad_scaler(device, enabled=self.amp)

        self.start_epoch = 0
        self.best_val_acc = 0.0
        self.best_epoch = 0
        self.epochs_without_improve = 0
        self.class_names: Optional[List[str]] = None   # 由 train.py 注入，随检查点一起保存
        self.history: Dict[str, List[float]] = {
            "epoch": [], "train_loss": [], "train_acc": [],
            "val_loss": [], "val_acc": [], "lr": [], "epoch_time": [],
            # AMP 稳定性证据：每轮 GradScaler 的动态 scale（关闭 AMP 时为 1.0）
            "amp_scale": [],
        }

    # ------------------------------------------------------------------ #
    # 工具方法
    # ------------------------------------------------------------------ #
    @property
    def best_ckpt_path(self) -> str:
        return os.path.join(self.cfg.ckpt_dir, "best.pt")

    @property
    def last_ckpt_path(self) -> str:
        return os.path.join(self.cfg.ckpt_dir, "last.pt")

    @property
    def history_path(self) -> str:
        return os.path.join(self.cfg.log_dir, "history.json")

    @property
    def history_csv_path(self) -> str:
        return os.path.join(self.cfg.log_dir, "history.csv")

    @property
    def curves_path(self) -> str:
        return os.path.join(self.cfg.fig_dir, "training_curves.png")

    def save_history(self) -> None:
        """把训练历史落盘（``history.json`` + ``history.csv``）。

        该函数在**每一轮结束时**调用。之所以不等训练全部结束再写，
        是因为 Colab 免费实例可能中途断连：逐轮增量落盘可保证已完成的指标不丢失，
        同时 ``history.csv`` 可直接作为报告中的"逐轮指标表"。
        """
        save_json(self.history, self.history_path)
        write_history_csv(self.history, self.history_csv_path)

    def _build_checkpoint(self, epoch: int) -> Dict[str, object]:
        """打包一个完整的训练状态（模型 + 优化器 + 调度器 + 历史）。"""
        return {
            "epoch": int(epoch),
            "model_state": self.model.state_dict(),
            "optimizer_state": self.optimizer.state_dict(),
            "scheduler_state": self.scheduler.state_dict(),
            "scaler_state": self.scaler.state_dict() if self.scaler is not None else None,
            "best_val_acc": float(self.best_val_acc),
            "best_epoch": int(self.best_epoch),
            "history": self.history,
            "config": self.cfg.to_dict(),
            "class_names": getattr(self, "class_names", None),
            # AMP 稳定性证据（可追溯：fp16 下溢事故的判定依据）
            "amp": {
                "enabled": bool(self.amp),
                "dtype": (str(self.amp_dtype).replace("torch.", "") if self.amp_dtype else None),
                "min_scale": self.amp_min_scale,
                "fell_back_to_fp32": bool(self.amp_fell_back),
            },
        }

    def _track_amp_scale(self) -> float:
        """记录并监控 GradScaler 的动态 scale，必要时**自动回退到 fp32**。

        为什么必须监控
        --------------
        fp16 依赖 GradScaler 放大损失以避开下溢。若模型在 fp16 下频繁溢出（inf/NaN），
        scaler 会把 scale 连续减半；scale 被压到 ``0.0156`` 量级后，
        真实量级只有 ``1e-6`` 的梯度乘上 scale 后仍小于 fp16 最小次正规数 ``6e-8``，
        反向传播会将其取整为**精确的 0** —— 对应参数从此不再更新
        （检查点中 ``exp_avg_sq`` 恒为 0），训练表现为"loss 不下降、精度停在随机水平"。

        本方法每轮读取一次 scale 并记入训练历史；一旦跌破
        :data:`AMP_SCALE_FLOOR`，且配置允许（``train.amp_fallback``），
        就**自动关闭 AMP、改为 fp32 继续训练**（模型参数本就是 fp32 主副本，
        优化器状态可无缝续用），从而在不中断 Colab 会话的前提下挽救整轮训练。
        """
        if self.scaler is None:
            return 1.0
        scale = float(self.scaler.get_scale())
        if self.amp_min_scale is None or scale < self.amp_min_scale:
            self.amp_min_scale = scale
        if self.amp and bool(getattr(self.cfg.train, "amp_fallback", True)) and scale < AMP_SCALE_FLOOR:
            epoch_no = len(self.history["epoch"]) + 1
            self.logger.warning(
                f"[amp] 第 {epoch_no} 轮 GradScaler scale={scale:.3e} 已跌破安全阈值 "
                f"{AMP_SCALE_FLOOR:g}：fp16 梯度正在下溢，部分参数已停止更新"
                f"（典型表现：loss 不再下降）。现将 AMP 自动降级为 fp32 并继续训练，"
                f"后续轮次不再受 fp16 动态范围限制。"
            )
            self.amp = False
            self.amp_dtype = None
            self.scaler = None
            self.amp_fell_back = True
        return scale

    def save_checkpoint(self, path: str, epoch: int) -> None:
        """保存检查点（含模型、优化器、调度器、随机状态与训练历史）。"""
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        torch.save(self._build_checkpoint(epoch), path)

    def load_checkpoint(self, path: str) -> int:
        """加载检查点并恢复训练状态，返回下一个待训练的 epoch 编号。

        兼容两种加载场景：
        * 本地存在 CUDA：普通加载；
        * 从 GPU 保存、在 CPU 上加载：使用 ``map_location`` 兜底。
        """
        if not os.path.isfile(path):
            raise FileNotFoundError(f"找不到检查点文件：{path}")
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)

        self.model.load_state_dict(checkpoint["model_state"])
        if "optimizer_state" in checkpoint:
            try:
                self.optimizer.load_state_dict(checkpoint["optimizer_state"])
            except Exception as err:  # pragma: no cover - 参数组不匹配时跳过
                self.logger.warning(f"优化器状态恢复失败（将使用全新状态）：{err}")
        if "scheduler_state" in checkpoint:
            try:
                self.scheduler.load_state_dict(checkpoint["scheduler_state"])
            except Exception as err:  # pragma: no cover
                self.logger.warning(f"调度器状态恢复失败：{err}")
        if self.scaler is not None and checkpoint.get("scaler_state"):
            try:
                self.scaler.load_state_dict(checkpoint["scaler_state"])
            except Exception:  # pragma: no cover
                pass

        self.best_val_acc = float(checkpoint.get("best_val_acc", 0.0))
        self.best_epoch = int(checkpoint.get("best_epoch", 0))
        if checkpoint.get("history"):
            self.history = checkpoint["history"]

        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        self.logger.info(
            f"[resume] 已从 {path} 恢复：下一轮 epoch={start_epoch}，"
            f"历史最优验证准确率={self.best_val_acc:.2f}%"
        )
        return start_epoch

    def plot_curves(self, show: bool = False) -> None:
        """按当前训练历史绘制并保存训练曲线。"""
        plot_training_curves(
            self.history,
            save_path=self.curves_path,
            title=f"ConvNeXt-{self.cfg.model.name} on CIFAR-10",
            show=show,
        )

    # ------------------------------------------------------------------ #
    # 单轮训练
    # ------------------------------------------------------------------ #
    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        """训练一个 epoch，返回 ``{"loss": ..., "acc": ...}``。

        关键实现点
        ----------
        * 用 ``apply_mixup_cutmix`` 判断本 batch 是否应用 Mixup/CutMix；
          未应用时返回 ``targets_a == targets_b == targets, lam=1.0``，
          因此损失函数可以统一写成 :func:`mixup_criterion` 而无需分支判断。
        * 训练准确率同样按 Mixup 的软标签计算：``lam*acc(y_a) + (1-lam)*acc(y_b)``。
        * 损失在反向传播前除以 ``accum_steps``，使累积后的梯度等价于大 batch。
        """
        self.model.train()
        loss_meter = AverageMeter("train_loss")
        acc_meter = AverageMeter("train_acc")

        # ---- 每个 epoch 的 step 上限（仅用于限时冒烟测试）---- #
        # 设置后，一个 epoch 的实际耗时只取决于 step 数，与数据集规模、硬件性能解耦；
        # 这对"只有 2 vCPU 的 Colab CPU 运行时"尤为重要。
        max_steps = self.cfg.train.max_steps_per_epoch
        total_steps = len(self.train_loader)
        num_steps = total_steps if max_steps is None else min(total_steps, max(1, int(max_steps)))

        epoch_start = time.time()
        self.optimizer.zero_grad(set_to_none=True)

        for step, (images, targets) in enumerate(self.train_loader):
            if step >= num_steps:          # 达到上限，本 epoch 提前结束
                break
            images = images.to(self.device, non_blocking=True)
            # 强制转换为 int64，兼容不同数据集返回不同整型的情况（CrossEntropyLoss 要求 Long）
            targets = targets.to(self.device, non_blocking=True).long()

            # ---- Mixup / CutMix 数据增强 ---- #
            images, targets_a, targets_b, lam, applied = apply_mixup_cutmix(
                images, targets,
                use_mixup=bool(self.cfg.train.use_mixup),
                prob=float(self.cfg.train.mixup_prob),
                mixup_alpha=float(self.cfg.train.mixup_alpha),
                cutmix_alpha=float(self.cfg.train.cutmix_alpha),
            )

            # ---- 前向 + 损失 ---- #
            with autocast_context(self.device, enabled=self.amp, dtype=self.amp_dtype):
                outputs = self.model(images)
                loss = mixup_criterion(self.criterion, outputs, targets_a, targets_b, lam)
                loss_for_backward = loss / self.accum_steps

            # ---- 反向 ---- #
            if self.scaler is not None:
                self.scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            # ---- 梯度累积到设定步数后统一更新参数 ---- #
            is_last_step = (step + 1) == num_steps
            if (step + 1) % self.accum_steps == 0 or is_last_step:
                if self.scaler is not None:
                    # 先 unscale 再裁剪，保证裁剪作用于真实梯度尺度
                    self.scaler.unscale_(self.optimizer)
                if float(self.cfg.train.grad_clip) > 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), max_norm=float(self.cfg.train.grad_clip)
                    )
                if self.scaler is not None:
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    self.optimizer.step()
                self.optimizer.zero_grad(set_to_none=True)

            # ---- 统计（用原始 batch 尺寸加权） ---- #
            batch_size = targets.size(0)
            with torch.no_grad():
                preds = outputs.detach().argmax(dim=1)
                acc = (lam * (preds == targets_a).float()
                       + (1.0 - lam) * (preds == targets_b).float()).mean().item()
            loss_meter.update(loss.item(), batch_size)
            acc_meter.update(acc, batch_size)

            if (step + 1) % 50 == 0 or is_last_step:
                self.logger.info(
                    f"  epoch {epoch + 1:>3d} | step {step + 1:>4d}/{num_steps} | "
                    f"loss {loss_meter.avg:.4f} | acc {acc_meter.avg * 100:5.2f}% | "
                    f"lr {self.optimizer.param_groups[0]['lr']:.2e}"
                    + (f" | 已用时 {human_time(time.time() - epoch_start)}" if is_last_step else "")
                )

        if num_steps < total_steps:
            self.logger.info(
                f"  epoch {epoch + 1} 提前结束：本 epoch 仅训练 {num_steps}/{total_steps} 个 step"
                f"（--max-steps-per-epoch={max_steps}），因此本轮指标只代表该子集"
            )

        return {"loss": loss_meter.avg, "acc": acc_meter.avg, "time": time.time() - epoch_start}

    # ------------------------------------------------------------------ #
    # 完整训练流程
    # ------------------------------------------------------------------ #
    def fit(self) -> Dict[str, List[float]]:
        """执行完整训练流程，返回训练历史。

        每一轮的流程：训练 -> 验证 -> 更新学习率 -> 记录历史 -> 保存检查点 -> 绘制曲线。
        """
        cfg = self.cfg
        epochs = int(cfg.train.epochs)

        # ---------------- 断点续训 ---------------- #
        # --resume <路径> : 从指定检查点恢复
        # --resume auto   : 若本次实验目录下已存在 last.pt 则自动恢复（Colab 断线重连专用）
        resume_path = cfg.train.resume
        if isinstance(resume_path, str) and resume_path.strip().lower() == "auto":
            if os.path.isfile(self.last_ckpt_path):
                resume_path = self.last_ckpt_path
                self.logger.info(f"[resume] --resume auto：检测到未完成的训练，自动从 {resume_path} 续训")
            else:
                resume_path = None
                self.logger.info("[resume] --resume auto：未找到 last.pt，将从头开始训练")
        if resume_path:
            self.start_epoch = self.load_checkpoint(resume_path)
            self.save_history()   # 立刻把恢复出来的历史落盘，避免中途中断时丢失

        total_params = sum(p.numel() for p in self.model.parameters())
        if cfg.train.max_steps_per_epoch is not None:
            self.logger.warning(
                f"[train] 已启用 --max-steps-per-epoch={cfg.train.max_steps_per_epoch}："
                f"每个 epoch 只训练固定数量的 step，因此产生的指标**不代表完整 epoch**，"
                f"仅用于限时冒烟测试。"
            )
        self.logger.info("=" * 96)
        self.logger.info(f"开始训练：epochs={epochs}（本轮从 epoch {self.start_epoch + 1} 开始）| "
                         f"batch_size={self.batch_size} x accum={self.accum_steps} "
                         f"(等效 batch={self.batch_size * self.accum_steps})")
        self.logger.info(f"优化器=AdamW(lr={cfg.train.lr}, wd={cfg.train.weight_decay}) | "
                         f"调度=warmup({cfg.train.warmup_epochs}) + cosine -> {cfg.train.min_lr} | "
                         f"AMP={describe_amp(self.device, self.amp, self.amp_dtype)} | "
                         f"Mixup/CutMix={cfg.train.use_mixup} | "
                         f"参数量={total_params / 1e6:.2f}M")
        if self.amp and self.amp_dtype == torch.float16:
            self.logger.warning(
                "[amp] 当前使用 fp16（本设备不支持原生 bf16）。若每轮日志中的 "
                "amp_scale 持续低于 1，说明 fp16 梯度已在下溢、部分参数不再更新；"
                "此时建议换用支持 bf16 的 GPU，或加 --no-amp 以 fp32 训练。"
            )
        self.logger.info("=" * 96)

        for epoch in range(self.start_epoch, epochs):
            current_lr = float(self.optimizer.param_groups[0]["lr"])
            self.logger.info(f"[Epoch {epoch + 1}/{epochs}] lr={current_lr:.3e}")

            # ---- 训练一轮 ---- #
            train_stats = self.train_one_epoch(epoch)

            # ---- 验证 ---- #
            val_metrics = evaluate(
                self.model, self.val_loader, self.criterion, self.device, amp=self.amp, topk=(1, 5)
            )

            # ---- 更新学习率（按 epoch 调度） ---- #
            self.scheduler.step()

            # ---- AMP 稳定性监控（scale 崩塌时自动降级为 fp32） ---- #
            amp_scale = self._track_amp_scale()

            # ---- 记录历史 ---- #
            self.history["epoch"].append(epoch + 1)
            self.history["train_loss"].append(float(train_stats["loss"]))
            self.history["train_acc"].append(float(train_stats["acc"]))
            self.history["val_loss"].append(float(val_metrics["loss"]))
            self.history["val_acc"].append(float(val_metrics["top1"]) / 100.0)
            self.history["lr"].append(current_lr)
            self.history["epoch_time"].append(float(train_stats["time"]))
            self.history["amp_scale"].append(float(amp_scale))
            # 逐轮增量落盘：即使训练中途断连，已完成轮次的指标也不会丢失
            self.save_history()

            amp_note = f" | amp_scale={amp_scale:.3e}" if self.scaler is not None else ""
            self.logger.info(
                f"[Epoch {epoch + 1}/{epochs}] 完成 | "
                f"train_loss={train_stats['loss']:.4f} train_acc={train_stats['acc'] * 100:.2f}% | "
                f"val_loss={val_metrics['loss']:.4f} val_top1={val_metrics['top1']:.2f}% "
                f"val_top5={val_metrics['top5']:.2f}% | 用时 {human_time(train_stats['time'])}{amp_note}"
            )

            # ---- 保存检查点 ---- #
            self.save_checkpoint(self.last_ckpt_path, epoch)
            if val_metrics["top1"] > self.best_val_acc:
                self.best_val_acc = float(val_metrics["top1"])
                self.best_epoch = epoch + 1
                self.epochs_without_improve = 0
                self.save_checkpoint(self.best_ckpt_path, epoch)
                self.logger.info(f"  -> 刷新最优验证准确率：{self.best_val_acc:.2f}%（已保存 best.pt）")
            else:
                self.epochs_without_improve += 1

            # ---- 周期性/最终绘制曲线 ---- #
            plot_every = int(cfg.train.plot_every)
            if plot_every > 0 and ((epoch + 1) % plot_every == 0 or (epoch + 1) == epochs):
                self.plot_curves(show=False)

            # ---- 早停 ---- #
            patience = int(cfg.train.early_stop_patience)
            if patience > 0 and self.epochs_without_improve >= patience:
                self.logger.info(
                    f"[early-stop] 连续 {patience} 轮验证准确率未提升，提前结束训练于 epoch {epoch + 1}。"
                )
                break

        # 保存训练历史（供报告与可视化复用）
        self.save_history()
        self.plot_curves(show=False)
        self.logger.info(
            f"训练结束 | 历史最优验证准确率 = {self.best_val_acc:.2f}%（epoch {self.best_epoch}）| "
            f"曲线已保存至 {self.curves_path}"
        )
        if self.amp_min_scale is not None:
            self.logger.info(
                f"[amp] GradScaler 最小 scale = {self.amp_min_scale:.3e}"
                + ("（曾因 scale 崩塌自动降级为 fp32）" if self.amp_fell_back else "")
            )
        self.logger.info(
            f"[logs] 逐轮指标已保存：{os.path.abspath(self.history_path)} 与 "
            f"{os.path.abspath(self.history_csv_path)}"
        )
        return self.history


### 8. 训练入口脚本


`train.py`（323 行）


In [ ]:
%%writefile train.py
# -*- coding: utf-8 -*-
"""
ConvNeXt + CIFAR-10 训练入口脚本。

用法（Google Colab 中直接运行）::

    !python train.py --epochs 60 --model tiny --batch-size 128

以「实验隔离」方式运行（推荐：不同实验各自独立成目录，互不覆盖）::

    !python train.py --run-tag full_tiny_60e --epochs 60 --model tiny
    !python train.py --run-tag ablation_patchify_stem --model pico --stem-mode patchify
    !python train.py --run-tag seed43 --model pico --epochs 30 --seed 43

完整流程
--------
1. 解析配置、创建输出目录、开启日志；
2. 固定随机种子、选择计算设备（CUDA / MPS / CPU）；
3. **显存自适应**探测 batch size（不足则自动下调，并用梯度累积补偿等效 batch）；
4. 构建 CIFAR-10 数据加载器（训练 45000 / 验证 5000 / 测试 10000）；
5. 构建 ConvNeXt、AdamW、warmup + cosine 调度器，执行训练；
6. 载入最优检查点在**测试集**上评估，输出准确率、逐类准确率与混淆矩阵；
7. 绘制并保存损失曲线、准确率曲线与混淆矩阵图。

产物（默认写入 ``./outputs/``，按"结果三分类"组织）
--------------------------------------------------
* ``models/best.pt``    —— 【模型】验证集最优模型（含优化器/调度器/历史/配置）
* ``models/last.pt``    —— 【模型】最后一轮的模型（可用于断点续训）
* ``logs/train.log``    —— 【日志】完整训练日志（带时间戳）
* ``logs/history.json`` —— 【日志】逐轮损失 / 准确率 / 学习率 / 耗时
* ``figures/training_curves.png``      —— 【图表】损失与准确率曲线
* ``figures/confusion_matrix.png``     —— 【图表】测试集混淆矩阵
* ``figures/per_class_accuracy.png``   —— 【图表】逐类准确率
* ``figures/data_samples.png``         —— 【图表】数据样例
* ``metrics/test_metrics.json``        —— 【指标】最终测试指标
* ``metrics/metrics.csv``              —— 【指标】一行式指标汇总表（便于拼接成对比/消融表）
* ``metrics/config.json``              —— 【指标】本次运行的全部超参数（可复现性证据）

实验目录隔离
------------
指定 ``--run-tag <name>`` 后，上述全部产物写入 ``<output-dir>/<name>/`` 而非 ``<output-dir>/``，
从而保证「全量训练 / 对比实验 / 消融实验 / 多种子重复」等多次运行**互不覆盖、互不交叉引用**。
"""

from __future__ import annotations

import os
import time

import numpy as np
import torch

from configs import parse_args
from src.data import build_dataloaders
from src.engine.evaluator import analyze_predictions, collect_predictions, evaluate
from src.engine.trainer import Trainer
from src.models.convnext import build_convnext, convnext_model_summary
from src.utils.device import (
    compute_accum_steps,
    describe_device,
    get_device,
    gpu_total_memory_gb,
    resolve_batch_size,
)
from src.utils.logger import get_logger
from src.utils.csv_logger import write_metrics_csv
from src.utils.misc import count_parameters, format_params, human_time, save_json
from src.utils.seed import set_seed
from src.utils.tta import describe_tta
from src.utils.visualize import plot_confusion_matrix, plot_data_samples, plot_per_class_accuracy

_IMAGE_SIZE = 32  # CIFAR-10 图像尺寸，供显存探测与模型摘要使用


def main() -> None:
    """训练主流程。"""
    wall_start = time.time()

    # ------------------------------------------------------------------ #
    # 1. 配置与日志
    # ------------------------------------------------------------------ #
    cfg = parse_args()
    cfg.make_dirs()
    logger = get_logger(
        name="convnext_train",
        log_file=os.path.join(cfg.log_dir, "train.log"),
    )
    logger.info("=" * 96)
    logger.info(f"实验名称：{cfg.exp_name}")
    logger.info(f"输出根目录：{os.path.abspath(cfg.output_dir)}")
    if cfg.run_tag:
        logger.info(f"实验隔离标签：{cfg.run_tag} -> 本次产物写入 {os.path.abspath(cfg.run_root)}")
    else:
        logger.info(f"未设置实验标签（--run-tag），产物直接写入 {os.path.abspath(cfg.run_root)}")
    cfg.save(os.path.join(cfg.result_dir, "config.json"))
    logger.info("=" * 96)

    # ------------------------------------------------------------------ #
    # 2. 随机种子与设备
    # ------------------------------------------------------------------ #
    set_seed(seed=int(cfg.train.seed), deterministic=bool(cfg.train.deterministic))
    device = get_device(cfg.device)
    logger.info(f"[device] {describe_device(device)}")
    logger.info(f"[seed] 全局随机种子 = {cfg.train.seed} | 确定性模式 = {cfg.train.deterministic}")

    # ------------------------------------------------------------------ #
    # 3. 显存自适应 batch size
    # ------------------------------------------------------------------ #
    def build_model() -> torch.nn.Module:
        """模型构建闭包（供显存探测与正式训练复用同一结构）。"""
        return build_convnext(cfg.model, num_classes=cfg.data.num_classes)

    actual_batch_size = resolve_batch_size(
        build_model_fn=build_model,
        img_size=_IMAGE_SIZE,
        num_classes=int(cfg.data.num_classes),
        device=device,
        requested_batch_size=int(cfg.train.batch_size),
        min_batch_size=int(cfg.train.min_batch_size),
        max_batch_size=int(cfg.train.max_batch_size),
        amp=bool(cfg.train.amp),
        auto=bool(cfg.train.auto_batch_size),
        logger=logger,
    )
    # 若 batch size 被自动调小，则用梯度累积把等效 batch 补回期望值
    accum_steps = compute_accum_steps(int(cfg.train.batch_size), actual_batch_size)
    if actual_batch_size != int(cfg.train.batch_size):
        logger.info(
            f"[batch-size] 期望 {cfg.train.batch_size} -> 实际 {actual_batch_size}，"
            f"采用梯度累积 {accum_steps} 步（等效 batch = {actual_batch_size * accum_steps}）"
        )

    # ------------------------------------------------------------------ #
    # 4. 数据
    # ------------------------------------------------------------------ #
    train_loader, val_loader, test_loader, class_names = build_dataloaders(
        cfg, batch_size=actual_batch_size, logger=logger
    )

    # 保存一批数据样例图（用于报告中的"数据集说明"）
    try:
        sample_images, sample_labels = next(iter(val_loader))
        plot_data_samples(
            sample_images[:16].cpu().numpy(),
            sample_labels[:16].cpu().numpy(),
            class_names,
            save_path=os.path.join(cfg.fig_dir, "data_samples.png"),
            mean=cfg.data.mean,
            std=cfg.data.std,
        )
        logger.info(f"[figures] 数据样例图已保存：{os.path.join(cfg.fig_dir, 'data_samples.png')}")
    except Exception as err:  # pragma: no cover - 绘图失败不应中断训练
        logger.warning(f"绘制数据样例图失败（不影响训练）：{err}")

    # ------------------------------------------------------------------ #
    # 5. 模型
    # ------------------------------------------------------------------ #
    model = build_model()
    total_params, trainable_params = count_parameters(model)
    logger.info("[model] 结构摘要：")
    for line in convnext_model_summary(cfg.model, num_classes=cfg.data.num_classes,
                                       input_size=_IMAGE_SIZE).split("\n"):
        logger.info(f"    {line}")
    logger.info(
        f"[model] 总参数量 = {format_params(total_params)}（{total_params:,}）| "
        f"可训练参数量 = {format_params(trainable_params)}"
    )

    # ------------------------------------------------------------------ #
    # 6. 训练
    # ------------------------------------------------------------------ #
    trainer = Trainer(
        cfg=cfg,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        logger=logger,
        accum_steps=accum_steps,
        batch_size=actual_batch_size,
    )
    trainer.class_names = class_names
    history = trainer.fit()

    # ------------------------------------------------------------------ #
    # 7. 载入最优权重并在测试集上评估
    # ------------------------------------------------------------------ #
    logger.info("=" * 96)
    logger.info("开始在**测试集**上评估验证集最优模型（best.pt）……")
    best_ckpt = trainer.best_ckpt_path
    if os.path.isfile(best_ckpt):
        checkpoint = torch.load(best_ckpt, map_location=device, weights_only=False)
        trainer.model.load_state_dict(checkpoint["model_state"])
        logger.info(f"[evaluate] 已载入 {best_ckpt}（对应 epoch {checkpoint.get('epoch', -1) + 1}）")
    else:
        logger.warning("[evaluate] 未找到 best.pt，使用当前（最后一轮）权重进行评估")

    criterion = torch.nn.CrossEntropyLoss(label_smoothing=float(cfg.train.label_smoothing))
    use_tta = bool(cfg.train.tta)
    if use_tta:
        logger.info(f"[test] 启用测试时增强 —— {describe_tta(cfg.train.tta_hflip)}")
    test_metrics = evaluate(
        trainer.model, test_loader, criterion, device, amp=trainer.amp, topk=(1, 5),
        tta=use_tta, tta_hflip=bool(cfg.train.tta_hflip),
    )
    logger.info(
        f"[test] loss={test_metrics['loss']:.4f} | "
        f"Top-1 = {test_metrics['top1']:.2f}% | Top-5 = {test_metrics['top5']:.2f}% | "
        f"样本数 = {int(test_metrics['num_samples'])} | TTA = {use_tta}"
    )

    # ------------------------------------------------------------------ #
    # 8. 逐类准确率 / 混淆矩阵 / 图表
    # ------------------------------------------------------------------ #
    # 与上面的 evaluate 保持完全相同的推理口径（是否 TTA 必须一致），
    # 否则混淆矩阵/逐类准确率会与总体 Top-1 对不上
    y_true, y_pred = collect_predictions(
        trainer.model, test_loader, device, amp=trainer.amp,
        tta=use_tta, tta_hflip=bool(cfg.train.tta_hflip),
    )
    analysis = analyze_predictions(
        y_true, y_pred, class_names,
        logger=logger,
        save_path=os.path.join(cfg.result_dir, "per_class_metrics.json"),
    )

    plot_confusion_matrix(
        np.asarray(analysis["confusion_matrix"]),
        class_names,
        save_path=os.path.join(cfg.fig_dir, "confusion_matrix.png"),
        normalize=True,
    )
    plot_per_class_accuracy(
        [analysis["per_class_accuracy"][name] for name in class_names],
        class_names,
        save_path=os.path.join(cfg.fig_dir, "per_class_accuracy.png"),
    )

    # ------------------------------------------------------------------ #
    # 9. 汇总并落盘
    # ------------------------------------------------------------------ #
    total_seconds = time.time() - wall_start
    summary = {
        "experiment": cfg.exp_name,
        "algorithm": "ConvNeXt (A ConvNet for the 2020s, CVPR 2022; arXiv:2201.03545)",
        "architecture": {
            "preset": cfg.model.name,
            "stem_mode": cfg.model.stem_mode,
            "drop_path_rate": cfg.model.drop_path_rate,
            "depths": list(cfg.model.depths) if cfg.model.depths else None,
            "dims": list(cfg.model.dims) if cfg.model.dims else None,
            "num_params": int(total_params),
        },
        "dataset": {
            "name": cfg.data.dataset,
            "is_synthetic": str(cfg.data.dataset).strip().lower() == "synthetic",
            "split": f"train {len(train_loader.dataset)} / val {len(val_loader.dataset)} / "
                     f"test {len(test_loader.dataset)}",
            # 只有"真实 CIFAR-10 + 全量数据"才算全量实验
            "is_full_scale": bool(
                str(cfg.data.dataset).strip().lower() in ("cifar10", "cifar-10", "cifar")
                and len(train_loader.dataset) == 45000
                and len(test_loader.dataset) == 10000
            ),
            "train_subset_size": cfg.data.train_subset_size,
            "test_subset_size": cfg.data.test_subset_size,
            "val_ratio": cfg.data.val_ratio,
            "data_root": os.path.abspath(cfg.data.data_root),
        },
        "device": describe_device(device),
        "gpu_total_memory_gb": round(gpu_total_memory_gb(device), 2),
        "batch_size_requested": int(cfg.train.batch_size),
        "batch_size_actual": int(actual_batch_size),
        "grad_accum_steps": int(accum_steps),
        "effective_batch_size": int(actual_batch_size * accum_steps),
        "epochs": int(cfg.train.epochs),
        "epochs_completed": len(history.get("epoch", [])),
        # 若不为 None，说明本轮被 --max-steps-per-epoch 限制，指标只代表该 step 子集
        "max_steps_per_epoch": cfg.train.max_steps_per_epoch,
        "best_val_acc": round(float(trainer.best_val_acc), 4),
        "best_val_epoch": int(trainer.best_epoch),
        "test_metrics": {
            "loss": round(float(test_metrics["loss"]), 6),
            "top1_acc": round(float(test_metrics["top1"]), 4),
            "top5_acc": round(float(test_metrics["top5"]), 4),
            "num_samples": int(test_metrics["num_samples"]),
        },
        "evaluation": {
            "tta": use_tta,
            "tta_views": (["identity", "hflip"] if (use_tta and cfg.train.tta_hflip) else ["identity"]),
            "note": "loss 由不做 TTA 的原始前向计算；Top-1/Top-5 使用 TTA 平均 softmax 概率",
        },
        "per_class_accuracy": {k: round(float(v) * 100.0, 4)
                               for k, v in analysis["per_class_accuracy"].items()},
        "macro_accuracy": round(float(analysis["macro_accuracy"]), 4),
        "total_training_time": human_time(total_seconds),
        "seed": int(cfg.train.seed),
        # 混合精度可追溯性：fp16 的 GradScaler scale 若崩塌到 <1，
        # 会导致部分参数梯度下溢为 0（曾造成"loss 不降、精度停在 25%"的假收敛）。
        # 这里记录最小 scale 与是否触发过自动降级，作为结果可信度的判定依据。
        "amp": {
            "enabled": bool(trainer.amp),
            "dtype": (str(trainer.amp_dtype).replace("torch.", "") if trainer.amp_dtype else None),
            "min_scale": trainer.amp_min_scale,
            "fell_back_to_fp32": bool(trainer.amp_fell_back),
        },
    }
    save_json(summary, os.path.join(cfg.result_dir, "test_metrics.json"))
    # 同时导出一行式指标汇总表，便于把多次实验直接拼接成对比/消融表格
    csv_path = write_metrics_csv(summary, os.path.join(cfg.result_dir, "metrics.csv"))
    logger.info(f"[metrics] 指标汇总表已保存：{os.path.abspath(csv_path)}")

    logger.info("=" * 96)
    logger.info(f"全部完成！总耗时 {human_time(total_seconds)}")
    logger.info(f"最终测试集 Top-1 准确率：{test_metrics['top1']:.2f}%")
    logger.info(f"本次实验产物目录：{os.path.abspath(cfg.run_root)}")
    logger.info(f"结果文件：{os.path.abspath(cfg.result_dir)}")
    logger.info(f"图表文件：{os.path.abspath(cfg.fig_dir)}")
    logger.info("=" * 96)


if __name__ == "__main__":
    main()


### 9. 独立评估入口脚本


`evaluate.py`（226 行）


In [ ]:
%%writefile evaluate.py
# -*- coding: utf-8 -*-
"""
ConvNeXt + CIFAR-10 独立评估入口脚本。

用于在不重新训练的前提下，用已保存的检查点对**测试集**做完整评估：

    !python evaluate.py --ckpt ./outputs/models/best.pt
    !python evaluate.py --ckpt ../05_results/<实验标签>/models/best.pt

若省略 ``--ckpt``，默认使用 ``<output-dir>/<run-tag>/models/best.pt``；
评估产物会写入该检查点所属实验自己的目录，不与其它实验交叉。

输出内容
--------
* 测试集损失、Top-1 / Top-5 准确率；
* 逐类准确率（按类别打印）与宏平均；
* 混淆矩阵图、逐类准确率条形图；
* ``<实验目录>/metrics/eval_metrics.json`` 等结构化结果文件。

设计要点
--------
1. **自动恢复训练配置**：检查点中保存了完整的训练配置（``config`` 字段），
   本脚本会以它为基准重建模型结构，因此**无需手动指定** ``--model`` / ``--stem-mode``
   等结构参数即可正确加载权重（若显式传入命令行参数，则以命令行优先）。
2. **划分一致性**：使用与训练**完全相同的随机种子**划分验证集，
   测试集始终是官方划分的 10000 张图像，不存在数据泄漏或划分漂移。
"""

from __future__ import annotations

import argparse
import os
from typing import List, Optional, Tuple

import numpy as np
import torch

from configs import Config, parse_args
from src.data import build_dataloaders
from src.engine.evaluator import analyze_predictions, collect_predictions, evaluate
from src.models.convnext import build_convnext, convnext_model_summary
from src.utils.device import describe_device, get_device
from src.utils.logger import get_logger
from src.utils.misc import count_parameters, format_params, save_json
from src.utils.seed import set_seed
from src.utils.tta import describe_tta
from src.utils.visualize import plot_confusion_matrix, plot_per_class_accuracy

_IMAGE_SIZE = 32


def parse_eval_args(argv: Optional[List[str]] = None) -> Tuple[Config, str, List[str]]:
    """解析评估脚本参数。

    先提取评估专用的 ``--ckpt``，其余参数原样交给通用配置解析器。

    返回
    ----
    (cfg, ckpt_path, remaining_args)
        ``cfg`` 为仅含命令行覆盖项的配置；``remaining_args`` 用于在读取检查点后
        以检查点配置为基准做二次解析。
    """
    parser = argparse.ArgumentParser(add_help=False)
    parser.add_argument("--ckpt", type=str, default=None, help="待评估的检查点路径")
    known, remaining = parser.parse_known_args(argv)

    cfg = parse_args(remaining)
    ckpt_path = known.ckpt or os.path.join(cfg.ckpt_dir, "best.pt")
    return cfg, ckpt_path, list(remaining)


def main() -> None:
    """评估主流程。"""
    cfg, ckpt_path, remaining_args = parse_eval_args()

    if not os.path.isfile(ckpt_path):
        raise FileNotFoundError(
            f"找不到检查点 {os.path.abspath(ckpt_path)}，请先运行 train.py，"
            f"或用 --ckpt 指定正确路径。"
        )

    # ------------------------------------------------------------------ #
    # 1. 读取检查点，并用其中保存的训练配置作为基准重建配置
    # ------------------------------------------------------------------ #
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    restored_from_ckpt = False
    if isinstance(checkpoint, dict) and isinstance(checkpoint.get("config"), dict):
        base_cfg = Config.from_dict(checkpoint["config"])
        # 命令行的显式参数优先级更高（parse_args 只覆盖被显式传入的项）
        cfg = parse_args(remaining_args, base=base_cfg)
        restored_from_ckpt = True

    cfg.make_dirs()
    logger = get_logger(name="convnext_eval", log_file=os.path.join(cfg.log_dir, "evaluate.log"))

    logger.info("=" * 100)
    logger.info(f"[evaluate] 检查点：{os.path.abspath(ckpt_path)}")
    if restored_from_ckpt:
        logger.info(
            f"[evaluate] 已从检查点恢复训练时的模型配置：preset={cfg.model.name} | "
            f"stem_mode={cfg.model.stem_mode} | drop_path={cfg.model.drop_path_rate}"
        )

    # ------------------------------------------------------------------ #
    # 2. 设备与随机种子（与训练保持一致，保证数据划分不变）
    # ------------------------------------------------------------------ #
    set_seed(seed=int(cfg.train.seed), deterministic=bool(cfg.train.deterministic))
    device = get_device(cfg.device)
    logger.info(f"[device] {describe_device(device)}")

    # ------------------------------------------------------------------ #
    # 3. 数据（评估只需要测试集，这里一并构建以便复用同一套划分逻辑）
    # ------------------------------------------------------------------ #
    eval_batch_size = max(64, int(cfg.train.batch_size) * 2)
    _, _, test_loader, class_names = build_dataloaders(
        cfg, batch_size=int(cfg.train.batch_size), eval_batch_size=eval_batch_size, logger=logger
    )

    # ------------------------------------------------------------------ #
    # 4. 模型与权重
    # ------------------------------------------------------------------ #
    model = build_convnext(cfg.model, num_classes=cfg.data.num_classes)
    total_params, _ = count_parameters(model)
    for line in convnext_model_summary(cfg.model, num_classes=cfg.data.num_classes,
                                       input_size=_IMAGE_SIZE).split("\n"):
        logger.info(f"    {line}")
    logger.info(f"[model] 总参数量 = {format_params(total_params)}")
    model = model.to(device)

    state_dict = checkpoint.get("model_state", checkpoint)  # 兼容只保存 state_dict 的简单情况
    if isinstance(checkpoint, dict) and "class_names" in checkpoint and checkpoint["class_names"]:
        class_names = list(checkpoint["class_names"])
    try:
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
    except RuntimeError as err:
        raise RuntimeError(
            f"权重与模型结构不匹配。\n{err}\n"
            f"提示：本脚本会自动从检查点恢复结构；若仍报错，请显式指定 "
            f"--model / --stem-mode（当前为 {cfg.model.name} / {cfg.model.stem_mode}）。"
        ) from err
    if missing or unexpected:
        logger.warning(f"权重加载存在差异 | 缺失键 {len(missing)} 个 | 多余键 {len(unexpected)} 个")
    logger.info(f"[evaluate] 已载入权重（训练时 epoch = {checkpoint.get('epoch', 'N/A') if isinstance(checkpoint, dict) else 'N/A'}）")

    # ------------------------------------------------------------------ #
    # 5. 评估
    # ------------------------------------------------------------------ #
    amp = bool(cfg.train.amp) and device.type == "cuda"
    use_tta = bool(cfg.train.tta)
    if use_tta:
        logger.info(f"[test] 启用测试时增强 —— {describe_tta(cfg.train.tta_hflip)}")
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=float(cfg.train.label_smoothing))
    test_metrics = evaluate(
        model, test_loader, criterion, device, amp=amp, topk=(1, 5),
        tta=use_tta, tta_hflip=bool(cfg.train.tta_hflip),
    )
    logger.info(
        f"[test] loss={test_metrics['loss']:.4f} | "
        f"Top-1 = {test_metrics['top1']:.2f}% | Top-5 = {test_metrics['top5']:.2f}% | "
        f"样本数 = {int(test_metrics['num_samples'])} | TTA = {use_tta}"
    )

    # ------------------------------------------------------------------ #
    # 6. 逐类指标与混淆矩阵
    # ------------------------------------------------------------------ #
    # 与上面的 evaluate 保持完全相同的推理口径（是否 TTA 必须一致）
    y_true, y_pred = collect_predictions(
        model, test_loader, device, amp=amp,
        tta=use_tta, tta_hflip=bool(cfg.train.tta_hflip),
    )
    analysis = analyze_predictions(
        y_true, y_pred, class_names,
        logger=logger,
        save_path=os.path.join(cfg.result_dir, "per_class_metrics.json"),
    )

    plot_confusion_matrix(
        np.asarray(analysis["confusion_matrix"]),
        class_names,
        save_path=os.path.join(cfg.fig_dir, "confusion_matrix.png"),
        normalize=True,
    )
    plot_per_class_accuracy(
        [analysis["per_class_accuracy"][name] for name in class_names],
        class_names,
        save_path=os.path.join(cfg.fig_dir, "per_class_accuracy.png"),
    )

    # ------------------------------------------------------------------ #
    # 7. 落盘
    # ------------------------------------------------------------------ #
    summary = {
        "checkpoint": os.path.abspath(ckpt_path),
        "device": describe_device(device),
        "batch_size": int(eval_batch_size),
        "model": {
            "preset": cfg.model.name,
            "stem_mode": cfg.model.stem_mode,
            "num_params": int(total_params),
        },
        "test_metrics": {
            "loss": round(float(test_metrics["loss"]), 6),
            "top1_acc": round(float(test_metrics["top1"]), 4),
            "top5_acc": round(float(test_metrics["top5"]), 4),
            "num_samples": int(test_metrics["num_samples"]),
        },
        "evaluation": {
            "tta": use_tta,
            "tta_views": (["identity", "hflip"] if (use_tta and cfg.train.tta_hflip) else ["identity"]),
            "note": "loss 由不做 TTA 的原始前向计算；Top-1/Top-5 使用 TTA 平均 softmax 概率",
        },
        "macro_accuracy": round(float(analysis["macro_accuracy"]), 4),
        "per_class_accuracy": {k: round(float(v) * 100.0, 4)
                               for k, v in analysis["per_class_accuracy"].items()},
        "confusion_matrix": analysis["confusion_matrix"],
        "seed": int(cfg.train.seed),
    }
    out_path = os.path.join(cfg.result_dir, "eval_metrics.json")
    save_json(summary, out_path)
    logger.info(f"[evaluate] 结果已保存：{os.path.abspath(out_path)}")
    logger.info(f"[evaluate] 图表已保存：{os.path.abspath(cfg.fig_dir)}")
    logger.info("=" * 100)


if __name__ == "__main__":
    main()


### 10. 工具脚本 —— CIFAR-10 高速下载器（镜像体检 + 多连接分块）


`scripts/fetch_cifar10.py`（814 行）


In [ ]:
%%writefile scripts/fetch_cifar10.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
CIFAR-10 高速下载器：镜像校验 + 多连接分块 + 断点续传 + MD5 校验 + 自动解压。

为什么需要它
------------
官方源（``www.cs.toronto.edu/~kriz``）在部分网络环境下**单连接速率极低**：
本机实测仅 **0.11 MB/s**，完整下载 162 MB 需要约 **25 分钟** —— 这正是
"快速冒烟测试无法在 2 分钟内完成"的首要原因。

该源**支持 HTTP Range**，因此可以拆成 N 个分片并行下载，把聚合带宽提升一个量级：
本机 16 连接实测 **1.33 MB/s（约 12 倍）**，完整下载 + 校验 + 解压共 **123 s**。

一次真实故障与由此确立的设计约束
--------------------------------
Colab 上曾出现"整包进度显示 100% 却最终失败"的现象：分片 ``part_15`` 反复
``HTTPError``，重试 30 次后停在 ``10655754/10656121`` 字节。

根因**不是**网络抖动，而是所选镜像 ``cs231n.stanford.edu`` 上那份归档被**截断**了：

===============================================  ==========================
服务器                                          ``Content-Length``
===============================================  ==========================
``cs231n.stanford.edu``（速率最快，被选中）      ``170497704``（**短 367 字节**）
``data.brainchip.com``                           ``170498071``（正确）
===============================================  ==========================

于是 ::

    Range: bytes=159841950-170498070  ->  206  Content-Range: bytes 159841950-170497703/170497704
    Range: bytes=170497704-170498070  ->  416  Requested Range Not Satisfiable

服务器把请求**裁剪到自己的真实文件末尾**（因此 ``part_15`` 恰好停在 10655754），
续传时请求已位于其 EOF 之外的区间，被判定为**不可满足**并返回 **416** ——
这是一个**永久性**错误，重试 30 次当然还是 416。而进度按"已收字节 / 本地常量
``ARCHIVE_SIZE``"计算，``170497704/170498071 = 99.9998%`` 显示成了 ``100.0%``，
掩盖了真相。

由此确立四条硬约束（本脚本的实现依据）：

1. **先校验远端长度再下载**：用 ``HEAD`` 取 ``Content-Length``（失败则退回
   ``Range: bytes=0-0`` 解析 ``Content-Range``），与 ``ARCHIVE_SIZE`` 不符的镜像
   直接判定为**已损坏**并剔除 —— 它即使 16 个分片全部"成功"，MD5 也必然错。
2. **区分致命错误与可重试错误**：``416/404/403/...`` 属于请求本身不合法，
   立刻失败并**切换镜像**；只有超时、连接重置、``5xx``、``429`` 才退避重试。
3. **分块要小且可单独重下**：固定大小分块（默认 4 MiB）排队领取，
   单个分块失败只需要重下 4 MiB，而不是 10 MiB 的整段。
4. **最终以 MD5 为准，任何环节失败都要能回退**：长度 + MD5 双重校验，
   失败则清理残留 → 换下一个镜像 → 最后退回单连接。

脚本流程
--------
1. **镜像体检**：``HEAD`` 查长度（剔除损坏镜像）→ 有界读取采样 1 MB 测速 →
   探测 Range 支持；
2. **择优下载**：优先"长度正确 + 支持 Range + 速率最高"，用分块队列并发下载，
   分块落盘保留、断点续传、指数退避重试；
3. **回退链**：当前镜像多连接失败 → 尝试单连接 → 换下一个镜像（均自动清理残留）；
4. **完整性校验**：合并后比对文件大小 + MD5，期望值取自工程常量（避免硬编码漂移）；
5. **自动解压**：产出 torchvision 可直接读取的 ``cifar-10-batches-py`` 目录。

用法
----
    python scripts/fetch_cifar10.py                        # 下载到 03_code/data
    python scripts/fetch_cifar10.py --dest ../04_data/cifar10
    python scripts/fetch_cifar10.py --threads 8 --chunk-mb 4   # Colab 推荐值（默认）
    python scripts/fetch_cifar10.py --probe-only           # 只体检镜像、不下载
    python scripts/fetch_cifar10.py --force                # 忽略缓存重新下载
"""

from __future__ import annotations

import argparse
import hashlib
import os
import queue
import random
import re
import shutil
import sys
import tarfile
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

# 让脚本能直接以 `python scripts/fetch_cifar10.py` 运行（把工程根目录加入模块搜索路径）
PROJECT_ROOT = Path(__file__).resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.cifar10 import (  # noqa: E402  (必须在上面的 sys.path 处理之后导入)
    ARCHIVE_MD5,
    ARCHIVE_NAME,
    ARCHIVE_SIZE,
    BASE_FOLDER,
)

#: 候选镜像（按经验排序；脚本会实测速率并校验长度后择优）
DEFAULT_MIRRORS: List[str] = [
    "https://www.cs.toronto.edu/~kriz/" + ARCHIVE_NAME,
    "http://cs231n.stanford.edu/" + ARCHIVE_NAME,
    "http://data.brainchip.com/dataset-mirror/cifar10/" + ARCHIVE_NAME,
]

#: 统一的 User-Agent（部分站点会拒绝 urllib 默认 UA）
USER_AGENT = "Mozilla/5.0 (compatible; cifar10-fetcher/2.0)"

#: 单次探测读取的字节数（1 MB，足以估算速率）
PROBE_BYTES = 1 << 20
#: 单个分块的读写块大小
CHUNK_SIZE = 1 << 18

#: 默认分块大小（4 MiB）：太小会放大请求开销，太大则单块失败的重下代价高
DEFAULT_CHUNK_SIZE = 4 << 20
#: 默认并发连接数：8 已接近多数镜像的饱和点，16 容易触发限流且收益递减
DEFAULT_THREADS = 8
#: 默认单分块最大重试次数（不含首次尝试）
DEFAULT_MAX_RETRIES = 8
#: 默认 socket 超时（秒）：作用于单次连接/读取操作
DEFAULT_TIMEOUT = 60.0
#: 退避参数：delay = U[0.5, 1.0] * min(CAP, BASE * FACTOR ** attempt)
BACKOFF_BASE = 1.0
BACKOFF_FACTOR = 2.0
BACKOFF_CAP = 30.0

#: 值得重试的 HTTP 状态码（临时性故障 / 限流）
RETRYABLE_HTTP_STATUS = frozenset({408, 425, 429, 500, 502, 503, 504})
#: 致命的 HTTP 状态码（重试无意义：请求本身不合法或资源不存在/越界）
FATAL_HTTP_STATUS = frozenset({400, 401, 403, 404, 405, 410, 416, 451})

_CONTENT_RANGE_RE = re.compile(r"^bytes\s+(\d+)-(\d+)/(\d+|\*)$", re.IGNORECASE)


# --------------------------------------------------------------------------- #
# 日志
# --------------------------------------------------------------------------- #
_log_lock = threading.Lock()


def log(message: str) -> None:
    """带时间戳的线程安全输出。"""
    with _log_lock:
        print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


# --------------------------------------------------------------------------- #
# 异常与工具
# --------------------------------------------------------------------------- #
class FatalDownloadError(RuntimeError):
    """不可重试的错误（区间越界 416、资源不存在 404、服务器忽略 Range 等）。

    这类错误重试多少次结果都一样，应当**立即**放弃当前镜像并切换到备用镜像。
    """


@dataclass
class Mirror:
    """一个候选镜像的体检结果。"""

    url: str
    speed: Optional[float]           # 采样下行速率（MB/s）；None 表示探测失败
    range_ok: bool                   # 是否支持 HTTP Range
    remote_size: Optional[int]       # 远端口径的文件大小；None 表示服务器未提供

    @property
    def size_ok(self) -> Optional[bool]:
        """远端大小是否与期望一致；None 表示无法判断（服务器未提供长度）。"""
        if self.remote_size is None:
            return None
        return self.remote_size == ARCHIVE_SIZE

    @property
    def usable(self) -> bool:
        """已知长度不符的镜像一定是坏的（MD5 必然失败），直接排除。"""
        return self.size_ok is not False


def md5_of(path: Path, chunk_size: int = 1 << 22) -> str:
    """计算文件 MD5。"""
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def backoff_delay(attempt: int) -> float:
    """指数退避 + 抖动。

    纯指数退避会让 8 个并发分块在同一时刻一起醒来（惊群），
    因此在 ``[0.5, 1.0] * window`` 上取随机值把重试打散。
    """
    window = min(BACKOFF_CAP, BACKOFF_BASE * (BACKOFF_FACTOR ** attempt))
    return random.uniform(window * 0.5, window)


def parse_content_range(value: str) -> Tuple[Optional[int], Optional[int]]:
    """解析 ``Content-Range: bytes 0-99/170498071``，返回 ``(起始偏移, 总长度)``。"""
    match = _CONTENT_RANGE_RE.match((value or "").strip())
    if not match:
        return None, None
    total = None if match.group(3) == "*" else int(match.group(3))
    return int(match.group(1)), total


def is_retryable(error: BaseException) -> bool:
    """判断异常是否值得重试（临时性故障）还是应立即失败（永久性故障）。"""
    if isinstance(error, FatalDownloadError):
        return False
    if isinstance(error, urllib.error.HTTPError):
        if error.code in FATAL_HTTP_STATUS:
            return False
        # 其余 4xx/5xx 中，仅重试已知的临时性状态码
        return error.code in RETRYABLE_HTTP_STATUS
    # 超时、连接重置、IncompleteRead、DNS 抖动等一律视为临时性
    return True


# --------------------------------------------------------------------------- #
# 镜像体检
# --------------------------------------------------------------------------- #
def remote_size(url: str, timeout: float = 15.0) -> Optional[int]:
    """获取远端文件大小：优先 ``HEAD``，不支持时退回 ``Range: bytes=0-0``。

    这是定位"镜像截断"故障的关键一步 —— 只有拿到服务器**自己的**文件长度，
    才能发现"这份归档比官方少 367 字节"。
    """
    try:
        request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(request, timeout=timeout) as response:
            length = response.headers.get("Content-Length")
            if length and length.strip().isdigit() and int(length) > 0:
                return int(length)
    except Exception:  # noqa: BLE001  某些服务器禁用 HEAD，退回 Range 探测
        pass

    try:
        request = urllib.request.Request(
            url, headers={"Range": "bytes=0-0", "User-Agent": USER_AGENT}
        )
        with urllib.request.urlopen(request, timeout=timeout) as response:
            _, total = parse_content_range(response.headers.get("Content-Range", ""))
            if total is not None:
                return total
            if response.status == 200:
                length = response.headers.get("Content-Length")
                if length and length.strip().isdigit():
                    return int(length)
    except Exception:  # noqa: BLE001
        pass
    return None


def probe_speed(url: str, nbytes: int = PROBE_BYTES, deadline: float = 8.0) -> Optional[float]:
    """采样测量单个连接的下行速率（MB/s）；不可用或过慢时返回 None。

    两个必须注意的点
    ----------------
    1. **有界读取**：若镜像忽略 Range 头而返回 200，
       无界的 ``resp.read()`` 会去读取整个归档（162 MB），导致探测长时间卡住。
    2. **硬性截止时间**：``urlopen`` 的 timeout 只作用于**单次 socket 操作**，
       慢速服务器每次都能在超时内返回少量数据，累加起来仍可能耗时数十秒
       （实测多伦多源探测耗时 42 s）。因此这里在读取循环内主动检查截止时间，
       一旦超时立即放弃该镜像。
    """
    start = time.time()
    try:
        request = urllib.request.Request(url, headers={
            "Range": f"bytes=0-{nbytes - 1}",
            "User-Agent": USER_AGENT,
            "Accept-Encoding": "identity",
        })
        received = 0
        with urllib.request.urlopen(request, timeout=deadline) as response:
            while received < nbytes:
                if time.time() - start > deadline:     # 硬性截止，避免被慢速源拖住
                    break
                block = response.read(min(1 << 16, nbytes - received))
                if not block:
                    break
                received += len(block)
    except Exception:
        return None

    elapsed = max(1e-6, time.time() - start)
    if elapsed > deadline or received < nbytes // 4:   # 太慢或数据太少，视为不可用
        return None
    return received / (1 << 20) / elapsed


def supports_range(url: str, timeout: float = 8.0) -> bool:
    """判断镜像是否支持 HTTP Range（多连接下载的前提）。"""
    try:
        request = urllib.request.Request(
            url, headers={"Range": "bytes=0-99", "User-Agent": USER_AGENT}
        )
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status = response.status
            data = response.read(100)                  # 有界读取
        return status == 206 and len(data) == 100
    except Exception:
        return False


def inspect_mirrors(mirrors: List[str]) -> List[Mirror]:
    """逐个体检候选镜像：远端长度 → 采样速率 → Range 支持。"""
    results: List[Mirror] = []
    for url in mirrors:
        size = remote_size(url)
        if size is None:
            log(f"  长度未知              <- {url}")
        elif size != ARCHIVE_SIZE:
            log(f"  长度不符 {size} (期望 {ARCHIVE_SIZE}，差 {ARCHIVE_SIZE - size} 字节) "
                f"-> 判定为损坏镜像 <- {url}")
        else:
            log(f"  长度正确 {size}       <- {url}")

        speed = probe_speed(url)
        if speed is None:
            log(f"  不可用（探测超时/过慢）<- {url}")
            continue

        range_ok = supports_range(url)
        mirror = Mirror(url, speed, range_ok, size)
        results.append(mirror)
        log(f"  可用 {speed:6.2f} MB/s | Range={str(range_ok):5s} | "
            f"长度={'正确' if mirror.size_ok else ('未知' if mirror.size_ok is None else '不符')}"
            f" <- {url}")
    return results


# --------------------------------------------------------------------------- #
# 分块下载
# --------------------------------------------------------------------------- #
def is_archive_valid(path: Path) -> bool:
    """判断归档文件是否完整（大小 + MD5 双重校验）。"""
    try:
        if not path.is_file() or path.stat().st_size != ARCHIVE_SIZE:
            return False
        return md5_of(path) == ARCHIVE_MD5
    except Exception:
        return False


def is_extracted(dest: Path) -> bool:
    """判断是否已解压就绪。"""
    return (dest / BASE_FOLDER / "test_batch").is_file()


def plan_chunks(expect_size: int, chunk_size: int) -> List[Tuple[int, int]]:
    """把 ``[0, expect_size)`` 切成固定大小的闭区间列表 ``[(start, end), ...]``。"""
    return [
        (start, min(expect_size - 1, start + chunk_size - 1))
        for start in range(0, expect_size, chunk_size)
    ]


def download_chunk(
    url: str,
    start: int,
    end: int,
    part: Path,
    max_retries: int = DEFAULT_MAX_RETRIES,
    timeout: float = DEFAULT_TIMEOUT,
) -> None:
    """下载字节区间 ``[start, end]`` 到分块文件，支持断点续传 + 指数退避重试。

    关键约束
    --------
    * 服务器**必须**回 ``206``；若回 ``200``（忽略了 Range），绝不能按偏移追加，
      否则数据会整体错位 —— 这里直接判定为致命错误。
    * 校验 ``Content-Range`` 的起始偏移与总长度，防止"服务器裁剪区间"被静默接受
      （这正是 ``part_15`` 停在 10655754 字节的直接原因）。
    * 致命错误（416 等）立即抛出，交由上层切换镜像，不做无谓重试。
    """
    want = end - start + 1
    last_error: Optional[BaseException] = None

    for attempt in range(max_retries + 1):
        have = part.stat().st_size if part.is_file() else 0
        if have == want:
            return
        if have > want:
            # 上一次响应给了超出区间的数据（Range 被忽略 / 并发写入），只能整块重下
            part.unlink(missing_ok=True)
            have = 0

        try:
            request = urllib.request.Request(url, headers={
                "Range": f"bytes={start + have}-{end}",
                "User-Agent": USER_AGENT,
                # 必须禁用压缩：经过 gzip 传输后 Range 语义与 Content-Length 都会错位
                "Accept-Encoding": "identity",
            })
            with urllib.request.urlopen(request, timeout=timeout) as response:
                if response.status != 206:
                    raise FatalDownloadError(
                        f"服务器忽略了 Range（HTTP {response.status}，期望 206）"
                    )
                got_start, total = parse_content_range(response.headers.get("Content-Range", ""))
                if got_start is None:
                    raise FatalDownloadError("响应缺少合法的 Content-Range 头")
                if got_start != start + have:
                    raise FatalDownloadError(
                        f"Content-Range 起始偏移不符（收到 {got_start}，期望 {start + have}）"
                    )
                if total is not None and total != ARCHIVE_SIZE:
                    raise FatalDownloadError(
                        f"远端文件长度为 {total}，与期望 {ARCHIVE_SIZE} 不符（镜像已损坏）"
                    )

                with open(part, "ab") as out:
                    while True:
                        block = response.read(CHUNK_SIZE)
                        if not block:
                            break
                        out.write(block)
        except BaseException as error:  # noqa: BLE001
            if isinstance(error, FatalDownloadError) or not is_retryable(error):
                raise FatalDownloadError(
                    f"分片 [{start}-{end}] 遇到不可重试错误：{type(error).__name__}: {error}"
                ) from error
            last_error = error
            if attempt >= max_retries:
                break
            delay = backoff_delay(attempt)
            if attempt < 3 or attempt % 2 == 0:
                log(f"  分片 [{start}-{end}] 第 {attempt + 1}/{max_retries + 1} 次尝试失败"
                    f"（{type(error).__name__}: {error}），{delay:.1f}s 后重试")
            time.sleep(delay)
        # 循环回到顶部：重新读取落盘大小，已收满则返回

    have = part.stat().st_size if part.is_file() else 0
    raise RuntimeError(
        f"分片 [{start}-{end}] 下载不完整（{have}/{want} 字节，共 {max_retries + 1} 次尝试）"
        + (f"，最后错误：{type(last_error).__name__}: {last_error}" if last_error else "")
    )


def cleanup_partials(dest: Path) -> None:
    """清理分块目录与临时合并文件（切换镜像前必须调用，避免脏数据被复用）。"""
    part_dir = dest / "_parts"
    if part_dir.is_dir():
        shutil.rmtree(part_dir, ignore_errors=True)
    (dest / "_archive_merging.bin").unlink(missing_ok=True)


def multi_connection_download(
    url: str,
    dest: Path,
    threads: int = DEFAULT_THREADS,
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    max_retries: int = DEFAULT_MAX_RETRIES,
    timeout: float = DEFAULT_TIMEOUT,
) -> Path:
    """多连接分块下载并合并，返回归档文件路径。

    实现要点
    --------
    * **工作队列**：分块索引放进队列，由 ``threads`` 个工作线程领取 ——
      先完成的分块会自动去领下一个，避免"慢分块拖住线程池"；
    * **分块独立重下**：每个分块单独落盘、单独续传、单独重试，
      失败只会重下该分块（默认 4 MiB）；
    * **快速熔断**：任一分块遇到致命错误（如 416）立即置位 ``abort`` 事件，
      其余线程尽快退出，由上层切换镜像。
    """
    part_dir = dest / "_parts"
    part_dir.mkdir(parents=True, exist_ok=True)

    # 分块大小是分块文件偏移的语义来源：与上次不一致则必须整体作废（否则数据错位）
    marker = part_dir / ".chunk_size"
    previous = marker.read_text().strip() if marker.is_file() else None
    if previous is not None and previous != str(chunk_size):
        log(f"  分块大小由 {previous} 变为 {chunk_size}，丢弃旧分块重新开始")
        cleanup_partials(dest)
        part_dir.mkdir(parents=True, exist_ok=True)
    marker.write_text(str(chunk_size))

    chunks = plan_chunks(ARCHIVE_SIZE, chunk_size)
    part_path = lambda index: part_dir / f"part_{index:04d}"  # noqa: E731

    # 断点续传：统计已有且未越界的分块进度
    resumed = sum(
        1 for index, (s, e) in enumerate(chunks)
        if part_path(index).is_file() and 0 < part_path(index).stat().st_size <= e - s + 1
    )
    log(f"启用 {len(chunks)} 个分块 × {chunk_size / (1 << 20):.1f} MiB，"
        f"并发 {threads} 连接"
        + (f"（复用 {resumed} 个已有分块）" if resumed else ""))

    start_time = time.time()
    stop_event = threading.Event()
    abort_event = threading.Event()
    pending: "queue.Queue[int]" = queue.Queue()
    for index in range(len(chunks)):
        pending.put(index)
    progress = {"done": 0}
    progress_lock = threading.Lock()

    def reporter() -> None:
        while not stop_event.wait(20):
            got = sum(
                part_path(i).stat().st_size for i in range(len(chunks)) if part_path(i).is_file()
            )
            elapsed = max(1e-6, time.time() - start_time)
            with progress_lock:
                done = progress["done"]
            log(f"  进度 {got / (1 << 20):7.1f}/{ARCHIVE_SIZE / (1 << 20):.1f} MB "
                f"({100 * got / ARCHIVE_SIZE:5.1f}%) 分块 {done}/{len(chunks)} "
                f"平均 {got / (1 << 20) / elapsed:.2f} MB/s 已用 {elapsed:.0f}s")

    def worker() -> None:
        while not abort_event.is_set():
            try:
                index = pending.get_nowait()
            except queue.Empty:
                return
            start, end = chunks[index]
            try:
                download_chunk(url, start, end, part_path(index), max_retries, timeout)
            except BaseException as error:  # noqa: BLE001
                failures.append((index, start, end, error))
                if isinstance(error, FatalDownloadError):
                    abort_event.set()       # 致命错误：熔断，其余线程尽快退出
                return
            else:
                with progress_lock:
                    progress["done"] += 1

    failures: List[Tuple[int, int, int, BaseException]] = []
    reporter_thread = threading.Thread(target=reporter, daemon=True)
    reporter_thread.start()
    try:
        with ThreadPoolExecutor(max_workers=max(1, threads)) as pool:
            list(pool.map(lambda _: worker(), range(max(1, threads))))
    finally:
        stop_event.set()

    if failures:
        fatal = next(
            (item for item in failures if isinstance(item[3], FatalDownloadError)), failures[0]
        )
        raise fatal[3]

    elapsed = max(1e-6, time.time() - start_time)
    got = sum(part_path(i).stat().st_size for i in range(len(chunks)))
    log(f"  分块下载完成：{got / (1 << 20):.1f} MB / {elapsed:.0f}s = "
        f"{got / (1 << 20) / elapsed:.2f} MB/s")

    return merge_and_verify(chunks, part_path, dest)


def merge_and_verify(chunks: List[Tuple[int, int]], part_path, dest: Path) -> Path:
    """按序合并分块，并做「文件大小 + MD5」双重校验。"""
    for index, (start, end) in enumerate(chunks):
        path = part_path(index)
        size = path.stat().st_size if path.is_file() else 0
        if size != end - start + 1:
            raise RuntimeError(
                f"分片 part_{index:04d} 长度异常（{size}/{end - start + 1} 字节）"
            )

    merge_tmp = dest / "_archive_merging.bin"
    with open(merge_tmp, "wb") as out:
        for index in range(len(chunks)):
            with open(part_path(index), "rb") as handle:
                for block in iter(lambda: handle.read(1 << 22), b""):
                    out.write(block)

    actual_size = merge_tmp.stat().st_size
    if actual_size != ARCHIVE_SIZE:
        merge_tmp.unlink(missing_ok=True)
        raise RuntimeError(f"合并后文件大小异常：{actual_size}/{ARCHIVE_SIZE} 字节")

    log(f"  校验 MD5（{ARCHIVE_SIZE / (1 << 20):.1f} MB）...")
    digest = md5_of(merge_tmp)
    if digest != ARCHIVE_MD5:
        merge_tmp.unlink(missing_ok=True)
        raise RuntimeError(f"MD5 校验失败：期望 {ARCHIVE_MD5}，实际 {digest}")
    log(f"  校验通过：大小 {ARCHIVE_SIZE} 字节，MD5 {digest}")

    archive = dest / ARCHIVE_NAME
    os.replace(merge_tmp, archive)
    shutil.rmtree(dest / "_parts", ignore_errors=True)
    return archive


def single_connection_download(
    url: str,
    dest: Path,
    max_retries: int = DEFAULT_MAX_RETRIES,
    timeout: float = DEFAULT_TIMEOUT,
) -> Path:
    """单连接流式下载（最后的回退方案），支持断点续传 + 重试。"""
    archive = dest / ARCHIVE_NAME
    part = dest / (ARCHIVE_NAME + ".partial")

    for attempt in range(max_retries + 1):
        have = part.stat().st_size if part.is_file() else 0
        if have > ARCHIVE_SIZE:
            part.unlink(missing_ok=True)
            have = 0
        if have == ARCHIVE_SIZE:
            break

        log(f"单连接下载（第 {attempt + 1}/{max_retries + 1} 次尝试，从 "
            f"{have / (1 << 20):.1f} MB 处续传）...")
        headers = {"User-Agent": USER_AGENT, "Accept-Encoding": "identity"}
        if have:
            headers["Range"] = f"bytes={have}-"
        request = urllib.request.Request(url, headers=headers)
        start_time = time.time()
        last_report = 0.0
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                # 有断点却收到 200，说明服务器忽略了 Range：必须从头覆盖，不能追加
                mode = "ab"
                if have and response.status != 206:
                    log("  服务器忽略了 Range，从头重新下载")
                    have, mode = 0, "wb"
                with open(part, mode) as out:
                    while True:
                        block = response.read(1 << 20)
                        if not block:
                            break
                        out.write(block)
                        if time.time() - last_report > 15:
                            last_report = time.time()
                            got = out.tell()
                            elapsed = max(1e-6, time.time() - start_time)
                            log(f"  进度 {got / (1 << 20):7.1f}/{ARCHIVE_SIZE / (1 << 20):.1f} MB "
                                f"({100 * got / ARCHIVE_SIZE:5.1f}%) "
                                f"平均 {got / (1 << 20) / elapsed:.2f} MB/s")
        except BaseException as error:  # noqa: BLE001
            if isinstance(error, urllib.error.HTTPError) and error.code in FATAL_HTTP_STATUS:
                raise FatalDownloadError(
                    f"单连接遇到不可重试错误：HTTP {error.code} {error.reason}"
                ) from error
            if attempt >= max_retries:
                raise
            delay = backoff_delay(attempt)
            log(f"  单连接中断（{type(error).__name__}: {error}），{delay:.1f}s 后重试")
            time.sleep(delay)
        # 循环回到顶部：检查落盘大小，收满则结束

    size = part.stat().st_size if part.is_file() else 0
    if size != ARCHIVE_SIZE:
        raise RuntimeError(f"下载不完整：{size}/{ARCHIVE_SIZE} 字节")

    log(f"  校验 MD5（{size / (1 << 20):.1f} MB）...")
    digest = md5_of(part)
    if digest != ARCHIVE_MD5:
        part.unlink(missing_ok=True)
        raise RuntimeError(f"MD5 校验失败：期望 {ARCHIVE_MD5}，实际 {digest}")
    log(f"  校验通过：大小 {size} 字节，MD5 {digest}")
    os.replace(part, archive)
    return archive


def download_with_fallback(
    mirrors: List[Mirror],
    dest: Path,
    threads: int,
    chunk_size: int,
    max_retries: int,
    timeout: float,
) -> Path:
    """按优先级依次尝试各镜像；每个镜像内部先多连接、再单连接。

    任何一次尝试失败都会**清理残留**后再进入下一档，保证不会把
    上一个镜像的半成品当成断点续传的基础（脏数据会导致 MD5 静默错误）。
    """
    failures: List[str] = []
    for rank, mirror in enumerate(mirrors, start=1):
        modes: List[Tuple[str, bool]] = []
        if mirror.range_ok:
            modes.append(("多连接分块", True))
        modes.append(("单连接", False))

        for label, use_multi in modes:
            log(f"[尝试 {rank}/{len(mirrors)}] 镜像 {mirror.url} —— {label}")
            try:
                if use_multi:
                    archive = multi_connection_download(
                        mirror.url, dest, threads, chunk_size, max_retries, timeout
                    )
                else:
                    archive = single_connection_download(mirror.url, dest, max_retries, timeout)
                log(f"[成功] {label} 完成：{archive}")
                return archive
            except BaseException as error:  # noqa: BLE001
                log(f"[回退] {label} 失败：{type(error).__name__}: {error}")
                failures.append(f"{mirror.url} / {label} / {type(error).__name__}: {error}")
                cleanup_partials(dest)
                (dest / (ARCHIVE_NAME + ".partial")).unlink(missing_ok=True)

    raise RuntimeError("所有镜像与下载方式均失败：\n  - " + "\n  - ".join(failures))


# --------------------------------------------------------------------------- #
# 解压
# --------------------------------------------------------------------------- #
def extract(archive: Path, dest: Path) -> None:
    """解压归档，产出 ``<dest>/cifar-10-batches-py``。"""
    if is_extracted(dest):
        log("已存在解压结果，跳过解压")
        return
    log(f"解压 {archive.name} ...")
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(dest)
    log(f"解压完成：{dest / BASE_FOLDER}")


# --------------------------------------------------------------------------- #
# 主流程
# --------------------------------------------------------------------------- #
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="CIFAR-10 高速下载器（镜像体检 + 多连接分块 + MD5 校验 + 解压）",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument("--dest", type=str, default=str(PROJECT_ROOT / "data"),
                        help="数据目录（与 train.py 的 --data-root 保持一致）")
    parser.add_argument("--threads", type=int, default=DEFAULT_THREADS, help="并发连接数")
    parser.add_argument("--chunk-mb", type=float, default=DEFAULT_CHUNK_SIZE / (1 << 20),
                        help="单个分块大小（MiB）")
    parser.add_argument("--max-retries", type=int, default=DEFAULT_MAX_RETRIES,
                        help="单个分块的最大重试次数（不含首次尝试）")
    parser.add_argument("--timeout", type=float, default=DEFAULT_TIMEOUT,
                        help="单次 socket 操作超时（秒）")
    parser.add_argument("--mirrors", type=str, nargs="*", default=None, help="自定义镜像 URL 列表")
    parser.add_argument("--probe-only", action="store_true", help="只体检镜像，不下载")
    parser.add_argument("--force", action="store_true", help="忽略已有缓存，强制重新下载")
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    dest = Path(args.dest).expanduser().resolve()
    dest.mkdir(parents=True, exist_ok=True)
    archive = dest / ARCHIVE_NAME
    mirrors = [str(item) for item in (args.mirrors or DEFAULT_MIRRORS)]
    chunk_size = max(1 << 20, int(args.chunk_mb * (1 << 20)))
    threads = max(1, int(args.threads))

    log("=" * 78)
    log(f"目标目录：{dest}")
    log(f"归档期望：{ARCHIVE_NAME}  {ARCHIVE_SIZE} 字节 "
        f"({ARCHIVE_SIZE / (1 << 20):.1f} MB)  MD5={ARCHIVE_MD5}")
    log("=" * 78)

    # ---------------- 0. 缓存命中则直接复用 ---------------- #
    if not args.force and is_archive_valid(archive):
        log("归档已存在且 MD5 校验通过，跳过下载")
    elif not args.force and is_extracted(dest):
        log("数据已解压就绪（未保留归档文件），跳过下载")
        _report(dest)
        return 0
    else:
        # ---------------- 1. 镜像体检 ---------------- #
        log("体检候选镜像：远端长度 -> 采样 1 MB 测速 -> Range 支持 ...")
        inspected = inspect_mirrors(mirrors)
        for mirror in inspected:
            if not mirror.usable:
                log(f"[剔除] 远端长度为 {mirror.remote_size}，与期望 {ARCHIVE_SIZE} 不符"
                    f"（镜像已损坏）<- {mirror.url}")

        candidates = [mirror for mirror in inspected if mirror.usable]
        if not candidates:
            log("失败：所有镜像均不可用或长度不符。请检查网络，或改用 "
                "--dataset synthetic 走零下载冒烟测试。")
            return 1

        if args.probe_only:
            log("仅体检（--probe-only），结束。")
            return 0

        # 排序：先看是否支持 Range（决定能否多连接加速），再看速率
        candidates.sort(key=lambda item: (item.range_ok, item.speed or 0.0), reverse=True)
        log(f"镜像尝试顺序：{', '.join(item.url for item in candidates)}")

        # ---------------- 2. 下载（含回退链） ---------------- #
        try:
            archive = download_with_fallback(
                candidates, dest, threads, chunk_size, args.max_retries, args.timeout
            )
        except Exception as error:  # noqa: BLE001
            log(f"下载失败：{error}")
            return 1

    # ---------------- 3. 解压 ---------------- #
    extract(archive, dest)
    _report(dest)
    return 0


def _report(dest: Path) -> None:
    """打印最终目录结构与关键文件大小。"""
    log("-" * 78)
    log("数据就绪：")
    for path in sorted(dest.rglob("*")):
        if path.is_file():
            log(f"  {path.relative_to(dest)}  ({path.stat().st_size / (1 << 20):.1f} MB)")
    log("提示：把它交给训练脚本 —— python train.py --data-root " + str(dest))
    log("-" * 78)


if __name__ == "__main__":
    raise SystemExit(main())


### 4. 数据模块 —— 划分协议与 DataLoader 组装


`src/data/loader.py`（281 行）


In [ ]:
%%writefile src/data/loader.py
# -*- coding: utf-8 -*-
"""
数据集组装与 DataLoader 构建。

本模块负责把 :mod:`src.data.cifar10` / :mod:`src.data.synthetic` 提供的"数据底座"
组装成训练 / 验证 / 测试三个数据集与对应的 DataLoader。

数据划分协议（与既有结果严格一致，不可随意更改）
------------------------------------------------
* **训练池**：官方训练集 50000 张，用 ``torch.Generator().manual_seed(--seed)``
  的 ``torch.randperm(50000)`` 打乱；
* **验证集**：取该排列的**后 10%**（默认 5000 张），**不参与训练**；
* **测试集**：官方测试集 10000 张，**完全不参与训练与调参**；
* **快速验证子集**（可选 ``--train-subset-size`` / ``--test-subset-size``）：
  从同一排列中截取前 N 个，因此与全量划分**完全一致且可互相校验**。

由于索引语义直接决定已归档结果的可复现性，本模块的随机数使用方式与顺序
**必须保持原样**；`04_data/splits/*.pt` 中保存的索引可用于回归校验。

性能说明（实测优化）
--------------------
``torchvision.datasets.CIFAR10`` 每次构造都会对全部批次文件做 MD5 完整性校验，
并把 pickle 反序列化到内存。原实现为"带增强训练集 / 不带增强验证集 / 测试集"
各构造一个实例，导致：

* CIFAR-10 被实例化 **3 次** → 完整性校验被触发 **6 次**（约 186 MB × 6 的哈希）；
* 训练集 pickle（146 MB）被读入内存 **2 份**。

实测热路径耗时 4.5 s，其中几乎全部由重复 MD5 校验贡献。现改为：
**每个 split 只构造一个 ``transform=None`` 的底座**，再通过
:class:`TransformSubset` 叠加不同变换 —— 划分与索引语义不变，开销显著下降。
"""

from __future__ import annotations

import torch
from torch.utils.data import DataLoader, Dataset
from typing import List, Optional, Sequence, Tuple

from ..utils.logger import get_logger
from ..utils.seed import make_generator, seed_worker
from .cifar10 import (
    CIFAR10_CLASSES,
    build_cifar10_base,
    build_test_transform,
    build_train_transform,
    is_extracted,
)
from .synthetic import build_synthetic_dataset

#: 官方划分规模（用于判断是否为全量实验）
OFFICIAL_TRAIN_SIZE = 50000
OFFICIAL_TEST_SIZE = 10000

#: 识别 CIFAR-10 的数据集名称别名
_CIFAR10_ALIASES = {"cifar10", "cifar-10", "cifar"}
#: 识别合成数据的名称别名
_SYNTHETIC_ALIASES = {"synthetic", "fake", "dummy"}


# --------------------------------------------------------------------------- #
# 变换包装器
# --------------------------------------------------------------------------- #
class TransformSubset(Dataset):
    """在**同一个底座数据集**上，用指定索引与指定变换取样本。

    为什么需要它
    ------------
    训练集需要"带增强"与"不带增强（验证用）"两个版本，而
    ``torchvision.datasets.CIFAR10`` 的 ``transform`` 在构造时就固定了。
    若各构造一个实例，会重复触发 MD5 完整性校验与 pickle 反序列化。
    本包装器让两种变换共享同一个 ``transform=None`` 的底座，从而消除该开销，
    同时**不改变任何索引语义**（取出的是同一个底座的第 ``indices[i]`` 个样本）。

    参数
    ----
    base : Dataset
        底座数据集，其 ``__getitem__`` 返回 ``(PIL.Image, int)``。
    indices : Sequence[int]
        底座中的样本下标（顺序即子集顺序）。
    transform : callable | None
        作用于 PIL 图像的变换；None 表示不变换。
    """

    def __init__(self, base: Dataset, indices: Sequence[int], transform=None) -> None:
        self.base = base
        self.indices: List[int] = list(indices)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int) -> Tuple[object, int]:
        image, target = self.base[self.indices[index]]
        if self.transform is not None:
            image = self.transform(image)
        return image, target


# --------------------------------------------------------------------------- #
# 底座数据集
# --------------------------------------------------------------------------- #
def build_base_datasets(cfg, logger=None) -> Tuple[Dataset, Dataset, str]:
    """按 ``cfg.data.dataset`` 构造 ``(train_base, test_base, dataset_name)``。

    * ``dataset="CIFAR10"``：真实数据集；仅在缓存缺失时才允许下载，
      避免每次都走"下载分支"而多触发一轮完整性校验；
    * ``dataset="synthetic"``：本地生成的合成数据，**零下载**，
      用于网络受限环境下的流程冒烟测试。

    返回的底座均 **不带变换**（``transform=None``），由 :class:`TransformSubset` 叠加。
    """
    name = str(getattr(cfg.data, "dataset", "CIFAR10")).strip()
    key = name.lower()

    if key in _SYNTHETIC_ALIASES:
        train_base = build_synthetic_dataset(cfg, train=True, logger=logger)
        test_base = build_synthetic_dataset(cfg, train=False, logger=logger)
        return train_base, test_base, "synthetic"

    if key in _CIFAR10_ALIASES:
        root = cfg.data.data_root
        cached = is_extracted(root)
        if logger is not None:
            logger.info(f"[data] CIFAR-10 缓存状态：{'已就绪（跳过下载与重复校验）' if cached else '缺失，将自动下载'}")
        # 只在缓存缺失时要求下载；测试集与训练集来自同一个归档，因此无需重复下载
        train_base = build_cifar10_base(cfg, train=True, download=not cached, logger=logger)
        test_base = build_cifar10_base(cfg, train=False, download=False, logger=logger)
        return train_base, test_base, "CIFAR-10"

    raise ValueError(
        f"未知的 dataset='{name}'，可选：'CIFAR10'（真实数据）或 'synthetic'（零下载合成数据）"
    )


# --------------------------------------------------------------------------- #
# 数据集
# --------------------------------------------------------------------------- #
def build_datasets(cfg, logger=None) -> Tuple[Dataset, Dataset, Dataset, List[str]]:
    """构造并返回 ``(train_set, val_set, test_set, class_names)``。

    划分逻辑与历史结果严格一致（见模块文档字符串）。启用"快速验证子集"时，
    日志会打印**显式警告**，提醒其指标不可与全量结果比较。
    """
    log = logger if logger is not None else get_logger()
    train_base, test_base, dataset_name = build_base_datasets(cfg, logger=logger)

    train_transform = build_train_transform(
        mean=cfg.data.mean, std=cfg.data.std,
        rand_augment=cfg.data.rand_augment, random_erase=cfg.data.random_erase,
    )
    eval_transform = build_test_transform(mean=cfg.data.mean, std=cfg.data.std)

    # ---------------- 训练池的固定种子排列 ---------------- #
    total = len(train_base)
    generator = torch.Generator().manual_seed(int(cfg.train.seed))
    perm = torch.randperm(total, generator=generator).tolist()

    # ---------------- 可选的"快速验证子集" ---------------- #
    pool_indices = perm
    subset_size = getattr(cfg.data, "train_subset_size", None)
    used_train_subset = False
    if subset_size is not None and 0 < int(subset_size) < total:
        pool_indices = perm[: int(subset_size)]
        used_train_subset = True

    pool_size = len(pool_indices)
    num_val = int(pool_size * float(cfg.data.val_ratio))
    num_train = pool_size - num_val

    train_indices = pool_indices[:num_train]
    val_indices = pool_indices[num_train:]

    # 训练与验证共享同一个底座，只是变换不同（这是本次性能优化的核心）
    train_set = TransformSubset(train_base, train_indices, train_transform)
    val_set = TransformSubset(train_base, val_indices, eval_transform)

    # ---------------- 测试集（可选子集） ---------------- #
    test_indices = list(range(len(test_base)))
    used_test_subset = False
    test_size = getattr(cfg.data, "test_subset_size", None)
    if test_size is not None and 0 < int(test_size) < len(test_base):
        # 使用与训练集不同的种子偏移，避免两个抽样序列相互关联
        test_generator = torch.Generator().manual_seed(int(cfg.train.seed) + 1)
        shuffled = torch.randperm(len(test_base), generator=test_generator).tolist()
        test_indices = shuffled[: int(test_size)]
        used_test_subset = True

    test_set = TransformSubset(test_base, test_indices, eval_transform)

    if used_train_subset or used_test_subset:
        log.warning(
            "[data] 已启用『快速验证子集』模式："
            + (f"训练池 {pool_size} / 原 {total} 张" if used_train_subset else "")
            + ("，" if used_train_subset and used_test_subset else "")
            + (f"测试集 {len(test_set)} / 原 {len(test_base)} 张" if used_test_subset else "")
            + "。该模式下的指标**不能**与全量数据结果直接比较，仅用于快速验证流程。"
        )
    if dataset_name == "synthetic":
        log.warning("[data] 当前使用合成数据，全部指标仅供流程验证，不具备任何科学意义。")

    class_names = list(getattr(train_base, "classes", CIFAR10_CLASSES))
    return train_set, val_set, test_set, class_names


# --------------------------------------------------------------------------- #
# DataLoader
# --------------------------------------------------------------------------- #
def build_dataloaders(
    cfg,
    batch_size: int,
    eval_batch_size: Optional[int] = None,
    logger=None,
) -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:
    """构建训练 / 验证 / 测试三个 DataLoader。

    参数
    ----
    cfg : Config
        全局配置对象。
    batch_size : int
        训练 batch size（建议传入显存自适应探测后的结果）。
    eval_batch_size : int | None
        验证/测试 batch size；为 None 时取 ``2 * batch_size``（推理无梯度，可更大）。
    logger : Logger | None
        日志器，用于打印数据集规模与划分信息。

    返回
    ----
    (train_loader, val_loader, test_loader, class_names)
    """
    log = logger if logger is not None else get_logger()
    train_set, val_set, test_set, class_names = build_datasets(cfg, logger=logger)

    if eval_batch_size is None:
        eval_batch_size = max(1, int(batch_size) * 2)

    device_type = "cuda" if torch.cuda.is_available() else "cpu"
    use_pin_memory = bool(cfg.data.pin_memory) and device_type == "cuda"
    num_workers = int(cfg.data.num_workers)

    # 多进程相关设置：num_workers=0 时必须关闭 persistent_workers
    common_kwargs = dict(
        num_workers=num_workers,
        pin_memory=use_pin_memory,
        persistent_workers=(num_workers > 0),
        worker_init_fn=seed_worker if num_workers > 0 else None,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=int(batch_size),
        shuffle=True,                                     # 训练集每个 epoch 重新打乱
        drop_last=False,
        generator=make_generator(int(cfg.train.seed)),    # 固定 shuffle 顺序（可复现）
        **common_kwargs,
    )
    val_loader = DataLoader(
        val_set,
        batch_size=int(eval_batch_size),
        shuffle=False,
        drop_last=False,
        **common_kwargs,
    )
    test_loader = DataLoader(
        test_set,
        batch_size=int(eval_batch_size),
        shuffle=False,
        drop_last=False,
        **common_kwargs,
    )

    log.info(
        f"[data] {class_names and '数据集'} 划分完成 | 训练 {len(train_set)} 张 | "
        f"验证 {len(val_set)} 张 | 测试 {len(test_set)} 张 | 类别数 {len(class_names)}"
    )
    log.info(
        f"[data] DataLoader | train batch={batch_size} | eval batch={eval_batch_size} | "
        f"num_workers={num_workers} | pin_memory={use_pin_memory}"
    )
    return train_loader, val_loader, test_loader, class_names


### 4. 数据模块 —— 合成数据（零下载冒烟测试用）


`src/data/synthetic.py`（132 行）


In [ ]:
%%writefile src/data/synthetic.py
# -*- coding: utf-8 -*-
"""
合成数据集（**零下载**）：用于在无网络或网络缓慢时验证整条训练流水线。

设计目标
--------
1. **完全离线**：不访问网络、不依赖任何已下载文件，构造即用 ——
   这是"快速冒烟测试"能在数十秒内完成的关键（详见 ``--dataset synthetic``）。
2. **形态与 CIFAR-10 完全一致**：32×32×3、10 类、返回 ``(PIL.Image, int)``，
   因此可以直接复用同一套数据变换、同一个模型与同一套评估/可视化代码，
   验证的是**真实代码路径**而非另写一条"假路径"。
3. **确定性**：由固定随机种子一次性生成固定的张量集合，多次运行结果一致，可复现。
4. **可学习**：并非纯噪声 —— 每个类别叠加一个固定的"类模板"信号再混入噪声，
   使模型在小规模下也能真的学到东西。这样损失/准确率曲线会呈现**正常下降趋势**，
   从而验证"指标会动、绘图正常、检查点会刷新"；若用纯随机标签，
   曲线只能证明"代码不报错"，无法证明"训练有效"。

与真实数据集的关系
------------------
* 合成数据的指标**没有科学意义**，绝不能写入报告的实验结果；
* 它只用于环境自检与流水线冒烟测试（对应协作计划中的"先打通 MVP"原则）。
  结果文件中通过 ``dataset.name = "synthetic"`` 与 ``is_full_scale = false`` 双重标记。
"""

from __future__ import annotations

from typing import List, Optional, Tuple

import numpy as np
from PIL import Image
from torch.utils.data import Dataset

from .cifar10 import CIFAR10_CLASSES, IMAGE_SIZE


class SyntheticCIFAR(Dataset):
    """与 CIFAR-10 同形态的合成图像数据集（本地生成，零下载）。

    参数
    ----
    size : int
        样本数量。
    num_classes : int
        类别数量，默认 10（与 CIFAR-10 一致）。
    image_size : int
        图像边长，默认 32。
    seed : int
        生成种子。相同种子 + 相同参数 ⇒ 完全相同的图像与标签（可复现）。
    template_weight : float
        类模板在单张图像中的权重（0~1）。越大类间越易区分、任务越简单。
        默认 0.55，使模型在少量轮次内即可看到明显优于随机的准确率。
    transform : callable | None
        可选变换。本工程通常传 None，由 ``TransformSubset`` 叠加训练/评估变换。
    """

    def __init__(
        self,
        size: int = 4096,
        num_classes: int = 10,
        image_size: int = IMAGE_SIZE,
        seed: int = 1234,
        template_weight: float = 0.55,
        transform=None,
    ) -> None:
        self.size = int(size)
        self.num_classes = int(num_classes)
        self.image_size = int(image_size)
        self.seed = int(seed)
        self.transform = transform
        self.classes: List[str] = list(CIFAR10_CLASSES[: self.num_classes])

        rng = np.random.RandomState(self.seed)

        # 1) 每个类别一个固定的"类模板"（低频结构信号）
        templates = rng.randint(0, 256, size=(self.num_classes, image_size, image_size, 3))
        templates = templates.astype(np.float32)

        # 2) 逐样本随机标签，并按标签取模板
        labels = rng.randint(0, self.num_classes, size=self.size).astype(np.int64)
        template = templates[labels]                       # (N, H, W, 3)

        # 3) 混入均匀噪声：模板权重越高越易分类
        noise = rng.randint(0, 256, size=(self.size, image_size, image_size, 3)).astype(np.float32)
        weight = float(template_weight)
        data = np.clip(weight * template + (1.0 - weight) * noise, 0.0, 255.0)

        # 与 CIFAR-10 保持一致：uint8、HWC 布局，便于复用同一套 PIL 变换
        self.data: np.ndarray = data.astype(np.uint8)
        self.targets: List[int] = labels.tolist()

    # ------------------------------------------------------------------ #
    def __len__(self) -> int:
        return self.size

    def __getitem__(self, index: int) -> Tuple[object, int]:
        """返回 ``(PIL.Image, int)``，与 ``torchvision.datasets.CIFAR10`` 完全一致。"""
        image = Image.fromarray(self.data[index])
        if self.transform is not None:
            image = self.transform(image)
        return image, int(self.targets[index])

    def extra_repr(self) -> str:
        return (f"size={self.size}, num_classes={self.num_classes}, "
                f"image_size={self.image_size}, seed={self.seed}")


def build_synthetic_dataset(cfg, train: bool, logger=None) -> Dataset:
    """按配置构造合成数据集（训练/测试两份，种子不同以免完全重叠）。

    参数
    ----
    cfg : Config
        读取 ``data.synthetic_train_size`` / ``data.synthetic_test_size`` /
        ``data.num_classes`` 与 ``train.seed``。
    train : bool
        True 返回训练用合成集，False 返回测试用合成集。
    """
    size = int(cfg.data.synthetic_train_size if train else cfg.data.synthetic_test_size)
    seed = int(cfg.train.seed) + (0 if train else 10_000)
    dataset = SyntheticCIFAR(
        size=size,
        num_classes=int(cfg.data.num_classes),
        image_size=IMAGE_SIZE,
        seed=seed,
        transform=None,
    )
    if logger is not None:
        logger.warning(
            f"[data] 使用**合成数据**（dataset=synthetic，{size} 张，seed={seed}）——"
            f"零下载、仅用于流程冒烟测试；其指标没有科学意义，禁止写入报告结果。"
        )
    return dataset


## 4. 运行

工程源码已全部就绪。建议按下面的顺序执行；其中 3-B 与 3-C 是**唯一需要联网**的步骤，
而 3-A 冒烟测试**完全不需要联网**，可在几十秒内确认环境是否正常。

| 步骤 | 内容 | 预计耗时 | 需要联网 |
|---|---|---|---|
| 3-A | 快速冒烟测试（本地合成数据 + 受限 step，**零下载**） | GPU 约 30 s / CPU 约 1 min | 不需要 |
| 3-B | （可选）下载并缓存 CIFAR-10（多连接加速 + MD5 校验） | 约 2 分钟（视网络） | 需要 |
| 3-C | 正式训练（ConvNeXt-Tiny, 60 epochs, `--no-amp`） | T4 上约 3~5 小时（fp32） | 首次需要 |
| 3-D | 展示训练曲线与评估指标 | 秒级 | 不需要 |
| 3-E | （可选）独立评估测试集 | 数十秒 | 不需要 |
| 3-F | （可选）打包下载全部产物 | 秒级 | 不需要 |

> **T4 上请勿使用 fp16 AMP**：本模型的 LayerScale 初值为 `1e-6`，浅层 Block 的梯度量级只有 `1e-6`，
> 一旦 GradScaler 的 scale 被 fp16 溢出压到 `0.0156` 这类小值，梯度会下溢为精确的 0，
> 训练表现为「loss 不下降、精度停在 ~25%」的假收敛（详见 `src/utils/amp.py` 的模块说明）。
> 3-C 默认已加 `--no-amp`；换到 L4/A100 时可去掉（程序会自动使用 bfloat16）。

> **在本机 GPU 上运行（Windows / Linux，非 Colab）**：先装 CUDA 版 PyTorch（命令见
> `requirements.txt`），然后把命令换成 `python train.py ... --device cuda`（`--device auto` 也会自动选到 CUDA）。
> 注意事项：① Ampere 及以后（RTX 30/40、L4、A100）程序自动用 **bfloat16**，T4 等回退 fp16（建议加 `--no-amp`）；
> ② Windows 长跑建议 `--num-workers 0`（多进程 DataLoader 的共享内存提交可能耗尽，报 `error 1455`）；
> ③ 显存较小的卡依赖自动探测即可 —— 探针要求峰值 ≤ 可用显存 70%，会为训练留出余量；
> ④ 若长跑中断，用 `--resume auto` 原地续训。详见 `03_code/README.md` 第 2.2 / 6 / 12.4 / 12.5 节。


In [ ]:
#@title 3-A. 快速冒烟测试（零下载，约 30 秒 ~ 1 分钟）
import torch

# 冒烟测试的唯一目标是：确认"环境 + 代码链路"通畅。
# 为此它做了两件事：
#   1) 使用**本地生成的合成数据**（--dataset synthetic）—— 完全不下载任何东西。
#      原先生成器的冒烟测试会触发 CIFAR-10 下载（162 MB）。官方源单连接在部分网络下
#      只有约 0.1 MB/s，仅下载就要 25 分钟，这是它无法在 2 分钟内完成的首要原因。
#   2) 用 --max-steps-per-epoch 给训练量设上界 —— 总耗时只取决于 step 数，
#      与数据集规模、硬件性能解耦（Colab 纯 CPU 运行时仅 2 vCPU，这点很关键）。
GPU = torch.cuda.is_available()
if GPU:
    MODEL, BATCH, TRAIN_N, TEST_N, STEPS, EPOCHS = "atto", 128, 2048, 1024, 20, 2
else:
    MODEL, BATCH, TRAIN_N, TEST_N, STEPS, EPOCHS = "atto", 32, 512, 256, 4, 2

print(f"运行时={'GPU' if GPU else 'CPU'} | model={MODEL} batch={BATCH} "
      f"train={TRAIN_N} test={TEST_N} steps/epoch={STEPS} epochs={EPOCHS}")

!python train.py --run-tag smoke_test --dataset synthetic --synthetic-train-size {TRAIN_N} --synthetic-test-size {TEST_N} --model {MODEL} --batch-size {BATCH} --max-steps-per-epoch {STEPS} --epochs {EPOCHS} --exp-name smoke_test


In [ ]:
#@title 3-B. （可选）下载并缓存 CIFAR-10（多连接加速）
# （可选）提前把 CIFAR-10 下载到工作目录，供正式训练使用。
#
# 为什么要单独跑这一步：官方源单连接在部分网络环境下只有约 0.1 MB/s，
# 162 MB 需要约 25 分钟；而该源支持 HTTP Range，因此脚本用**多连接分块**
# 把聚合带宽提升一个量级（实测 1.3 ~ 8 MB/s，下载+校验+解压约 20 ~ 120 秒）。
# 脚本会自动：镜像体检（剔除被截断的损坏镜像）-> 分块并行下载（断点续传、指数退避）
# -> 校验文件大小 + MD5 -> 解压；任何一步失败都会自动回退（降并发 -> 单连接 -> 换镜像）。
#
# 参数说明（默认值已按 Colab 调优，通常无需改动）：
#   --threads 8      并发连接数：8 已接近多数镜像的饱和点，16 容易触发限流且收益递减
#   --chunk-mb 4     单个分块大小：太小放大请求开销，太大则单块失败的重下代价高
#   --max-retries 8  单分块最大重试次数（配合指数退避，上限 30s）
!python scripts/fetch_cifar10.py --dest ./data --threads 8 --chunk-mb 4

# 只体检镜像、不下载（想先看哪个镜像可用、速率多快时用）：
# !python scripts/fetch_cifar10.py --dest ./data --probe-only

# 若某个镜像分块反复失败，脚本会自动降并发/改单连接/换镜像；也可手动指定镜像：
# !python scripts/fetch_cifar10.py --dest ./data --threads 4 --mirrors http://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz

# 想跨会话复用、避免每次重新下载？把数据放到 Google Drive 再挂载即可：
#   from google.colab import drive; drive.mount('/content/drive')
#   !python scripts/fetch_cifar10.py --dest /content/drive/MyDrive/cifar10
#   之后训练时加  --data-root /content/drive/MyDrive/cifar10


In [ ]:
#@title 3-C. 正式训练（ConvNeXt-Tiny, 60 epochs）
# 正式训练：ConvNeXt-Tiny（论文 ConvNeXt-T 的结构）+ 60 epochs
#   --batch-size 128 只是"期望值"，显存不足会自动下调并用梯度累积补偿（见日志 [batch-size]）
#   --stem-mode conv 为 CIFAR-10 小分辨率的适配版 Stem（3x3, stride=1）
#   --dataset 默认 CIFAR10（真实数据）；若已执行过 3-B，数据已缓存，不会重复下载
#   --run-tag 指定实验标签 -> 全部产物写入 outputs/<标签>/，多次实验互不覆盖
#
#   --no-amp：**T4 上必须加**。原因：本模型 LayerScale 初值为 1e-6，18 个残差块叠加后
#             浅层 Block 的真实梯度只有 1e-6 甚至更低；fp16 的最小次正规数是 6e-8，
#             一旦 GradScaler 的动态 scale 因 fp16 溢出被压到 0.0156 量级，
#             梯度乘上 scale 后仍会下溢成**精确的 0**，这些参数从此不再更新
#             （实测后果：60 轮 loss 不动、测试精度只有 25.7%，是"假收敛"）。
#             fp32 没有这个动态范围问题。若换到 L4/A100（原生支持 bf16），
#             程序会自动改用 bfloat16，此时可以去掉 --no-amp（bf16 与 fp32 同阶码范围）。
#
#   自检方法：看每轮日志末尾的 amp_scale，正常应稳定在 65536 附近；
#             若持续 < 1，说明 fp16 已在丢梯度（程序会自动降级为 fp32 并打印告警）。
#
#   --resume auto：若本次实验目录下已有 last.pt，则自动续训（Colab 断线重连专用：
#                 重连后**重新执行本单元格**即可接着训练，不会从头再来）
#
# 若想更快看到结果（T4 fp32 更慢，可换更小的模型/更少的轮数）：
#   RUN_TAG = "pico_30e"
#   !python train.py --run-tag {RUN_TAG} --epochs 30 --model pico --batch-size 128 --no-amp
RUN_TAG = "full_convnext_tiny_cifar10"
!python train.py --run-tag {RUN_TAG} --epochs 60 --model tiny --batch-size 128 --stem-mode conv --no-amp

# 若中途断连，只需把上一行改成带 --resume auto 的版本再执行一次：
# !python train.py --run-tag {RUN_TAG} --epochs 60 --model tiny --batch-size 128 --stem-mode conv --no-amp --resume auto


In [ ]:
#@title 3-D. 展示训练曲线与评估指标
import json, os
from IPython.display import Image, display, Markdown

# 与训练单元格保持一致的实验标签；若改了标签，这里同步修改即可
# （只跑了 3-A 冒烟测试时，把下面改成 "smoke_test" 即可查看其结果）
RUN_TAG = "full_convnext_tiny_cifar10"
RUN_DIR = os.path.join("outputs", RUN_TAG)

FIG_DIR = os.path.join(RUN_DIR, "figures")
RESULT_FILE = os.path.join(RUN_DIR, "metrics", "test_metrics.json")
CSV_FILE = os.path.join(RUN_DIR, "metrics", "metrics.csv")

print("实验目录 :", os.path.abspath(RUN_DIR))
if os.path.isdir(RUN_DIR):
    for root, _dirs, files in os.walk(RUN_DIR):
        for name in sorted(files):
            p = os.path.join(root, name)
            print(f"  {p}  ({os.path.getsize(p):,} B)")

# ---- 1) 训练曲线：损失 / 准确率 / 学习率 ---- #
curves = os.path.join(FIG_DIR, "training_curves.png")
if os.path.isfile(curves):
    display(Markdown("### 训练曲线（损失 / 准确率 / 学习率）"))
    display(Image(filename=curves))
else:
    print("尚未生成训练曲线，请先运行训练单元格。")

# ---- 2) 混淆矩阵与逐类准确率 ---- #
for fname, caption in [
    ("confusion_matrix.png", "### 测试集混淆矩阵（按真实类别归一化）"),
    ("per_class_accuracy.png", "### 测试集逐类准确率"),
    ("data_samples.png", "### CIFAR-10 数据样例"),
]:
    path = os.path.join(FIG_DIR, fname)
    if os.path.isfile(path):
        display(Markdown(caption))
        display(Image(filename=path))

# ---- 3) 最终测试指标 ---- #
if os.path.isfile(RESULT_FILE):
    with open(RESULT_FILE, "r", encoding="utf-8") as f:
        summary = json.load(f)
    display(Markdown("### 最终测试指标"))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

# ---- 4) 指标汇总表（metrics.csv）---- #
if os.path.isfile(CSV_FILE):
    display(Markdown("### 指标汇总表（metrics.csv）"))
    print(open(CSV_FILE, "r", encoding="utf-8").read())


In [ ]:
#@title 3-E. （可选）独立评估测试集
# 用训练得到的验证集最优模型，独立对测试集再评估一次（不重新训练）。
# 脚本会自动从检查点内的 config 恢复模型结构，因此无需再指定 --model / --stem-mode。
RUN_TAG = "full_convnext_tiny_cifar10"

# 常规评估（不启用 TTA）
!python evaluate.py --ckpt ./outputs/{RUN_TAG}/models/best.pt

# 启用测试时增强（原图 + 水平翻转，平均 softmax 概率）：耗时约翻倍，通常 +0.2~0.5 个百分点的 Top-1
!python evaluate.py --ckpt ./outputs/{RUN_TAG}/models/best.pt --tta


In [ ]:
#@title 3-F. （可选）打包下载全部产物
import os, shutil

archive_base = "/content/convnext_cifar10_outputs"
zip_path = shutil.make_archive(archive_base, "zip", "outputs")
print("已打包 :", zip_path, f"({os.path.getsize(zip_path) / 1024 ** 2:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)      # 仅在 Colab 中可用
except Exception as err:
    print("（非 Colab 环境，已跳过自动下载）:", err)


---

## 说明与对应关系

### 与模块化工程的一致性

本 Notebook 的第 2–10 步会用 `%%writefile` 生成与 `03_code/` **完全相同**的源码文件，
因此 Notebook 内运行的结果与命令行运行 `python train.py` 完全一致：

| 模块 | 文件 | 职责 |
|---|---|---|
| 配置 | `configs.py` | 数据 / 模型 / 训练超参数与命令行解析 |
| 数据 | `src/data/cifar10.py` | CIFAR-10 常量、变换与数据底座 |
| 数据 | `src/data/synthetic.py` | 合成数据（零下载冒烟测试用） |
| 数据 | `src/data/loader.py` | 划分协议、`TransformSubset` 与 DataLoader 组装 |
| 数据工具 | `scripts/fetch_cifar10.py` | CIFAR-10 高速下载器（镜像体检 + 多连接分块 + MD5 校验 + 回退链） |
| 模型 | `src/models/convnext.py` | ConvNeXt 定义（倒置瓶颈、7x7 深度卷积、LayerNorm、Layer Scale、DropPath） |
| 训练 | `src/engine/trainer.py` | 训练循环、AdamW、warmup+cosine、AMP、梯度累积、检查点 |
| 评估 | `src/engine/evaluator.py` | Top-1/Top-5、逐类准确率、混淆矩阵（支持 TTA 口径） |
| 测试时增强 | `src/utils/tta.py` | 原图 + 水平翻转两路 TTA，平均 softmax 概率 |
| CSV 导出 | `src/utils/csv_logger.py` | `history.csv`（逐轮）与 `metrics.csv`（结果汇总） |
| 可视化 | `src/utils/visualize.py` | 损失曲线、准确率曲线、混淆矩阵、逐类准确率 |
| 显存自适应 | `src/utils/device.py` | 探测可用 batch size，不足时自动下调并可用梯度累积补偿 |

### 输出产物

```
outputs/
└── <run-tag>/               # 每次实验一个独立目录（由 --run-tag 指定，互不覆盖）
    ├── figures/   【图表】 training_curves.png, confusion_matrix.png,
    │                      per_class_accuracy.png, data_samples.png
    ├── logs/      【日志】 train.log, evaluate.log, history.json, history.csv
    ├── models/    【模型】 best.pt, last.pt
    └── metrics/   【指标】 config.json, metrics.csv, test_metrics.json,
                           per_class_metrics.json, eval_metrics.json
```

### 参考文献

Zhuang Liu, Hanzi Mao, Chao-Yuan Wu, Christoph Feichtenhofer, Trevor Darrell, Saining Xie.
*A ConvNet for the 2020s.* CVPR 2022, pp. 11976–11986. arXiv:2201.03545.
官方实现：https://github.com/facebookresearch/ConvNeXt
